Industrial Case

In [ ]:
#!/usr/bin/env python3
"""
================================================================================
RAMP-ML — 9 SKU REALI — Codice unificato
  - Holding corretto e tecnologia-specifico: McCormick su B*w e B*v
  - Rimozione doppio conteggio materia prima
  - Backorder LT analitico
  - Warm-start minimale: solo binarie + n3DP (no variabili continue)
  - CORREZIONE CG: calcolo corretto EOQ e Base Stock per Materia Prima 
================================================================================
"""

import numpy as np
import gurobipy as gp
from gurobipy import GRB
from scipy.stats import poisson
from itertools import combinations
import time


def expected_backorder(mu_L, beta):
    if mu_L < 1e-9 or beta <= 0: return mu_L
    return max(mu_L*(1-poisson.cdf(beta-1,mu_L))-beta*(1-poisson.cdf(beta,mu_L)), 0.0)

def expected_on_hand(mu_L, beta):
    return max(beta - mu_L + expected_backorder(mu_L, beta), 0.0)

def optimal_beta(mu_L, c_hold, b_val):
    if mu_L < 1e-9: return 0, 0.0
    best_c, best_b = float('inf'), 0
    for b in range(0, int(mu_L*4)+20):
        t = c_hold*b + b_val*expected_backorder(mu_L, b)
        if t < best_c: best_c, best_b = t, b
        elif t > best_c*1.5 and b > mu_L: break
    return best_b, best_c


def solve_stochastic(
    nI, nD, nC, nS, mu_ic, alpha_ic, SL, SL_raw,
    c_CM, o, h_d, b_i, L_CM, n_i, c_raw, den_raw, unit_raw,
    o_raw, L_raw, L_AM, Leas, Cap_3DP, a_i, t_central,
    t_dic=None, B_cap=None, costo_km=100.0,
    coord_DC=None, coord_C=None, L_card=None, U_card=None,
    mip_gap=0.01, time_limit=600, xi=2.0, seed=42,
    warm_start_sol=None, verbose=False, scenario_name="default"
):
    t_start = time.time()
    I, D, C, S = range(nI), range(nD), range(nC), range(nS)
    pi_s = 1.0 / nS
    SL_i = np.full(nI, SL) if np.isscalar(SL) else np.asarray(SL, dtype=float)

    if t_dic is None and coord_DC is not None and coord_C is not None:
        dist = np.array([[np.sqrt((coord_DC[d,0]-coord_C[c,0])**2 +
                     (coord_DC[d,1]-coord_C[c,1])**2)
             for c in range(nC)] for d in range(nD)])
        t_dic = np.stack([dist * costo_km] * nI, axis=1)

    prod_i  = n_i * c_CM
    vol_i   = (prod_i / 1.30) * 1e-6
    qbar_i  = (vol_i * den_raw) / unit_raw
    ptime_i = prod_i / (1.30 * 0.00525 * 3600)

    def beta_poisson(mu_val, L_val, SL_target):
        lam = mu_val * L_val
        if lam <= 1e-9: return 0
        for s in range(100000):
            if poisson.cdf(s, lam) >= SL_target: return int(s)
        return 99999

    mu_grid_set = {0.0}
    for i in I:
        for r in range(1, nC+1):
            for combo in combinations([float(mu_ic[i,c]) for c in C], r):
                mu_grid_set.add(round(sum(combo), 6))
    mu_grid = np.array(sorted(mu_grid_set))
    M = len(mu_grid)

    B_CM_grid = np.zeros((nI,nD,M)); B_AM_grid = np.zeros((nI,nD,M))
    C_EOQ_grid = np.zeros((nI,nD,M))
    for i in I:
        for d in D:
            for m, mu_m in enumerate(mu_grid):
                B_CM_grid[i,d,m] = beta_poisson(mu_m, L_CM[i], SL_i[i])
                B_AM_grid[i,d,m] = beta_poisson(mu_m, L_AM[i,d], SL_i[i])
                if mu_m > 0:
                    C_EOQ_grid[i,d,m] = np.sqrt(2*o*h_d[d]*c_CM[i]*mu_m)

    B_bar = B_CM_grid[:,:,-1]
    mu_max_i = np.array([mu_ic[i,:].sum() for i in I])

    contributi_raw = [float(qbar_i[i]*mu_ic[i,:].sum()) for i in I]
    if nI <= 15:
        mu_raw_grid_set = {0.0}
        for r in range(1, nI+1):
            for combo in combinations(contributi_raw, r):
                mu_raw_grid_set.add(round(sum(combo), 4))
        mu_raw_grid = np.array(sorted(mu_raw_grid_set))
    else:
        mu_raw_grid = np.linspace(0, sum(contributi_raw), 200)
    M_RAW = len(mu_raw_grid)
    B_raw_grid = np.zeros((nD,M_RAW)); C_EOQ_raw_grid = np.zeros((nD,M_RAW))
    B_bar_raw = np.zeros(nD)
    for d in D:
        for m, mu_m in enumerate(mu_raw_grid):
            B_raw_grid[d,m] = beta_poisson(mu_m, L_raw, SL_raw)
            if mu_m > 0: C_EOQ_raw_grid[d,m] = np.sqrt(2*o_raw*h_d[d]*c_raw*mu_m)
        B_bar_raw[d] = B_raw_grid[d,-1]

    rng = np.random.default_rng(seed)
    d_ics = rng.poisson(mu_ic[:,:,np.newaxis], size=(nI,nC,nS)).astype(float)

    Q_bar_CM = np.zeros((nI,nD)); F_bar = np.zeros((nI,nD))
    for i in I:
        for d in D:
            Q_bar_CM[i,d] = d_ics[i,:,:].sum(axis=0).max() * xi
            F_bar[i,d] = d_ics[i,:,:].max() * xi
    Q_bar_AM = np.full((nI,nD), Cap_3DP * xi)
    if L_card is None: L_card = np.ones(nI, dtype=int)
    if U_card is None: U_card = np.full(nI, nD, dtype=int)

    # ── Modello ───────────────────────────────────────────────────────────
    mdl = gp.Model(f"stoc_{scenario_name}")
    mdl.setParam("OutputFlag", 1 if verbose else 0)
    mdl.setParam("MIPGap", mip_gap); mdl.setParam("TimeLimit", time_limit)

    am = mdl.addVars(nI, vtype=GRB.BINARY, name="am")
    cm = mdl.addVars(nI, vtype=GRB.BINARY, name="cm")
    y  = mdl.addVars(nD, nI, nC, vtype=GRB.BINARY, name="y")
    dc = mdl.addVars(nD, nI, vtype=GRB.BINARY, name="dc")
    v  = mdl.addVars(nI, nD, vtype=GRB.BINARY, name="v")
    w  = mdl.addVars(nI, nD, vtype=GRB.BINARY, name="w")
    B  = mdl.addVars(nI, nD, lb=0.0, name="B")
    n3DP = mdl.addVars(nD, vtype=GRB.INTEGER, lb=0, name="n3DP")
    mu_var = mdl.addVars(nD, nI, lb=0.0, name="mu_var")
    C_EOQ_var = mdl.addVars(nD, nI, lb=0.0, name="C_EOQ_var")
    dcAM = mdl.addVars(nD, vtype=GRB.BINARY, name="dcAM")
    C_EOQ_raw_var = mdl.addVars(nD, lb=0.0, name="C_EOQ_raw_var")
    B_raw = mdl.addVars(nD, lb=0.0, name="B_raw")
    mu_raw_contrib = mdl.addVars(nD, nI, lb=0.0, name="mu_raw_contrib")
    mu_raw_total = mdl.addVars(nD, lb=0.0, name="mu_raw_total")
    beta_cm_aux = mdl.addVars(nI, nD, lb=0.0, name="beta_cm_aux")
    beta_am_aux = mdl.addVars(nI, nD, lb=0.0, name="beta_am_aux")
    bo_cm_aux = mdl.addVars(nI, nD, lb=0.0, name="bo_cm_aux")
    bo_am_aux = mdl.addVars(nI, nD, lb=0.0, name="bo_am_aux")
    q_CM = mdl.addVars(nI, nD, nS, lb=0.0, name="q_CM")
    q_AM = mdl.addVars(nI, nD, nS, lb=0.0, name="q_AM")
    f    = mdl.addVars(nI, nD, nC, nS, lb=0.0, name="f")
    u    = mdl.addVars(nI, nC, nS, lb=0.0, name="u")
    I_inv = mdl.addVars(nI, nD, nS, lb=0.0, name="I_inv")

    # Vincoli
    mdl.addConstrs((am[i]+cm[i]==1 for i in I), name="A1")
    mdl.addConstrs((y[d,i,c]<=dc[d,i] for d in D for i in I for c in C), name="B1")
    mdl.addConstrs((dc[d,i]<=gp.quicksum(y[d,i,c] for c in C) for d in D for i in I), name="B2")
    mdl.addConstrs((gp.quicksum(y[d,i,c] for d in D)==1 for i in I for c in C), name="assign")
    mdl.addConstrs((gp.quicksum(dc[d,i] for d in D)>=L_card[i] for i in I), name="B3_lo")
    mdl.addConstrs((gp.quicksum(dc[d,i] for d in D)<=U_card[i] for i in I), name="B3_up")
    mdl.addConstrs((v[i,d]<=am[i] for i in I for d in D), name="D1")
    mdl.addConstrs((v[i,d]<=dc[d,i] for i in I for d in D), name="D2")
    mdl.addConstrs((v[i,d]>=am[i]+dc[d,i]-1 for i in I for d in D), name="D2b")
    mdl.addConstrs((w[i,d]<=cm[i] for i in I for d in D), name="W1")
    mdl.addConstrs((w[i,d]<=dc[d,i] for i in I for d in D), name="W2")
    mdl.addConstrs((w[i,d]>=cm[i]+dc[d,i]-1 for i in I for d in D), name="W3")
    mdl.addConstrs((mu_var[d,i]==gp.quicksum(mu_ic[i,c]*y[d,i,c] for c in C)
                    for d in D for i in I), name="DG2_direct")
    mdl.addConstrs((mu_var[d,i]<=mu_max_i[i]*dc[d,i] for d in D for i in I), name="DG2_dc")

    N_PWL = 30
    for i in I:
        pts = np.linspace(0.0, max(mu_max_i[i], 0.01), N_PWL).tolist()
        for d in D:
            bcm = [float(beta_poisson(mu, L_CM[i], SL_i[i])) for mu in pts]
            bam = [float(beta_poisson(mu, L_AM[i,d], SL_i[i])) for mu in pts]
            eoq = [float(np.sqrt(2*o*h_d[d]*c_CM[i]*mu)) if mu>0 else 0.0 for mu in pts]
            mdl.addGenConstrPWL(mu_var[d,i], beta_cm_aux[i,d], pts, bcm, name=f"PWL_bCM_{d}_{i}")
            mdl.addGenConstrPWL(mu_var[d,i], beta_am_aux[i,d], pts, bam, name=f"PWL_bAM_{d}_{i}")
            mdl.addGenConstrPWL(mu_var[d,i], C_EOQ_var[d,i], pts, eoq, name=f"PWL_EOQ_{d}_{i}")

            # PWL backorder analitico da lead time
            bo_cm_pts = [b_i[i] * expected_backorder(pts[m]*L_CM[i], bcm[m]) for m in range(len(pts))]
            bo_am_pts = [b_i[i] * expected_backorder(pts[m]*L_AM[i,d], bam[m]) for m in range(len(pts))]
            mdl.addGenConstrPWL(mu_var[d,i], bo_cm_aux[i,d], pts, bo_cm_pts, name=f"PWL_boCM_{d}_{i}")
            mdl.addGenConstrPWL(mu_var[d,i], bo_am_aux[i,d], pts, bo_am_pts, name=f"PWL_boAM_{d}_{i}")

    mdl.addConstrs((B[i,d]>=beta_cm_aux[i,d]-B_bar[i,d]*(1-cm[i]) for i in I for d in D), name="DG4a")
    mdl.addConstrs((B[i,d]>=beta_am_aux[i,d]-B_bar[i,d]*(1-am[i]) for i in I for d in D), name="DG4b")
    mdl.addConstrs((B[i,d]<=B_bar[i,d]*dc[d,i] for i in I for d in D), name="DG5")
    
    if B_cap is not None:
        mdl.addConstrs((gp.quicksum(B[i,d] for i in I)<=B_cap for d in D), name="DG5b")
        if verbose:
            print(f"  [vincolo] B_cap={B_cap} attivo — max {B_cap} unità BS per DC")

    # ── [NUOVO] Costo BS tecnologia-specifico (McCormick su B×w e B×v) ────────────
    B_cm_lin = mdl.addVars(nI, nD, lb=0.0, name="B_cm_lin")
    B_am_lin = mdl.addVars(nI, nD, lb=0.0, name="B_am_lin")
    for i in I:
        for d in D:
            Bub = float(B_bar[i,d])
            # McCormick B × w
            mdl.addConstr(B_cm_lin[i,d] <= Bub * w[i,d])
            mdl.addConstr(B_cm_lin[i,d] <= B[i,d])
            mdl.addConstr(B_cm_lin[i,d] >= B[i,d] - Bub*(1-w[i,d]))
            # McCormick B × v
            mdl.addConstr(B_am_lin[i,d] <= Bub * v[i,d])
            mdl.addConstr(B_am_lin[i,d] <= B[i,d])
            mdl.addConstr(B_am_lin[i,d] >= B[i,d] - Bub*(1-v[i,d]))

    for d in D:
        for i in I: mdl.addConstr(dcAM[d]>=v[i,d], name=f"dcAM_lo_{d}_{i}")
        mdl.addConstr(dcAM[d]<=gp.quicksum(v[i,d] for i in I), name=f"dcAM_up_{d}")

    M_raw_big = float(max(qbar_i)*sum(mu_ic[i,:].sum() for i in I))*2
    for d in D:
        for i in I:
            mdl.addConstr(mu_raw_contrib[d,i]<=M_raw_big*v[i,d])
            mdl.addConstr(mu_raw_contrib[d,i]<=qbar_i[i]*mu_var[d,i])
            mdl.addConstr(mu_raw_contrib[d,i]>=qbar_i[i]*mu_var[d,i]-M_raw_big*(1-v[i,d]))
        mdl.addConstr(mu_raw_total[d]==gp.quicksum(mu_raw_contrib[d,i] for i in I))
        mdl.addConstr(mu_raw_total[d]<=M_raw_big*dcAM[d])

    mu_raw_max = float(sum(qbar_i[i]*mu_ic[i,:].sum() for i in I))
    mu_raw_pts = np.linspace(0.0, max(mu_raw_max,0.01), 30).tolist()
    for d in D:
        brp = [float(beta_poisson(mu, L_raw, SL_raw)) for mu in mu_raw_pts]
        erp = [float(np.sqrt(2*o_raw*h_d[d]*c_raw*mu)) if mu>0 else 0.0 for mu in mu_raw_pts]
        bra = mdl.addVar(lb=0.0, name=f"b_raw_aux_{d}")
        mdl.addGenConstrPWL(mu_raw_total[d], bra, mu_raw_pts, brp, name=f"PWL_Braw_{d}")
        mdl.addConstr(B_raw[d]>=bra); mdl.addConstr(B_raw[d]<=B_bar_raw[d]*dcAM[d])
        mdl.addGenConstrPWL(mu_raw_total[d], C_EOQ_raw_var[d], mu_raw_pts, erp, name=f"PWL_EOQraw_{d}")

    M_EOQ_big = float(C_EOQ_grid.max())*2
    C_EOQ_act = mdl.addVars(nD, nI, lb=0.0, name="C_EOQ_act")
    for d in D:
        for i in I:
            mdl.addConstr(C_EOQ_act[d,i]<=M_EOQ_big*w[i,d])
            mdl.addConstr(C_EOQ_act[d,i]<=C_EOQ_var[d,i])
            mdl.addConstr(C_EOQ_act[d,i]>=C_EOQ_var[d,i]-M_EOQ_big*(1-w[i,d]))

    ratio_pt = np.where(qbar_i>1e-9, ptime_i/qbar_i, 0.0)
    mdl.addConstrs((n3DP[d]>=gp.quicksum(ratio_pt[i]*mu_raw_contrib[d,i] for i in I)/Cap_3DP for d in D))

    # Linearizzazione Big-M per backorder LT (attiva solo la tecnologia scelta)
    M_BO_big = max(max(b_i[i]*mu_max_i[i] for i in I), 1.0) * 2
    BO_act = mdl.addVars(nD, nI, lb=0.0, name="BO_act")
    for d in D:
        for i in I:
            mdl.addConstr(BO_act[d,i] >= bo_cm_aux[i,d] - M_BO_big*(1-cm[i]))
            mdl.addConstr(BO_act[d,i] >= bo_am_aux[i,d] - M_BO_big*(1-am[i]))
            mdl.addConstr(BO_act[d,i] <= bo_cm_aux[i,d] + M_BO_big*(1-cm[i]))
            mdl.addConstr(BO_act[d,i] <= bo_am_aux[i,d] + M_BO_big*(1-am[i]))
            mdl.addConstr(BO_act[d,i] <= M_BO_big*dc[d,i])

    # ══ FIX HOLDING TECNOLOGIA-SPECIFICO e RIMOZIONE DOPPIO CONTEGGIO RAW ══
    cost_BS = gp.quicksum(
        c_CM[i]*(1+h_d[d])*B_cm_lin[i,d] + prod_i[i]*(1+h_d[d])*B_am_lin[i,d]
        for i in I for d in D)
    cost_EOQ = gp.quicksum(C_EOQ_act[d,i] for i in I for d in D)
    cost_raw_dc = gp.quicksum(C_EOQ_raw_var[d]+c_raw*(1+h_d[d])*B_raw[d] for d in D)
    cost_leas = gp.quicksum(Leas*n3DP[d] for d in D)
    cost_bo_lt = gp.quicksum(BO_act[d,i] for i in I for d in D)

    # Secondo stadio
    mdl.addConstrs((gp.quicksum(a_i[i]*q_AM[i,d,s] for i in I)<=Cap_3DP*n3DP[d] for d in D for s in S))
    mdl.addConstrs((B[i,d]+q_CM[i,d,s]+q_AM[i,d,s]-gp.quicksum(f[i,d,c,s] for c in C)==I_inv[i,d,s]
                    for i in I for d in D for s in S))
    mdl.addConstrs((gp.quicksum(f[i,d,c,s] for d in D)+u[i,c,s]==d_ics[i,c,s]
                    for i in I for c in C for s in S))
    mdl.addConstrs((f[i,d,c,s]<=F_bar[i,d]*dc[d,i] for i in I for d in D for c in C for s in S))
    mdl.addConstrs((q_CM[i,d,s]<=Q_bar_CM[i,d]*w[i,d] for i in I for d in D for s in S))
    mdl.addConstrs((q_AM[i,d,s]<=Q_bar_AM[i,d]*v[i,d] for i in I for d in D for s in S))
    mdl.addConstrs((gp.quicksum(pi_s*gp.quicksum(f[i,d,c,s] for d in D) for s in S)
                    >=alpha_ic[i,c]*gp.quicksum(pi_s*d_ics[i,c,s] for s in S)
                    for i in I for c in C))

    # ══ FIX HOLDING secondo stadio: h_d*c_CM*I_inv ══
    cost_2nd = gp.quicksum(pi_s*(
        gp.quicksum(c_CM[i]*q_CM[i,d,s] for i in I for d in D)+
        gp.quicksum(prod_i[i]*q_AM[i,d,s] for i in I for d in D)+
        gp.quicksum(t_dic[d,i,c]*f[i,d,c,s] for i in I for d in D for c in C)+
        gp.quicksum(h_d[d]*c_CM[i]*I_inv[i,d,s] for i in I for d in D)+
        gp.quicksum(b_i[i]*u[i,c,s] for i in I for c in C)
    ) for s in S)

    mdl.setObjective(cost_BS+cost_EOQ+cost_raw_dc+cost_leas+cost_bo_lt+cost_2nd, GRB.MINIMIZE)

    # VarHint
    def _bp(mu, L, sl):
        lam = mu*L
        if lam<1e-9: return 0
        for ss in range(10000):
            if poisson.cdf(ss, lam)>=sl: return ss
        return 9999
    for i in I:
        mu_c = float(mu_ic[i,:].sum())
        b_cm_h = _bp(mu_c, L_CM[i], SL_i[i])
        b_am_h = _bp(mu_c, L_AM[i,0], SL_i[i])
        risp = c_CM[i]*(1+h_d[0])*b_cm_h - prod_i[i]*(1+h_d[0])*b_am_h
        bo_cm = b_i[i]*expected_backorder(mu_c*L_CM[i], b_cm_h)
        bo_am = b_i[i]*expected_backorder(mu_c*L_AM[i,0], b_am_h)
        risp += bo_cm - bo_am
        costo = (prod_i[i]-c_CM[i])*mu_c
        netto = risp - costo
        if netto>0:
            am[i].VarHintVal=1.0; am[i].VarHintPri=10; cm[i].VarHintVal=0.0
        elif -Leas<netto<=0:
            am[i].VarHintVal=1.0; am[i].VarHintPri=5; cm[i].VarHintVal=0.0
        else:
            am[i].VarHintVal=0.0; am[i].VarHintPri=1; cm[i].VarHintVal=1.0
    mdl.update()

    # ══ WARM-START MINIMALE: solo binarie + n3DP ══
    if warm_start_sol is not None:
        ws_am  = warm_start_sol['am']
        ws_dc  = warm_start_sol['dc']
        ws_y   = warm_start_sol.get('y_sol', warm_start_sol.get('y'))
        ws_n3dp = warm_start_sol.get('n3dp', warm_start_sol.get('n3DP'))

        for i in I:
            am[i].Start = float(ws_am[i]); cm[i].Start = float(1-ws_am[i])
        for d in D:
            has_am_d = False
            for i in I:
                dc[d,i].Start = float(ws_dc[d,i])
                v[i,d].Start = float(ws_am[i] and ws_dc[d,i])
                w[i,d].Start = float((not ws_am[i]) and ws_dc[d,i])
                if ws_am[i] and ws_dc[d,i]: has_am_d = True
                for c in C:
                    y[d,i,c].Start = float(ws_y[d,i,c] > 0.5)
            dcAM[d].Start = float(has_am_d)

        # n3DP: coprire picchi SAA
        for d in D:
            n3dp_peak = 0
            for s in S:
                hours_s = 0.0
                for i in I:
                    if not (ws_am[i] and ws_dc[d,i]): continue
                    dem_s = sum(d_ics[i,c,s] for c in C if ws_y[d,i,c] > 0.5)
                    beta_ws = float(warm_start_sol['B'][i,d])
                    hours_s += a_i[i] * max(0.0, dem_s - beta_ws)
                n3dp_peak = max(n3dp_peak, int(np.ceil(hours_s/Cap_3DP)) if hours_s>0 else 0)
            n3DP[d].Start = float(max(ws_n3dp[d], n3dp_peak))

        mdl.setParam("StartNodeLimit", 100)
        mdl.update()
        if verbose:
            print(f"  [warm start] MINIMALE: {sum(int(ws_am[i]) for i in I)} AM, "
                  f"n3DP={[int(n3DP[d].Start) for d in D]}")

    mdl.optimize()
    solve_time = time.time() - t_start

    if mdl.Status not in [GRB.OPTIMAL, GRB.TIME_LIMIT, GRB.SUBOPTIMAL]:
        return {'status':'INFEASIBLE','obj':None,'obj_real':None,'gap':None,
                'time':solve_time,'am':None,'cm':None,'dc':None,'B':None,
                'n3DP':None,'cost_breakdown':None,'y_sol':None}

    status = {GRB.OPTIMAL:'OPTIMAL',GRB.TIME_LIMIT:'TIME_LIMIT',
              GRB.SUBOPTIMAL:'SUBOPTIMAL'}.get(mdl.Status, str(mdl.Status))

    def snap(val): return val if val>0.5 else 0.0
    am_sol = np.array([am[i].X>0.5 for i in I])
    cm_sol = np.array([cm[i].X>0.5 for i in I])
    dc_sol = np.array([[dc[d,i].X>0.5 for i in I] for d in D])
    B_sol  = np.array([[B[i,d].X for d in D] for i in I])
    n3DP_sol = np.array([max(0.0, n3DP[d].X) for d in D])
    w_sol = np.array([[w[i,d].X for d in D] for i in I])
    v_sol = np.array([[v[i,d].X for d in D] for i in I])
    y_sol = np.array([[[snap(y[d,i,c].X) for c in C] for i in I] for d in D])
    n_dc_per_sku = dc_sol.sum(axis=0)

    # Post-processing LP
    mu_var_sol = np.array([[sum(mu_ic[i,c]*y_sol[d,i,c] for c in C) for i in I] for d in D])
    C_EOQ_act_sol = np.zeros((nD,nI))
    for d in D:
        for i in I:
            if w_sol[i,d]>0.5:
                m = int(np.argmin(np.abs(mu_grid-mu_var_sol[d,i])))
                C_EOQ_act_sol[d,i] = C_EOQ_grid[i,d,m]
    mu_raw_sol = np.array([[qbar_i[i]*mu_var_sol[d,i]*float(v_sol[i,d]>0.5) for i in I] for d in D])
    dcAM_sol = np.array([any(v_sol[i,d]>0.5 for i in I) for d in D])
    C_EOQ_raw_sol = np.zeros(nD); B_raw_sol_arr = np.zeros(nD)
    for d in D:
        if dcAM_sol[d]:
            mu_raw_d = mu_raw_sol[d,:].sum()
            m = int(np.argmin(np.abs(mu_raw_grid-mu_raw_d)))
            C_EOQ_raw_sol[d] = C_EOQ_raw_grid[d,m]; B_raw_sol_arr[d] = B_raw_grid[d,m]

    cost_BS_sol = sum(
        c_CM[i]*(1+h_d[d])*B_sol[i,d] * (1 if w_sol[i,d]>0.5 else 0) +
        prod_i[i]*(1+h_d[d])*B_sol[i,d] * (1 if v_sol[i,d]>0.5 else 0)
        for i in I for d in D
    )
    cost_EOQ_sol = sum(C_EOQ_act_sol[d,i] for d in D for i in I)
    cost_raw_dc_sol = sum(C_EOQ_raw_sol[d]+c_raw*(1+h_d[d])*B_raw_sol_arr[d] for d in D)
    cost_leas_sol = sum(Leas*n3DP_sol[d] for d in D)
    bo_lt_sol = sum(BO_act[d,i].X for i in I for d in D)
    cost_1st = cost_BS_sol+cost_EOQ_sol+cost_raw_dc_sol+cost_leas_sol+bo_lt_sol

    pp = gp.Model("pp"); pp.setParam("OutputFlag",0)
    qp = pp.addVars(nI,nD,nS,lb=0.0); qa = pp.addVars(nI,nD,nS,lb=0.0)
    fp = pp.addVars(nI,nD,nC,nS,lb=0.0); up = pp.addVars(nI,nC,nS,lb=0.0)
    ip = pp.addVars(nI,nD,nS,lb=0.0)
    pp.addConstrs((gp.quicksum(a_i[i]*qa[i,d,s] for i in I)<=Cap_3DP*n3DP_sol[d] for d in D for s in S))
    pp.addConstrs((B_sol[i,d]+qp[i,d,s]+qa[i,d,s]-gp.quicksum(fp[i,d,c,s] for c in C)==ip[i,d,s]
                   for i in I for d in D for s in S))
    pp.addConstrs((gp.quicksum(fp[i,d,c,s] for d in D)+up[i,c,s]==d_ics[i,c,s]
                   for i in I for c in C for s in S))
    pp.addConstrs((fp[i,d,c,s]<=F_bar[i,d]*float(dc_sol[d,i]) for i in I for d in D for c in C for s in S))
    pp.addConstrs((qp[i,d,s]<=Q_bar_CM[i,d]*float(w_sol[i,d]>0.5) for i in I for d in D for s in S))
    pp.addConstrs((qa[i,d,s]<=Q_bar_AM[i,d]*float(v_sol[i,d]>0.5) for i in I for d in D for s in S))
    pp.setObjective(gp.quicksum(pi_s*(
        gp.quicksum(c_CM[i]*qp[i,d,s] for i in I for d in D)+
        gp.quicksum(prod_i[i]*qa[i,d,s] for i in I for d in D)+
        gp.quicksum(t_dic[d,i,c]*fp[i,d,c,s] for i in I for d in D for c in C)+
        gp.quicksum(h_d[d]*c_CM[i]*ip[i,d,s] for i in I for d in D)+
        gp.quicksum(b_i[i]*up[i,c,s] for i in I for c in C)) for s in S), GRB.MINIMIZE)
    pp.optimize()

    if pp.Status==GRB.OPTIMAL:
        cost_breakdown = {
            'base_stock': cost_BS_sol, 'EOQ_CM': cost_EOQ_sol,
            'materia_prima': 0.0,
            'EOQ_raw': sum(C_EOQ_raw_sol), 'holding_raw': sum(c_raw*(1+h_d[d])*B_raw_sol_arr[d] for d in D),
            'leasing_3DP': cost_leas_sol,
            'backorder_LT': bo_lt_sol,
            'trasporto': sum(pi_s*t_dic[d,i,c]*fp[i,d,c,s].X for i in I for d in D for c in C for s in S),
            'produzione_CM': sum(pi_s*c_CM[i]*qp[i,d,s].X for i in I for d in D for s in S),
            'produzione_AM': sum(pi_s*prod_i[i]*qa[i,d,s].X for i in I for d in D for s in S),
            'holding_inv': sum(pi_s*h_d[d]*c_CM[i]*ip[i,d,s].X for i in I for d in D for s in S),
            'backorder_2nd': sum(pi_s*b_i[i]*up[i,c,s].X for i in I for c in C for s in S),
        }
        obj_real = cost_1st+pp.ObjVal
    else:
        cost_breakdown = {}; obj_real = mdl.ObjVal

    return {'status':status,'obj':mdl.ObjVal,'obj_real':obj_real,
            'gap':mdl.MIPGap*100,'time':solve_time,
            'am':am_sol,'cm':cm_sol,'dc':dc_sol,'B':B_sol,
            'n3DP':n3DP_sol,'n_dc_per_sku':n_dc_per_sku,
            'cost_breakdown':cost_breakdown,'y_sol':y_sol,}


# ══════════════════════════════════════════════════════════════════════════
# CG v5 (compatta, con holding corretto per materia prima)
# ══════════════════════════════════════════════════════════════════════════
def column_generation(params, nI, nD, nC, Leas, Cap_3DP=5000, max_iter=80, verbose=True):
    t0 = time.time(); p = params
    table = {}; n_ent = 0
    for i in range(nI):
        table[i] = {}
        for d in range(nD):
            table[i][d] = {}
            for tech in ['CM','AM']:
                table[i][d][tech] = {}
                for mask in range(1,2**nC):
                    clients = [c for c in range(nC) if mask&(1<<c)]
                    mu_id = sum(p['mu_ic'][i,c] for c in clients)
                    if mu_id<1e-9: continue
                    trans = sum(p['t_dic'][d,i,c]*p['mu_ic'][i,c] for c in clients)
                    if tech=='CM':
                        mu_L=mu_id*p['L_CM'][i]; ch=p['c_CM'][i]*(1+p['h_d'][d])
                        beta,bsc=optimal_beta(mu_L,ch,p['b_i'][i])
                        cost=bsc+np.sqrt(2*p['o']*p['h_d'][d]*p['c_CM'][i]*mu_id)+p['c_CM'][i]*mu_id+trans+p['h_d'][d]*p['c_CM'][i]*expected_on_hand(mu_L,beta)
                        ph=0
                    else:
                        L_am=p['L_AM'][:,0][i]; mu_L=mu_id*L_am
                        c_am=p['c_CM'][i]*p['n_i'][i]
                        ch=c_am*(1+p['h_d'][d]) 
                        beta,bsc=optimal_beta(mu_L,ch,p['b_i'][i])
                        prod=c_am*mu_id
                        vol=c_am/(1.30*1e6); qr=vol*p['den_raw']/p['unit_raw']
                        
                        # -- CORREZIONE MATERIA PRIMA --
                        mu_raw = qr*mu_id
                        eoq_raw = np.sqrt(2*p['o_raw']*p['h_d'][d]*p['c_raw']*max(mu_raw,1e-9))
                        
                        lam_raw = mu_raw * p['L_raw']
                        beta_raw = 0
                        if lam_raw > 1e-9:
                            for s in range(10000):
                                if poisson.cdf(s, lam_raw) >= p['SL_raw']:
                                    beta_raw = s
                                    break
                        bsc_raw = p['c_raw']*(1+p['h_d'][d])*beta_raw
                        
                        cost = bsc + prod + eoq_raw + bsc_raw + trans + p['h_d'][d]*p['c_CM'][i]*expected_on_hand(mu_L,beta)
                        ph = c_am/(1.30*0.00525*3600)*mu_id
                        
                    table[i][d][tech][mask]=(cost,mu_id,beta,ph); n_ent+=1
    print(f"  [CG] Precomputazione: {n_ent:,} entry in {time.time()-t0:.1f}s")
    columns=[]; fm=(1<<nC)-1
    for i in range(nI):
        for d in range(nD):
            for tech in ['CM','AM']:
                if fm in table[i][d][tech]:
                    columns.append((d,i,tech,fm)+table[i][d][tech][fm])
    existing=set((c[0],c[1],c[2],c[3]) for c in columns)
    mdl=gp.Model("CG"); mdl.Params.OutputFlag=0; mdl.Params.Method=1
    n3dp={d:mdl.addVar(lb=0) for d in range(nD)}
    cover={(i,c):mdl.addConstr(gp.LinExpr()==1) for i in range(nI) for c in range(nC)}
    cap_c={d:mdl.addConstr(-Cap_3DP*n3dp[d]<=0) for d in range(nD)}
    mdl.setObjective(gp.quicksum(Leas*n3dp[d] for d in range(nD)),GRB.MINIMIZE); mdl.update()
    lv=[]
    for col in columns:
        dc_r,sk,te,mk,co,mu,be,ph=col; gc=gp.Column()
        for c in range(nC):
            if mk&(1<<c): gc.addTerms(1.0,cover[sk,c])
        if te=='AM' and ph>0: gc.addTerms(ph,cap_c[dc_r])
        lv.append(mdl.addVar(lb=0,ub=1,obj=co,column=gc)); mdl.update()
    lb=0
    for it in range(max_iter):
        mdl.optimize()
        if mdl.Status!=2: break
        lb=mdl.ObjVal; duals=np.zeros((nI,nC))
        for i in range(nI):
            for c in range(nC): duals[i,c]=cover[i,c].Pi
        wv=np.array([n3dp[d].X for d in range(nD)]); new=[]
        for i in range(nI):
            for d in range(nD):
                for tech in ['CM','AM']:
                    brc,bm,be=-1e12,0,None
                    for mask,entry in table[i][d][tech].items():
                        ds=sum(duals[i,c] for c in range(nC) if mask&(1<<c)); rc=ds-entry[0]
                        if rc>brc: brc,bm,be=rc,mask,entry
                    if be is None: continue
                    ra=brc
                    if tech=='AM' and wv[d]<0.5: ra-=Leas*(1-wv[d])
                    if ra>1e-4:
                        key=(d,i,tech,bm)
                        if key not in existing:
                            new.append((d,i,tech,bm)+be); existing.add(key)
        for nc in new:
            dc_r,sk,te,mk,co,mu,be,ph=nc; columns.append(nc); gc=gp.Column()
            for c in range(nC):
                if mk&(1<<c): gc.addTerms(1.0,cover[sk,c])
            if te=='AM' and ph>0: gc.addTerms(ph,cap_c[dc_r])
            lv.append(mdl.addVar(lb=0,ub=1,obj=co,column=gc)); mdl.update()
        if verbose and (it%5==0 or len(new)==0):
            print(f"  [CG] It {it:>3}: LB={lb:>10,.0f}  col={len(columns):>5}  new={len(new):>3}")
        if len(new)==0: break
    mdl.dispose()
    m2=gp.Model("MILP"); m2.Params.OutputFlag=1; m2.Params.TimeLimit=300
    l2=m2.addVars(len(columns),vtype=GRB.BINARY); n2=m2.addVars(nD,vtype=GRB.INTEGER,lb=0)
    for i in range(nI):
        for c in range(nC):
            m2.addConstr(gp.quicksum(l2[r] for r in range(len(columns)) if columns[r][1]==i and columns[r][3]&(1<<c))==1)
    for d in range(nD):
        m2.addConstr(gp.quicksum(columns[r][7]*l2[r] for r in range(len(columns)) if columns[r][0]==d and columns[r][2]=='AM' and columns[r][7]>0)<=Cap_3DP*n2[d])
    m2.setObjective(gp.quicksum(columns[r][4]*l2[r] for r in range(len(columns)))+gp.quicksum(Leas*n2[d] for d in range(nD)),GRB.MINIMIZE)
    m2.optimize()
    sel,obj,n3=[],float('inf'),[0]*nD
    if m2.Status in [GRB.OPTIMAL,GRB.TIME_LIMIT]:
        sel=[r for r in range(len(columns)) if l2[r].X>0.5]; obj=m2.ObjVal
        n3=[int(round(n2[d].X)) for d in range(nD)]
    m2.dispose()
    am_a=np.zeros(nI,dtype=int); dc_s=np.zeros((nD,nI),dtype=int)
    y_s=np.zeros((nD,nI,nC),dtype=int); B_s=np.zeros((nI,nD))
    for r in sel:
        d,i,te,mk,co,mu,be,ph=columns[r]
        if te=='AM': am_a[i]=1
        dc_s[d,i]=1; B_s[i,d]=be
        for c in range(nC):
            if mk&(1<<c): y_s[d,i,c]=1
    al=[i for i in range(nI) if am_a[i]]
    if verbose: print(f"\n  CG: {len(al)} AM {al}, costo={obj:,.0f}, n3dp={n3}, t={time.time()-t0:.1f}s")
    return {'am':am_a,'dc':dc_s,'y':y_s,'B':B_s,'n3dp':n3,'obj_milp':obj,'time':time.time()-t0}


def print_results(res,name,nI,nD,nC,c_CM,c_AM,b_i,L_CM,L_AM):
    print(f"\n{'='*70}\n{name}\n{'='*70}")
    if res['status']=='INFEASIBLE': print("  INFEASIBLE"); return
    print(f"  Status: {res['status']}  |  Costo: {res['obj_real']:,.2f}€  |  Gap: {res['gap']:.2f}%  |  Tempo: {res['time']:.1f}s")
    print(f"\n  {'SKU':<4} {'T':<3} {'DC':<12} {'B':>5} {'c_CM':>7} {'c_AM':>7} {'ratio':>6}")
    for i in range(nI):
        t="AM" if res['am'][i] else "CM"; dcs=[d for d in range(nD) if res['dc'][d,i]]
        bm=res['B'][i,dcs].mean() if dcs else 0
        print(f"  {i:<4} {t:<3} {str(dcs):<12} {bm:>5.1f} {c_CM[i]:>7.0f} {c_AM[i]:>7.0f} {c_AM[i]/c_CM[i]:>6.3f}")
    print(f"\n  Stampanti: {np.round(res['n3DP'],1)}")
    if res.get('cost_breakdown'):
        print(f"  Costi:")
        tot=0
        for k,v in res['cost_breakdown'].items():
            if abs(v)>0.01: print(f"    {k:<20}: {v:>12.2f}€"); tot+=v
        print(f"    {'TOTALE':<20}: {tot:>12.2f}€")


# ══════════════════════════════════════════════════════════════════════════
# PARAMETRI 9 SKU REALI
# ══════════════════════════════════════════════════════════════════════════
nI,nD,nC = 9,3,6
c_CM = np.array([1999.785,2373.285,3218.394,3774.222,4004.102,4384.373,5676.740,6073.243,6785.084])
c_AM = np.array([2632.362,2936.770,3366.513,3588.991,4332.874,5135.218,5387.065,7633.061,7633.061])
L_CM = np.array([13.,13.,13.,14.,16.,16.,16.,16.,17.])/52
L_AM_vec = np.array([3.,4.,5.,6.,7.,7.,8.,9.,9.596491228])/52
L_AM = np.column_stack([L_AM_vec]*nD)
ratio = c_AM/c_CM; n_i = ratio
coord_DC = np.array([[1.0,8.0],[1.0,2.0],[5.0,5.0]])
coord_C = np.array([[0.5,9.0],[2.0,8.0],[3.5,5.5],[5.5,4.5],[1.5,2.5],[0.5,1.0]])
mu_ic = np.array([[0.8,0.6,0.5,0.4,0.4,0.3],[0.7,0.5,0.5,0.4,0.3,0.3],
    [0.6,0.5,0.4,0.4,0.3,0.3],[0.5,0.4,0.4,0.3,0.3,0.2],
    [0.5,0.4,0.3,0.3,0.3,0.2],[0.4,0.4,0.3,0.3,0.2,0.2],
    [0.4,0.3,0.3,0.2,0.2,0.2],[0.3,0.3,0.2,0.2,0.2,0.2],
    [0.3,0.3,0.2,0.2,0.2,0.1]])
b_i = (c_CM*25).round(0)
SL_i = np.array([0.97,0.97,0.96,0.96,0.95,0.95,0.95,0.94,0.94])
alpha_ic = np.column_stack([SL_i]*nC)
Leas=15000.0; Cap_3DP=5000.0; h_d=np.full(nD,0.25)
o=50.0; o_raw=30.0; c_raw=50.0; den_raw=1200.0; unit_raw=0.5
L_raw=1.0/52; SL_raw=0.85; costo_km=80.0
a_i = c_AM/(1.30*0.00525*3600)
dist = np.array([[np.sqrt((coord_DC[d,0]-coord_C[c,0])**2+(coord_DC[d,1]-coord_C[c,1])**2)
    for c in range(nC)] for d in range(nD)])
t_dic = np.stack([dist*costo_km]*nI, axis=1)

# ══════════════════════════════════════════════════════════════════════════
# FUNZIONE PIPELINE CG → MILP (per sensitività)
# ══════════════════════════════════════════════════════════════════════════

def run_pipeline(Leas_val=15000., ratio_mult=1.0, L_CM_mult=1.0,
                 b_i_mult=1.0, SL_val=None, B_cap_val=None,
                 nS=50, mip_gap=0.05, time_limit=3600, verbose=False):
    """
    Esegue CG → MILP warm-start con i parametri specificati.
    I moltiplicatori si applicano ai valori base del caso 9 SKU.
    Ritorna un dict con tutti i risultati + metriche per ANOVA.
    """
    # Parametri modificabili
    Leas_run   = Leas_val
    c_AM_run   = c_CM * ratio_mult * (c_AM / c_CM)  # scala il ratio mantenendo la struttura
    n_i_run    = c_AM_run / c_CM
    L_CM_run   = L_CM * L_CM_mult
    b_i_run    = b_i * b_i_mult
    SL_run     = SL_val if SL_val is not None else SL_i
    if np.isscalar(SL_run):
        SL_run = np.full(nI, SL_run)
    alpha_run  = np.column_stack([SL_run] * nC)
    a_i_run    = c_AM_run / (1.30 * 0.00525 * 3600)

    label = (f"Leas={Leas_run:.0f} ratio×{ratio_mult:.1f} "
             f"LCM×{L_CM_mult:.1f} bi×{b_i_mult:.1f} "
             f"SL={SL_run[0]:.2f}")
    print(f"\n{'='*72}")
    print(f"RUN: {label}")
    print(f"{'='*72}")

    kwargs_run = dict(
        nI=nI, nD=nD, nC=nC, mu_ic=mu_ic, alpha_ic=alpha_run,
        SL=SL_run, SL_raw=SL_raw, c_CM=c_CM, o=o, h_d=h_d,
        b_i=b_i_run, L_CM=L_CM_run, n_i=n_i_run,
        c_raw=c_raw, den_raw=den_raw, unit_raw=unit_raw,
        o_raw=o_raw, L_raw=L_raw, L_AM=L_AM, Leas=Leas_run,
        Cap_3DP=Cap_3DP, a_i=a_i_run,
        t_central=0., t_dic=t_dic, xi=2.0, seed=42,
        B_cap=B_cap_val,
    )

    # CG
    cg_p = dict(c_CM=c_CM, n_i=n_i_run, L_CM=L_CM_run, L_AM=L_AM,
        SL=SL_run, mu_ic=mu_ic, h_d=h_d, b_i=b_i_run, t_dic=t_dic,
        Leas=Leas_run, o=o, o_raw=o_raw, c_raw=c_raw,
        den_raw=den_raw, unit_raw=unit_raw, 
        L_raw=L_raw, SL_raw=SL_raw) # <-- AGGIUNTI L_raw e SL_raw per la scorta materia prima
        
    res_cg = column_generation(cg_p, nI=nI, nD=nD, nC=nC,
                               Leas=Leas_run, Cap_3DP=Cap_3DP,
                               verbose=verbose)

    # MILP con warm-start
    res = solve_stochastic(**kwargs_run, nS=nS, mip_gap=mip_gap,
                           time_limit=time_limit,
                           warm_start_sol=res_cg, verbose=verbose,
                           scenario_name=label[:30])
    print_results(res, label, nI, nD, nC, c_CM, c_AM_run, b_i_run,
                  L_CM_run, L_AM)

    # Metriche per ANOVA
    if res['am'] is not None:
        n_am = int(res['am'].sum())
        # n_DC_CM medio (solo SKU CM)
        cm_skus = [i for i in range(nI) if not res['am'][i]]
        if cm_skus:
            n_dc_cm = np.mean([res['n_dc_per_sku'][i] for i in cm_skus])
        else:
            n_dc_cm = 0.0
        # Entropia Shannon media
        mu_var_sol = np.array([[sum(mu_ic[i,c]*res['y_sol'][d,i,c]
                                    for c in range(nC))
                                for i in range(nI)] for d in range(nD)])
        H_vals = []
        for i in range(nI):
            mu_dc = mu_var_sol[:,i]
            mu_tot = mu_dc.sum()
            if mu_tot > 1e-9:
                p = mu_dc / mu_tot
                p = p[p > 1e-9]
                H_vals.append(-np.sum(p * np.log(p)))
            else:
                H_vals.append(0.0)
        H_mean = np.mean(H_vals)
    else:
        n_am, n_dc_cm, H_mean = None, None, None

    # Stampa dettagliata assegnazione clienti e base stock per DC
    if res['am'] is not None:
        print(f"\n  DETTAGLIO ASSEGNAZIONE CLIENTI (y_sol):")
        for i in range(nI):
            tech = "AM" if res['am'][i] else "CM"
            dcs = [d for d in range(nD) if res['dc'][d,i]]
            for d in dcs:
                clients = [c for c in range(nC) if res['y_sol'][d,i,c] > 0.5]
                mu_d = sum(mu_ic[i,c] for c in clients)
                print(f"    SKU {i} ({tech}) → DC{d}: clienti={clients}, "
                      f"μ_DC={mu_d:.2f}, B={res['B'][i,d]:.1f}")

        print(f"\n  DETTAGLIO BASE STOCK PER DC:")
        for d in range(nD):
            skus_d = [i for i in range(nI) if res['dc'][d,i]]
            if not skus_d: continue
            b_tot = sum(res['B'][i,d] for i in skus_d)
            n_am_d = sum(1 for i in skus_d if res['am'][i])
            violation = ""
            if B_cap_val is not None and b_tot > B_cap_val + 0.01:
                violation = f"  ⚠ VIOLA B_cap={B_cap_val}!"
            print(f"    DC{d}: {len(skus_d)} SKU ({n_am_d} AM), "
                  f"B_tot={b_tot:.1f}, stampanti={res['n3DP'][d]:.0f}{violation}")

        # Backorder LT per SKU
        if res.get('cost_breakdown') and 'backorder_LT' in res['cost_breakdown']:
            print(f"\n  BACKORDER LT ANALITICO PER SKU:")
            print(f"    {'SKU':<5} {'Tech':<4} {'bo_LT':>10} {'b_i':>10} "
                  f"{'L_tech':>8} {'μ_tot':>6} {'β':>4} {'E[BO]':>8}")
            for i in range(nI):
                tech = "AM" if res['am'][i] else "CM"
                dcs = [d for d in range(nD) if res['dc'][d,i]]
                mu_tot = sum(mu_ic[i,c] for c in range(nC))
                for d in dcs:
                    mu_d = sum(mu_ic[i,c] for c in range(nC)
                               if res['y_sol'][d,i,c] > 0.5)
                    L_t = L_CM[i] if not res['am'][i] else L_AM[i,d]
                    beta_d = res['B'][i,d]
                    ebo = expected_backorder(mu_d * L_t, beta_d)
                    bo_cost = b_i[i] * ebo
                    print(f"    {i:<5} {tech:<4} {bo_cost:>10.2f} {b_i[i]:>10.0f} "
                          f"{L_t:>8.4f} {mu_d:>6.2f} {beta_d:>4.0f} {ebo:>8.4f}")

    result = {
        'label': label,
        'Leas': Leas_run, 'ratio_mult': ratio_mult,
        'L_CM_mult': L_CM_mult, 'b_i_mult': b_i_mult,
        'SL': SL_run[0], 'B_cap_val': B_cap_val,
        'n_am': n_am, 'n_dc_cm': n_dc_cm, 'H_mean': H_mean,
        'cost': res['obj_real'], 'gap': res['gap'],
        'time_cg': res_cg['time'], 'time_milp': res['time'],
        'cg_am': int(res_cg['am'].sum()),
        'res': res,
    }
    print(f"\n  METRICHE ANOVA: n_AM={n_am}, n_DC_CM={n_dc_cm:.2f}, "
          f"H̄={H_mean:.3f}, costo={res['obj_real']:,.0f}€")
    return result


# ══════════════════════════════════════════════════════════════════════════
# ESECUZIONE — Caso reale con backorder LT
# ══════════════════════════════════════════════════════════════════════════

results = []

# Run con backorder LT, Leas=15k, no B_cap
results.append(run_pipeline(Leas_val=15000, nS=30, time_limit=5400, verbose=True))

# Run con backorder LT + B_cap=5
results.append(run_pipeline(Leas_val=15000, B_cap_val=5, nS=30, time_limit=5400, verbose=True))

# Run con Leas=30k per vedere la transizione
results.append(run_pipeline(Leas_val=30000, nS=30, time_limit=5400, verbose=True))

# ── TABELLA RIASSUNTIVA ──
print("\n" + "="*90)
print("TABELLA RIASSUNTIVA — CON BACKORDER LT ANALITICO")
print("="*90)
print(f"\n  {'Leas':>8} {'B_cap':>6} {'n_AM':>5} {'nDC_CM':>7} {'H̄':>6} "
      f"{'Costo':>12} {'Gap':>6} {'t(s)':>6}")
print(f"  {'-'*65}")
for r in results:
    if r['n_am'] is not None:
        bcap = r.get('B_cap_val', '—')
        print(f"  {r['Leas']:>8.0f} {str(bcap):>6} {r['n_am']:>5} "
              f"{r['n_dc_cm']:>7.2f} {r['H_mean']:>6.3f} "
              f"{r['cost']:>12,.0f} {r['gap']:>5.1f}% {r['time_milp']:>6.0f}")

========================================================================
RUN: Leas=15000 ratio×1.0 LCM×1.0 bi×1.0 SL=0.97
========================================================================
  [CG] Precomputazione: 3,402 entry in 3.1s
Set parameter Username
Set parameter LicenseID to value 2800035
Academic license - for non-commercial use only - expires 2027-03-30
  [CG] It   0: LB=   181,446  col=   94  new= 40
  [CG] It   5: LB=   181,446  col=  257  new= 24
  [CG] It  10: LB=   181,446  col=  282  new=  0
Set parameter OutputFlag to value 1
Set parameter TimeLimit to value 300
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: 11th Gen Intel(R) Core(TM) i7-1165G7 @ 2.80GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  300

Optimize a model with 57 rows, 285 columns and 995 nonzeros (Min)
Model fingerprint: 0x4cef3762
Model has 285 linear objective coefficients
Variable types: 0 continuous, 285 integer (282 binary)
Coefficient statistics:
  Matrix range     [1e+00, 5e+03]
  Objective range  [5e+03, 3e+04]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]

Found heuristic solution: objective 220480.86327
Presolve removed 45 rows and 255 columns
Presolve time: 0.00s
Presolved: 12 rows, 30 columns, 57 nonzeros
Found heuristic solution: objective 220127.63957
Variable types: 0 continuous, 30 integer (30 binary)
Found heuristic solution: objective 202122.86629

Root relaxation: objective 1.868098e+05, 7 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

*    0     0               0    186809.81261 186809.813  0.00%     -    0s

Explored 1 nodes (7 simplex iterations) in 0.03 seconds (0.00 work units)
Thread count was 8 (of 8 available processors)

Solution count 4: 186810 202123 220128 220481 

Optimal solution found (tolerance 1.00e-04)
Best objective 1.868098126074e+05, best bound 1.868098126074e+05, gap 0.0000%

  CG: 9 AM [0, 1, 2, 3, 4, 5, 6, 7, 8], costo=186,810, n3dp=[1, 0, 0], t=3.2s
Set parameter OutputFlag to value 1
Set parameter MIPGap to value 0.05
Set parameter TimeLimit to value 5400
Set parameter StartNodeLimit to value 100
  [warm start] MINIMALE: 9 AM, n3DP=[1, 0, 0]
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: 11th Gen Intel(R) Core(TM) i7-1165G7 @ 2.80GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  5400
MIPGap  0.05
StartNodeLimit  100

Optimize a model with 10125 rows, 9513 columns and 36180 nonzeros (Min)
Model fingerprint: 0xa401c7b1
Model has 9027 linear objective coefficients
Model has 141 simple general constraints
  141 PWL
Variable types: 9246 continuous, 267 integer (264 binary)
Coefficient statistics:
  Matrix range     [4e-03, 5e+05]
  Objective range  [1e+00, 2e+04]
  Bounds range     [1e+00, 1e+00]
  RHS range        [3e-02, 5e+05]
  PWLCon x range   [4e-02, 1e+02]
  PWLCon y range   [0e+00, 1e+04]

Using variable hints.
User MIP start produced solution with objective 149988 (0.08s)
User MIP start produced solution with objective 149988 (0.09s)
Loaded user MIP start with objective 149988

Presolve removed 4702 rows and 1842 columns
Presolve time: 0.14s
Presolved: 5423 rows, 7671 columns, 27198 nonzeros
Presolved model has 141 SOS constraint(s)
Variable types: 7386 continuous, 285 integer (258 binary)

Root relaxation: objective 2.027407e+04, 1372 iterations, 0.03 seconds (0.02 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

     0     0 20274.0738    0  212 149987.512 20274.0738  86.5%     -    0s
     0     0 20274.0738    0  212 149987.512 20274.0738  86.5%     -    0s
     0     0 27895.7352    0  262 149987.512 27895.7352  81.4%     -    0s
     0     0 28025.5974    0  262 149987.512 28025.5974  81.3%     -    0s
     0     0 43629.7190    0  282 149987.512 43629.7190  70.9%     -    0s
     0     0 49946.3127    0  286 149987.512 49946.3127  66.7%     -    1s
     0     0 49946.3127    0  288 149987.512 49946.3127  66.7%     -    1s
     0     0 49946.3127    0  287 149987.512 49946.3127  66.7%     -    1s
     0     0 49946.3127    0  318 149987.512 49946.3127  66.7%     -    1s
     0     0 50205.5409    0  327 149987.512 50205.5409  66.5%     -    1s
     0     0 50212.5683    0  337 149987.512 50212.5683  66.5%     -    1s
     0     0 51182.0425    0  325 149987.512 51182.0425  65.9%     -    1s
     0     0 51618.9583    0  336 149987.512 51618.9583  65.6%     -    1s
     0     0 51623.6786    0  345 149987.512 51623.6786  65.6%     -    1s
     0     0 53394.2628    0  332 149987.512 53394.2628  64.4%     -    2s
     0     0 57332.6173    0  347 149987.512 57332.6173  61.8%     -    2s
     0     0 57332.6173    0  347 149987.512 57332.6173  61.8%     -    2s
     0     0 57332.6173    0  349 149987.512 57332.6173  61.8%     -    2s
     0     0 57332.6173    0  362 149987.512 57332.6173  61.8%     -    2s
     0     0 61660.6723    0  329 149987.512 61660.6723  58.9%     -    2s
     0     0 70147.9339    0  361 149987.512 70147.9339  53.2%     -    2s
     0     0 71926.5794    0  374 149987.512 71926.5794  52.0%     -    2s
     0     0 71926.5794    0  379 149987.512 71926.5794  52.0%     -    2s
     0     0 75290.6129    0  343 149987.512 75290.6129  49.8%     -    3s
     0     0 76478.1316    0  347 149987.512 76478.1316  49.0%     -    3s
     0     0 76510.3791    0  356 149987.512 76510.3791  49.0%     -    3s
     0     0 76517.0956    0  360 149987.512 76517.0956  49.0%     -    3s
     0     0 81959.1652    0  356 149987.512 81959.1652  45.4%     -    3s
     0     0 82255.7805    0  372 149987.512 82255.7805  45.2%     -    4s
     0     0 82546.9578    0  364 149987.512 82546.9578  45.0%     -    4s
     0     0 82593.9538    0  362 149987.512 82593.9538  44.9%     -    4s
     0     0 82598.1003    0  373 149987.512 82598.1003  44.9%     -    4s
     0     0 83347.2910    0  343 149987.512 83347.2910  44.4%     -    4s
     0     0 83956.5356    0  355 149987.512 83956.5356  44.0%     -    4s
     0     0 83956.5356    0  355 149987.512 83956.5356  44.0%     -    4s
     0     0 83980.4104    0  374 149987.512 83980.4104  44.0%     -    4s
     0     0 85748.6423    0  358 149987.512 85748.6423  42.8%     -    4s
     0     0 86031.4786    0  366 149987.512 86031.4786  42.6%     -    5s
     0     0 86074.1841    0  369 149987.512 86074.1841  42.6%     -    5s
     0     0 86078.1888    0  375 149987.512 86078.1888  42.6%     -    5s
     0     0 86390.1688    0  372 149987.512 86390.1688  42.4%     -    5s
     0     0 86412.7674    0  384 149987.512 86412.7674  42.4%     -    5s
     0     0 86561.6303    0  369 149987.512 86561.6303  42.3%     -    5s
     0     0 86574.8507    0  369 149987.512 86574.8507  42.3%     -    5s
     0     0 86574.8507    0  348 149987.512 86574.8507  42.3%     -    5s
     0     2 86574.8507    0  337 149987.512 86574.8507  42.3%     -    5s
   619   444 117174.986   15  181 149987.512 101094.849  32.6%   114   10s
  1244   900 138581.357   37 2388 149987.512 102153.742  31.9%  70.1   15s
  1252   906 124723.242   28 2607 149987.512 102153.742  31.9%  69.7   20s
  1258   910 127213.990   36 2694 149987.512 102153.742  31.9%  69.3   25s
  1263   918 108201.661   16 2455 149987.512 105068.013  29.9%   112   30s
  1508  1002     cutoff   68      149987.512 111694.492  25.5%   125   35s
  1612  1046 124380.345   34 1297 149987.512 113129.905  24.6%   128   40s
  1944  1199 137759.287   84  794 149987.512 114460.992  23.7%   117   45s
  2164  1228 145831.522  115  657 149987.512 114460.992  23.7%   109   50s
  2674  1336 130906.882   20 2129 149987.512 117450.144  21.7%   100   55s
  3287  1693 134938.981   29 1918 149987.512 118151.987  21.2%  90.8   60s
  4083  1773 133135.467   25 2232 149987.512 119040.877  20.6%  80.3   65s
  4923  2322 136475.929   53 1414 149987.512 119551.661  20.3%  72.4   70s
  5542  2865 126204.368   36 1927 149987.512 119738.681  20.2%  68.5   75s
  6367  3196 143072.273   37 2110 149987.512 119873.991  20.1%  64.4   80s
  7159  3729 141905.674   52  906 149987.512 119921.427  20.0%  61.1   86s
  8115  4376 128722.549   34 1394 149987.512 120205.078  19.9%  57.8   92s
  8609  4914 146951.024   66  924 149987.512 120267.971  19.8%  56.5   95s
  8966  4980 144145.267   57 1042 149987.512 120332.143  19.8%  55.7  100s
  9569  5601 148960.774   29 1695 149987.512 120429.767  19.7%  55.0  106s
 10289  6385 138497.621   48 1627 149987.512 120618.360  19.6%  53.9  111s
 11580  6676 149067.865   81  575 149987.512 120853.618  19.4%  52.1  115s
 12286  7038 142723.940   32 2214 149987.512 120990.139  19.3%  51.2  120s
 13388  7701 125735.317   33 2106 149987.512 121435.402  19.0%  50.3  127s
 13608  8174 139177.394   36 1954 149987.512 121588.286  18.9%  50.2  131s
 14150  8715 137964.565   30 1347 149987.512 121723.826  18.8%  50.4  135s
 15624  9120     cutoff   69      149987.512 122128.201  18.6%  49.6  140s
 16803  9877 145151.435   41 1744 149987.512 122407.660  18.4%  49.1  145s
 17709 10934 123397.968   32 2012 149987.512 122763.998  18.2%  48.8  151s
 18495 11411 137746.141   29 1525 149987.512 122833.303  18.1%  48.3  155s
 19850 12143 134256.180   23 2406 149987.512 123321.798  17.8%  47.7  160s
 21012 12548 145720.169   45  207 149987.512 123713.045  17.5%  47.3  222s
 21015 12550 139197.705   61 1880 149987.512 123713.045  17.5%  47.3  226s
 21021 12554 137969.803   63 2706 149987.512 123713.045  17.5%  47.3  234s
 21022 12555 148453.622   56 2722 149987.512 123713.045  17.5%  47.3  235s
 21023 12555 144408.178   66 2742 149987.512 123713.045  17.5%  47.3  240s
 21024 12556 145711.264   53 2692 149987.512 123713.045  17.5%  47.3  246s
 21026 12557 144035.733   53 2726 149987.512 123713.045  17.5%  47.3  254s
 21027 12558 144724.804   67 2702 149987.512 123713.045  17.5%  47.3  259s
 21028 12559 148898.370   65 2753 149987.512 123713.045  17.5%  47.3  264s
 21029 12559 134704.798   35 2707 149987.512 123713.045  17.5%  47.3  270s
 21030 12560 140828.104   51 2749 149987.512 123713.045  17.5%  47.3  276s
 21031 12561 148036.527   34 2725 149987.512 123713.045  17.5%  47.3  282s
 21032 12561 139994.967   63 2746 149987.512 123713.045  17.5%  47.2  287s
 21033 12562 145138.799   49 2712 149987.512 123713.045  17.5%  47.2  293s
 21034 12563 136287.771   42 2709 149987.512 123713.045  17.5%  47.2  298s
 21035 12563 146673.704   43 2695 149987.512 123713.045  17.5%  47.2  303s
 21036 12564 131186.557   41 2742 149987.512 123713.045  17.5%  47.2  308s
 21037 12565 125742.158   32 2692 149987.512 123713.045  17.5%  47.2  314s
 21038 12565 148610.085   69 2730 149987.512 123713.045  17.5%  47.2  322s
 21039 12566 145843.824   48 2690 149987.512 123713.045  17.5%  47.2  327s
 21040 12567 129975.876   39 2730 149987.512 123713.045  17.5%  47.2  333s
 21041 12567 136980.950   51 2706 149987.512 123713.045  17.5%  47.2  338s
 21042 12568 147460.118   56 2747 149987.512 123713.045  17.5%  47.2  344s
 21043 12569 135354.439   43 2692 149987.512 123713.045  17.5%  47.2  351s
 21044 12569 144113.031   71 2727 149987.512 123713.045  17.5%  47.2  357s
 21045 12570 139138.840   62 2702 149987.512 123713.045  17.5%  47.2  362s
 21046 12571 133120.556   42 2747 149987.512 123713.045  17.5%  47.2  368s
 21047 12571 124234.272   32 2721 149987.512 123713.045  17.5%  47.2  374s
 21048 12572 149083.492   60 2764 149987.512 123713.045  17.5%  47.2  379s
 21049 12573 142716.956   44 2743 149987.512 123713.045  17.5%  47.2  385s
 21051 12574 132240.005   38 2721 149987.512 123713.045  17.5%  47.2  395s
 21053 12575 139682.852   63 2711 149987.512 123713.045  17.5%  47.2  404s
 21054 12576 147325.193   53 2752 149987.512 123713.045  17.5%  47.2  408s
 21055 12577 132557.153   40 2734 149987.512 123713.045  17.5%  47.2  413s
 21056 12577 149858.543   86 2754 149987.512 123713.045  17.5%  47.2  417s
 21057 12578 127342.823   33 2747 149987.512 123713.045  17.5%  47.2  421s
 21058 12579 142782.624   30 2763 149987.512 123713.045  17.5%  47.2  425s
 21061 12581 136721.924   51 2763 149987.512 123713.045  17.5%  47.2  430s
 21065 12583 137749.593   59 2764 149987.512 123713.045  17.5%  47.2  436s
 21069 12586 146668.872   43 2733 149987.512 123713.045  17.5%  47.2  441s
 21073 12589 148125.360   69 2743 149987.512 123713.045  17.5%  47.2  447s
 21076 12591 135043.467   38 2787 149987.512 123713.045  17.5%  47.2  450s
 21079 12593 135160.019   35 2729 149987.512 123713.045  17.5%  47.1  455s
 21083 12595 143458.055   46 2738 149987.512 123713.045  17.5%  47.1  461s
 21087 12598 149666.527   61 2748 149987.512 123713.045  17.5%  47.1  466s
 21090 12600 129075.187   67 2782 149987.512 123713.045  17.5%  47.1  470s
 21093 12602 148643.338   50 2715 149987.512 123713.045  17.5%  47.1  476s
 21096 12604 137716.365   44 2781 149987.512 123713.045  17.5%  47.1  480s
 21099 12606 145342.754   56 2687 149987.512 123713.045  17.5%  47.1  485s
 21103 12609 141288.743   40 2719 149987.512 123713.045  17.5%  47.1  491s
 21106 12611 149753.049   59 2786 149987.512 123713.045  17.5%  47.1  496s
 21109 12613 148655.600   87 2760 149987.512 123713.045  17.5%  47.1  500s
 21112 12615 145720.169   45 2782 149987.512 123713.045  17.5%  47.1  505s
 21115 12617 139197.705   61 2694 149987.512 123713.045  17.5%  47.1  510s
 21119 12619 149595.863   72 2743 149987.512 123713.045  17.5%  47.1  516s
 21122 12621 148453.622   56 2776 149987.512 123713.045  17.5%  47.0  520s
 21125 12623 138440.078   55 2757 149987.512 123713.045  17.5%  47.0  525s
 21129 12626 134704.798   35 2746 149987.512 123713.045  17.5%  47.0  531s
 21131 12627 148036.527   34 2740 149987.512 123713.045  17.5%  47.0  535s
 21134 12629 136287.771   42 2784 149987.512 123713.045  17.5%  47.0  540s
 21137 12631 125742.158   32 2744 149987.512 123713.045  17.5%  47.0  545s
 21140 12633 129975.876   39 2773 149987.512 123713.045  17.5%  47.0  550s
 21143 12635 135354.439   43 2764 149987.512 123713.045  17.5%  47.0  557s
 21146 12637 133120.556   42 2783 149987.512 123713.045  17.5%  47.0  561s
 21149 12639 142716.956   44 2758 149987.512 123713.045  17.5%  47.0  565s
 21152 12641 140193.153   92 2776 149987.512 123713.045  17.5%  47.0  570s
 21155 12643 132557.153   40 2766 149987.512 123713.045  17.5%  47.0  575s
 21159 12646 129862.945   30 2744 149987.512 123713.045  17.5%  47.0  581s
 21162 12648 135417.854   45 2784 149987.512 123713.045  17.5%  47.0  586s
 21163 12649 137175.766   28 2741 149987.512 123713.045  17.5%  47.0  591s
 21165 12650 137749.593   59 2770 149987.512 123713.045  17.5%  47.0  595s
 21169 12653 146668.872   43 2768 149987.512 123713.045  17.5%  46.9  601s
 21171 12654 149684.911   74 2767 149987.512 123713.045  17.5%  46.9  607s
 21172 12658 123713.045   30 2670 149987.512 123713.045  17.5%  63.0  610s
 21182 12672 123713.045   33 2568 149987.512 123713.045  17.5%  64.1  616s
 21217 12692 132996.035   39 1889 149987.512 123713.045  17.5%  65.4  621s
 21244 12700 141927.544   45 1976 149987.512 123713.045  17.5%  65.8  629s
 21252 12706 143984.625   48 1833 149987.512 123713.045  17.5%  66.0  632s
 21294 12714     cutoff   58      149987.512 123713.045  17.5%  66.2  635s
 21351 12721 149858.914   71  473 149987.512 123713.045  17.5%  66.7  640s
 21389 12737 137701.711   38 2488 149987.512 123713.045  17.5%  68.2  648s
 21411 12754 infeasible   42      149987.512 123713.045  17.5%  68.2  651s
 21477 12745 123713.045   35 2446 149987.512 123713.045  17.5%  69.3  656s
 21563 12852 123713.045   48 1708 149987.512 123713.045  17.5%  70.4  661s
 21677 12867 146350.079   73 1163 149987.512 123713.045  17.5%  70.3  665s
 21801 12881 135702.201   36 2313 149987.512 123713.045  17.5%  71.2  671s
 21871 12894 123713.045   38 2262 149987.512 123713.045  17.5%  71.9  675s
 22014 12923 128657.665   50 1988 149987.512 123713.045  17.5%  72.6  681s
 22140 12948 139591.958   74 1798 149987.512 123713.045  17.5%  73.0  686s
 22318 12973 134002.588   47 1942 149987.512 123713.045  17.5%  73.0  691s
 22391 12979 145204.165   61 1764 149987.512 123713.045  17.5%  73.1  696s
 22502 12966 149918.513   70 1615 149987.512 123713.045  17.5%  73.6  703s
 22530 12967 infeasible   68      149987.512 123713.045  17.5%  73.7  706s
 22567 12963 134015.077   40 2388 149987.512 123713.045  17.5%  73.9  710s
 22620 13004 140241.738   44 2152 149987.512 123713.045  17.5%  74.3  716s
 22780 13017 146688.800   45  969 149987.512 123713.045  17.5%  74.9  723s
 22868 13035 147913.878   43 2153 149987.512 123713.045  17.5%  75.6  727s
 22967 13053 148847.876   55 2011 149987.512 123713.045  17.5%  76.0  734s
 23080 13044     cutoff   51      149987.512 123713.045  17.5%  76.4  739s
 23152 13063 123725.276   37 2410 149987.512 123713.045  17.5%  76.8  743s
 23227 13080 147467.498   50 2014 149987.512 123713.045  17.5%  77.3  748s
 23315 13095 129530.106   37 2286 149987.512 123713.045  17.5%  77.9  752s
 23404 13123 133626.103   40 2302 149987.512 123713.045  17.5%  78.3  757s
 23514 13155 133326.135   40 2403 149987.512 123713.045  17.5%  78.6  762s
 23648 13159 147352.670   51 2136 149987.512 123713.045  17.5%  78.8  768s
 23770 13194 147920.482   71 1606 149987.512 123713.045  17.5%  79.0  772s
 23900 13212 145618.191   49 2137 149987.512 123713.045  17.5%  79.6  779s
 24055 13164 123963.907   49 2071 149987.512 123713.045  17.5%  79.8  786s
 24060 13166 123854.132   49 2027 149987.512 123713.045  17.5%  79.8  794s
 24064 13168 124265.162   50 1986 149987.512 123713.045  17.5%  79.8  800s
 24069 13185 124795.022   51 1971 149987.512 123713.045  17.5%  79.8  806s
 24104 13218 124795.090   52 1969 149987.512 123713.045  17.5%  79.9  812s
 24191 13254 125696.743   53 1965 149987.512 123713.045  17.5%  80.0  818s
 24294 13281 144328.334   44 2178 149987.512 123713.045  17.5%  80.1  826s
 24423 13340 135562.962   49 1710 149987.512 123713.045  17.5%  80.5  832s
 24596 13338 146708.774   58 1523 149987.512 123713.045  17.5%  80.6  838s
 24702 13375 140244.108   45 2156 149987.512 123713.045  17.5%  81.1  844s
 24861 13495 147956.842   67 1559 149987.512 123713.045  17.5%  81.1  852s
 25266 13465 139800.455   46 2135 149987.512 123713.045  17.5%  81.1  861s
 25522 13635 131379.049   45 2021 149987.512 123713.045  17.5%  80.9  869s
 26116 13569 134369.068   43 2290 149987.512 123713.045  17.5%  81.1  876s
 26384 13886 148338.793   72 1373 149987.512 123713.045  17.5%  81.3  886s
 27284 13871 143170.630   47 2187 149987.512 123713.045  17.5%  81.5  895s
 27911 13885 infeasible   42      149987.512 123713.045  17.5%  82.0  903s
 28337 14397 146684.888   51 1444 149987.512 123713.045  17.5%  82.1  913s
 29669 14018 infeasible   69      149987.512 123713.045  17.5%  81.8  923s
 29832 14411     cutoff   52      149987.512 123713.045  17.5%  81.9  932s
 30912 14299 131513.692   45 2119 149987.512 123951.852  17.4%  82.0  942s
 31382 15011 136868.181   46 1993 149987.512 124169.697  17.2%  82.1  959s
 33209 14423 128871.792   35 2408 149987.512 124770.283  16.8%  82.1  966s
 33246 14446     cutoff   41      149987.512 124837.760  16.8%  82.3  972s
 33346 14617     cutoff   49      149987.512 124924.010  16.7%  82.2  977s
 33836 14513 136507.375   39 1906 149987.512 125251.123  16.5%  82.4  982s
 33969 14604 133591.809   46 2095 149987.512 125406.490  16.4%  82.7  988s
 34189 14689 141063.638   41 1878 149987.512 125417.366  16.4%  82.6  996s
 34543 14705     cutoff   43      149987.512 125678.190  16.2%  82.7 1003s
 34891 14635 131298.772   36 2345 149987.512 125774.902  16.1%  83.3 1010s
 34995 14837 144811.620   42 1168 149987.512 125774.902  16.1%  83.4 1016s
 35459 14703 149951.535   42 1878 149987.512 125954.877  16.0%  83.9 1024s
 35529 14895 133145.687   50 2181 149987.512 126018.487  16.0%  84.0 1030s
 36055 14795 136300.755   47 2031 149987.512 126344.057  15.8%  84.7 1035s
 36218 15016 136531.835   39 2544 149987.512 126464.466  15.7%  85.1 1041s
 36784 14870 145086.899   45 1875 149987.512 126739.249  15.5%  85.7 1049s
 36913 15076 147287.324   52 2052 149987.512 126825.312  15.4%  86.1 1055s
 37480 14955 134670.401   38 2276 149987.512 126982.673  15.3%  86.4 1060s
 37627 14973 131639.909   37 2538 149987.512 127132.603  15.2%  86.8 1066s
 37783 15288 132366.024   37 2407 149987.512 127214.910  15.2%  87.0 1072s
 38560 15075 133659.630   38 2499 149987.512 127385.166  15.1%  87.1 1078s
 38706 15268 135914.298   43 2265 149987.512 127385.166  15.1%  87.2 1084s
 39243 15284 136733.948   46 1932 149987.512 127598.919  14.9%  87.9 1089s
 39644 15292 144186.444   44 2322 149987.512 127766.374  14.8%  88.2 1095s
 40032 15247 133293.859   37 2457 149987.512 127885.030  14.7%  88.5 1100s
 40234 15714 144993.952   71 1408 149987.512 127952.856  14.7%  88.6 1107s
 41279 15420     cutoff   51      149987.512 128126.198  14.6%  88.2 1113s
 41399 15781 146754.893   50 2060 149987.512 128215.825  14.5%  88.2 1120s
 42250 15781 136344.399   49 2070 149987.512 128424.286  14.4%  88.1 1132s
 42819 15781 142475.828   56 1547 149987.512 128699.432  14.2%  88.6 1139s
 43262 15788 145787.390   47 1467 149987.512 128741.536  14.2%  88.9 1147s
 43690 15790 132164.016   55 2178 149987.512 128935.241  14.0%  89.5 1155s
 44042 15807 133736.192   37 2536 149987.512 128998.028  14.0%  89.7 1163s
 44462 15929 149937.682   45 2147 149987.512 129062.876  14.0%  90.0 1172s
 45126 15737 135429.005   44 2209 149987.512 129317.901  13.8%  90.5 1179s
 45210 16157     cutoff   49      149987.512 129342.400  13.8%  90.7 1187s
 46176 15856 139499.163   49 2102 149987.512 129506.795  13.7%  90.5 1194s
 46271 16251 133852.621   41 2488 149987.512 129556.231  13.6%  90.6 1201s
 47195 15994 146593.439   52 2029 149987.512 129689.830  13.5%  90.6 1207s
 47450 16189 131491.586   37 1905 149987.512 129847.590  13.4%  90.8 1213s
 48217 15955 133074.253   39 2065 149987.512 129933.864  13.4%  91.0 1219s
 48344 16217 141414.845   42 2319 149987.512 129959.009  13.4%  91.2 1225s
 49083 16018 136995.157   45 2230 149987.512 130122.734  13.2%  91.5 1230s
 49197 16166 134444.042   39 2370 149987.512 130137.374  13.2%  91.8 1237s
 49781 16031 138058.729   48 2190 149987.512 130290.838  13.1%  92.3 1243s
 49921 16224 136394.238   53 1638 149987.512 130319.000  13.1%  92.4 1249s
 50460 16268 143267.566   46 2315 149987.512 130396.003  13.1%  92.8 1256s
 51016 16107 141488.343   48 2054 149987.512 130543.398  13.0%  93.0 1261s
 51103 16275 144067.897   45 2408 149987.512 130552.864  13.0%  93.1 1267s
 51630 16213 143852.884   39 2148 149987.512 130612.731  12.9%  93.4 1272s
 51925 16159 133948.563   41 2415 149987.512 130646.726  12.9%  93.8 1279s
 52042 16214 135463.759   48 2228 149987.512 130720.446  12.8%  94.1 1285s
 52364 16190 infeasible   58      149987.512 130755.851  12.8%  94.2 1290s
 52587 16466 133219.070   40 2424 149987.512 130785.556  12.8%  94.3 1299s
 53538 16189 134732.786   41 1968 149987.512 130983.196  12.7%  94.7 1305s
 53668 16369 146599.605   49 2294 149987.512 131018.385  12.6%  95.0 1314s
 54205 16190 131090.152   38 2551 149987.512 131054.780  12.6%  94.9 1315s
 54379 16266 137990.114   40 2510 149987.512 131135.289  12.6%  95.1 1324s
 54803 16159 138062.880   41 2475 149987.512 131238.371  12.5%  95.6 1329s
 54915 16348 142232.543   38 2262 149987.512 131238.371  12.5%  96.0 1335s
 55506 16168 infeasible   45      149987.512 131338.991  12.4%  96.2 1341s
 55581 16303 139590.940   49 2001 149987.512 131338.991  12.4%  96.2 1347s
 56070 16208 139960.360   49 1935 149987.512 131519.863  12.3%  96.6 1353s
 56202 16475 140641.238   36 2491 149987.512 131538.059  12.3%  96.7 1359s
 56805 16346 134343.564   39 1921 149987.512 131594.578  12.3%  96.9 1364s
 57043 16289     cutoff   47      149987.512 131637.070  12.2%  97.3 1370s
 57121 16288 142991.214   40 1915 149987.512 131692.176  12.2%  97.4 1376s
 57223 16588 infeasible   49      149987.512 131705.825  12.2%  97.4 1382s
 57986 16397 134731.598   44 2095 149987.512 131805.466  12.1%  97.4 1392s
 58102 16649 145234.315   48 2098 149987.512 131846.865  12.1%  97.4 1398s
 58798 16695 140099.241   42 2151 149987.512 131932.994  12.0%  97.3 1405s
 59633 16894 147824.967   51 2088 149987.512 132060.005  12.0%  97.2 1416s
 60498 16926 149087.466   48 2104 149987.512 132219.689  11.8%  97.0 1420s
 60576 17115     cutoff   49      149987.512 132239.986  11.8%  97.0 1427s
 61242 17117     cutoff   52      149987.512 132327.969  11.8%  97.1 1432s
 61290 17221 144468.348   44 1765 149987.512 132360.550  11.8%  97.2 1436s
 61550 17288 149733.292   58  791 149987.512 132409.328  11.7%  97.4 1441s
 61718 17463 139038.326   44 1998 149987.512 132429.662  11.7%  97.3 1447s
 62248 17560     cutoff   50      149987.512 132527.687  11.6%  97.7 1452s
 62520 17759 149878.301   47 1354 149987.512 132577.278  11.6%  97.9 1458s
 63073 17816 142682.561   52 2051 149987.512 132628.535  11.6%  98.0 1467s
 63209 19095 141668.043   59 1625 149987.512 132628.535  11.6%  98.0 1495s
 66521 19121 infeasible   74      149987.512 132918.390  11.4%  98.1 1502s
 66594 19136 148411.972   76 1612 149987.512 132922.100  11.4%  98.2 1512s
 66665 19184 134707.296   50 1932 149987.512 132993.651  11.3%  98.2 1516s
 66759 19464 134613.210   48 1933 149987.512 133000.304  11.3%  98.2 1521s
 67402 19564 149202.365   80 1631 149987.512 133082.278  11.3%  98.1 1526s
 67762 19722 145199.294   43 2230 149987.512 133139.412  11.2%  98.0 1531s
 68130 19750     cutoff   44      149987.512 133183.700  11.2%  98.2 1536s
 68241 19970 140840.467   42 1784 149987.512 133190.376  11.2%  98.2 1542s
 68863 19988 146937.336   71 1404 149987.512 133236.418  11.2%  98.2 1546s
 68925 20223 149362.464   52 2067 149987.512 133280.605  11.1%  98.3 1551s
 69540 20257 149457.875   52  887 149987.512 133335.613  11.1%  98.3 1555s
 69671 20372 149151.474   49  978 149987.512 133357.580  11.1%  98.3 1560s
 70183 20385     cutoff   48      149987.512 133440.449  11.0%  98.4 1568s
 70249 20481 infeasible   59      149987.512 133444.820  11.0%  98.5 1573s
 70677 20496 141898.037   42 1794 149987.512 133502.501  11.0%  98.8 1577s
 70735 20647 143996.182   45 2269 149987.512 133503.758  11.0%  98.8 1583s
 71201 20683     cutoff   48      149987.512 133533.414  11.0%  99.0 1589s
 71255 20842 148331.359   56  708 149987.512 133533.414  11.0%  99.1 1594s
 71641 20856 149230.303   49  998 149987.512 133588.866  10.9%  99.3 1599s
 71733 21094 139423.183   41 1917 149987.512 133620.129  10.9%  99.5 1604s
 72299 21125 146571.550   65 1087 149987.512 133642.294  10.9%  99.5 1609s
 72382 21203 infeasible   65      149987.512 133657.206  10.9%   100 1614s
 72631 21218 infeasible   47      149987.512 133676.574  10.9%   100 1619s
 72731 21364 149329.729   54 2033 149987.512 133715.487  10.8%   100 1624s
 73139 21412 149380.335   43 1247 149987.512 133780.539  10.8%   100 1631s
 73248 21605 146832.051   47 2031 149987.512 133796.895  10.8%   100 1636s
 73818 21755 139116.265   41 2104 149987.512 133829.220  10.8%   100 1644s
 74183 21847 infeasible   53      149987.512 133845.450  10.8%   100 1652s
 74446 22393 142672.617   70  993 149987.512 133899.470  10.7%   100 1664s
 75632 22405 138843.653   52 2111 149987.512 133986.057  10.7%   100 1671s
 75655 22422 149777.857   55 2017 149987.512 133986.057  10.7%   100 1680s
 75744 22693 139675.985   52 2108 149987.512 134011.837  10.7%   100 1687s
 76631 22696 144502.981   66 1721 149987.512 134094.808  10.6%   100 1694s
 76655 22911 147846.394   67 1713 149987.512 134101.468  10.6%   100 1701s
 77194 22979     cutoff   46      149987.512 134161.751  10.6%   100 1707s
 77365 23202 143870.744   57 2070 149987.512 134172.887  10.5%   100 1713s
 78028 23411 149944.871   63 1484 149987.512 134252.063  10.5%   101 1719s
 78595 23487 149410.408   48 1554 149987.512 134294.791  10.5%   101 1728s
 78728 23853 139743.317   51 2087 149987.512 134308.509  10.5%   101 1733s
 79466 24156 143459.674   43 2161 149987.512 134358.790  10.4%   101 1739s
 80154 24180 139536.052   57 2199 149987.512 134417.451  10.4%   100 1747s
 80225 24480 147394.045   63  813 149987.512 134423.426  10.4%   101 1752s
 80897 24561 135693.014   41 2342 149987.512 134462.514  10.4%   100 1758s
 81037 24736 144693.576   68  692 149987.512 134471.839  10.3%   100 1763s
 81422 24880 148739.288   50 1383 149987.512 134504.494  10.3%   101 1768s
 81790 24930 149701.076   50 1923 149987.512 134544.540  10.3%   101 1774s
 81893 25181 141528.113   47 2114 149987.512 134551.647  10.3%   101 1779s
 82521 25272 137394.350   55 1660 149987.512 134589.191  10.3%   101 1785s
 82661 25566 149442.426   85  801 149987.512 134589.191  10.3%   101 1791s
 83291 25782 140817.285   40 2388 149987.512 134628.002  10.2%   101 1796s
 83785 25796 147780.196   65  909 149987.512 134649.525  10.2%   101 1802s
 83848 26026 139140.441   53 1955 149987.512 134661.200  10.2%   101 1807s
 84521 26159 146146.671   44 1567 149987.512 134721.165  10.2%   101 1815s
 85013 26197 147504.850   48 2283 149987.512 134765.705  10.1%   101 1821s
 85179 26455 146661.408   46 1574 149987.512 134778.661  10.1%   101 1830s
 86097 26520 138598.557   38 2440 149987.512 134843.449  10.1%   101 1836s
 86222 26682     cutoff   52      149987.512 134843.449  10.1%   101 1842s
 86710 26797 143925.080   44 1990 149987.512 134935.997  10.0%   101 1849s
 86952 27012 141797.958   41 2427 149987.512 134956.172  10.0%   101 1855s
 87546 27022 139955.416   39 2453 149987.512 134997.202  10.0%   102 1863s
 87590 27208 140409.535   42 2386 149987.512 134999.362  10.0%   102 1869s
 88237 27228 148061.505   61 1389 149987.512 135013.421  10.0%   102 1876s
 88328 27488 149543.700   54 1989 149987.512 135024.639  10.0%   102 1881s
 88990 27564 149353.629   80 1740 149987.512 135083.399  9.94%   102 1886s
 89254 27624 144596.552   42 1786 149987.512 135126.955  9.91%   102 1892s
 89412 27636 147524.431   44 1031 149987.512 135129.921  9.91%   102 1900s
 89472 27904 145594.490   45 2333 149987.512 135146.114  9.90%   102 1906s
 90196 27947 137485.849   50 1852 149987.512 135191.104  9.87%   102 1912s
 90294 28227 139526.265   55 1891 149987.512 135197.703  9.86%   102 1918s
 91010 28301 infeasible   48      149987.512 135229.606  9.84%   102 1922s
 91168 28597 140964.557   52 1371 149987.512 135229.606  9.84%   102 1928s
 91781 28769 136456.048   55 1942 149987.512 135298.803  9.79%   102 1934s
 92166 28943 infeasible   74      149987.512 135324.694  9.78%   102 1940s
 92488 28973 140453.951   48 2154 149987.512 135339.236  9.77%   102 1947s
 92561 29288 146808.426   53 1927 149987.512 135343.186  9.76%   102 1952s
 93221 29365 149673.612   68 1308 149987.512 135402.273  9.72%   102 1957s
 93557 29466 141526.514   41 2199 149987.512 135433.970  9.70%   102 1962s
 93909 29723 148262.189   44 2085 149987.512 135457.444  9.69%   102 1968s
 94554 29745 139350.921   53 1310 149987.512 135495.710  9.66%   102 1974s
 94647 30046 infeasible   46      149987.512 135509.367  9.65%   102 1980s
 95313 30364 139196.153   52 2001 149987.512 135557.484  9.62%   102 1988s
 96026 30365 143166.840   48 2014 149987.512 135606.480  9.59%   102 2001s
 96098 30505     cutoff   45      149987.512 135615.903  9.58%   102 2007s
 96589 30799 148457.090   45 1673 149987.512 135628.157  9.57%   102 2015s
 97210 30826 143880.302   43 2245 149987.512 135681.506  9.54%   102 2021s
 97273 30979     cutoff   47      149987.512 135682.436  9.54%   102 2026s
 97776 31106 139643.054   47 1505 149987.512 135734.009  9.50%   102 2031s
 98080 31389     cutoff   52      149987.512 135761.048  9.49%   102 2038s
 98688 31419 148862.252   47 1483 149987.512 135803.364  9.46%   102 2045s
 98780 31664 149913.515   47  574 149987.512 135821.846  9.44%   102 2051s
 99528 31675 147803.200   49 1356 149987.512 135855.777  9.42%   102 2057s
 99569 31783     cutoff   50      149987.512 135855.777  9.42%   102 2062s
 100063 31891 139702.709   37 2306 149987.512 135918.802  9.38%   103 2068s
 100383 32013 147242.486   46 2182 149987.512 135949.181  9.36%   103 2074s
 100771 32042 144833.040   40 1748 149987.512 135985.299  9.34%   103 2080s
 100827 32228 149620.523   49 2119 149987.512 135985.496  9.34%   103 2086s
 101389 32257     cutoff   53      149987.512 136024.112  9.31%   103 2094s
 101468 32579 148866.812   54 1874 149987.512 136025.355  9.31%   103 2100s
 102258 32759     cutoff   57      149987.512 136076.901  9.27%   103 2106s
 102611 32789 147429.805   44 2077 149987.512 136113.986  9.25%   103 2111s
 102691 32817 138140.515   39 2374 149987.512 136118.260  9.25%   103 2118s
 102760 32843 138218.144   40 2338 149987.512 136120.384  9.25%   103 2123s
 102882 33159 138595.354   39 2336 149987.512 136132.235  9.24%   103 2130s
 103601 33184 146235.173   82 1198 149987.512 136173.482  9.21%   103 2138s
 103684 33535 137393.200   51 1895 149987.512 136179.824  9.21%   103 2143s
 104395 33684 143073.786   38 1565 149987.512 136212.751  9.18%   103 2149s
 104767 34011 148584.780   77  792 149987.512 136230.209  9.17%   103 2155s
 105425 34068 141136.156   50 2183 149987.512 136256.455  9.15%   103 2160s
 105557 34417 145358.709   52 2140 149987.512 136256.455  9.15%   103 2168s
 106483 34421 148549.392   76  755 149987.512 136329.969  9.11%   103 2172s
 106525 34705 143235.924   48 2230 149987.512 136329.969  9.11%   103 2177s
 107127 34732 138255.273   55 2028 149987.512 136352.646  9.09%   102 2180s
 107600 34952 147161.155   75  551 149987.512 136370.160  9.08%   103 2188s
 107696 35156 137056.314   51 1670 149987.512 136386.043  9.07%   103 2192s
 108117 35197 infeasible   80      149987.512 136403.605  9.06%   103 2197s
 108185 35429 137914.655   51 1879 149987.512 136407.812  9.05%   103 2202s
 108651 35442 143916.589   47 2444 149987.512 136411.548  9.05%   103 2208s
 108699 35804 145134.589   48 2440 149987.512 136428.934  9.04%   103 2215s
 109631 36073     cutoff   51      149987.512 136501.059  8.99%   103 2223s
 110289 36098 139842.661   52 1997 149987.512 136525.708  8.98%   103 2228s
 110353 36242 148006.746   65 1415 149987.512 136525.708  8.98%   103 2232s
 110805 36274 148469.864   73 1364 149987.512 136566.335  8.95%   103 2238s
 110885 36621 146815.357   50 2045 149987.512 136571.134  8.94%   103 2242s
 111566 36848 139776.315   43 2038 149987.512 136614.256  8.92%   102 2247s
 112084 36871 146541.758   72  870 149987.512 136634.261  8.90%   102 2251s
 112170 37059     cutoff   46      149987.512 136650.477  8.89%   102 2256s
 112608 37171 140487.439   58 1717 149987.512 136666.982  8.88%   102 2260s
 112858 37398 148418.005   78  802 149987.512 136674.397  8.88%   102 2265s
 113481 37537 148057.415   78  481 149987.512 136703.231  8.86%   102 2272s
 113708 37733 145094.281   52 2061 149987.512 136712.674  8.85%   102 2277s
 114042 37922 144570.000   50 2167 149987.512 136726.521  8.84%   102 2282s
 114532 37925     cutoff   48      149987.512 136744.384  8.83%   102 2286s
 114537 37927 144662.435   49 2028 149987.512 136744.384  8.83%   102 2296s
 114574 38109 147142.617   53 1959 149987.512 136744.384  8.83%   102 2301s
 115095 38143 139609.972   40 2309 149987.512 136784.993  8.80%   102 2307s
 115170 38412 141949.640   45 2350 149987.512 136795.929  8.80%   102 2311s
 115752 38532 144115.083   65 1003 149987.512 136813.242  8.78%   102 2315s
 115994 39076 139078.282   52 1429 149987.512 136831.175  8.77%   102 2324s
 117138 39099 147936.783   75 1553 149987.512 136863.123  8.75%   102 2327s
 117192 39156 138400.874   52 1626 149987.512 136868.973  8.75%   102 2330s
 117839 39561 140538.561   60 1156 149987.512 136913.814  8.72%   102 2338s
 118159 39583 145660.247   48 2314 149987.512 136919.180  8.71%   102 2342s
 118227 39738     cutoff   48      149987.512 136921.852  8.71%   102 2346s
 118606 39912 140912.938   40 2506 149987.512 136944.929  8.70%   102 2352s
 118972 39944 infeasible   56      149987.512 136952.709  8.69%   102 2356s
 119077 40231 147110.518   49 2045 149987.512 136964.556  8.68%   102 2361s
 119649 40370 147407.601   62 1545 149987.512 136985.807  8.67%   102 2365s
 120511 40691 143507.330   63  917 149987.512 137003.482  8.66%   101 2376s
 120564 41073 143597.691   64  908 149987.512 137009.757  8.65%   101 2380s
 121381 41465 137795.731   52 1646 149987.512 137037.288  8.63%   101 2388s
 122061 41513 139247.153   40 2497 149987.512 137057.099  8.62%   101 2392s
 122216 41676 139461.692   55 1350 149987.512 137059.928  8.62%   101 2396s
 122588 41716     cutoff   47      149987.512 137061.765  8.62%   101 2402s
 122684 42094 142315.466   56 1555 149987.512 137072.077  8.61%   101 2406s
 123388 42112 146502.618   66  709 149987.512 137083.686  8.60%   101 2411s
 123428 42372 147635.191   69  690 149987.512 137084.340  8.60%   101 2415s
 124155 42723 144714.643   60  861 149987.512 137102.003  8.59%   101 2423s
 124732 42992 142511.159   56 1802 149987.512 137126.448  8.57%   101 2427s
 125286 43021 138767.645   51 1521 149987.512 137150.323  8.56%   101 2433s
 125347 43181 146187.324   63  764 149987.512 137150.323  8.56%   101 2437s
 125762 43300 146726.497   58 2117 149987.512 137165.138  8.55%   101 2442s
 125990 43353     cutoff   56      149987.512 137178.599  8.54%   101 2447s
 126085 43648 148653.380   72  898 149987.512 137186.993  8.53%   101 2451s
 126680 43811 139982.325   41 2422 149987.512 137201.626  8.52%   101 2458s
 127066 44020 142675.085   58 1637 149987.512 137204.967  8.52%   101 2466s
 127596 44084     cutoff   55      149987.512 137222.295  8.51%   101 2472s
 127759 44517 148790.179   73  727 149987.512 137230.599  8.51%   101 2481s
 128680 44563 148244.795   67  719 149987.512 137267.409  8.48%   101 2486s
 128761 44963 141636.191   54 1717 149987.512 137280.775  8.47%   101 2494s
 129583 45005 146246.044   41 2347 149987.512 137310.565  8.45%   101 2500s
 129703 45414 138268.149   55 1764 149987.512 137317.871  8.45%   101 2506s
 130584 45419 142761.263   41 1839 149987.512 137346.899  8.43%   101 2512s
 130617 45721     cutoff   53      149987.512 137348.804  8.43%   101 2518s
 131340 45732 142020.651   42 2339 149987.512 137363.733  8.42%   101 2524s
 131361 45999 149052.926   48 1218 149987.512 137363.733  8.42%   101 2530s
 132029 46092     cutoff   53      149987.512 137392.400  8.40%   101 2535s
 132632 46350 143350.016   62 1487 149987.512 137404.945  8.39%   101 2545s
 132884 46655 148454.999   68 1160 149987.512 137430.857  8.37%   101 2550s
 133585 46678 145062.894   43 2170 149987.512 137445.572  8.36%   101 2556s
 133656 46901 147876.263   43 2326 149987.512 137451.102  8.36%   101 2561s
 134208 47172 148974.748   53 2225 149987.512 137469.398  8.35%   101 2566s
 134828 47197 145012.963   45 1846 149987.512 137491.437  8.33%   101 2573s
 134909 47561 144645.679   62 1502 149987.512 137491.437  8.33%   101 2578s
 135667 47706     cutoff   73      149987.512 137509.387  8.32%   101 2583s
 136075 47732 140653.008   40 2340 149987.512 137531.388  8.30%   101 2590s
 136118 47960     cutoff   48      149987.512 137531.388  8.30%   101 2595s
 137490 48401 140448.336   53 1863 149987.512 137576.419  8.27%   100 2606s
 137690 49315 142489.407   56 1568 149987.512 137576.419  8.27%   100 2621s
 139666 49307 148735.230   50 1571 149987.512 137620.298  8.25%   100 2627s
 139687 49328 138917.186   58 1225 149987.512 137624.812  8.24%   100 2634s
 139732 49360 139574.313   57 1582 149987.512 137624.812  8.24%   100 2640s
 139787 49616     cutoff   68      149987.512 137628.822  8.24%   100 2645s
 140621 49924 149977.955   47 1880 149987.512 137648.886  8.23%   100 2653s
 141158 49945     cutoff   53      149987.512 137667.112  8.21%   100 2659s
 141238 50121 139275.827   53 1681 149987.512 137673.679  8.21%   100 2663s
 141606 50350 139536.601   55 2104 149987.512 137682.101  8.20%   100 2668s
 142151 50373 145756.714   66 1361 149987.512 137701.559  8.19%   100 2674s
 142178 50577 146474.171   67 1358 149987.512 137701.559  8.19%   100 2679s
 142766 50791 138008.053   44 2290 149987.512 137721.115  8.18%   100 2684s
 143192 50795 147077.286   44 2181 149987.512 137735.251  8.17%   100 2688s
 143266 50982     cutoff   45      149987.512 137736.037  8.17%   100 2693s
 143697 51074 146181.838   61  867 149987.512 137742.291  8.16%   100 2697s
 143901 51296     cutoff   45      149987.512 137752.041  8.16%   100 2703s
 144408 51310 148853.122   47 2065 149987.512 137761.937  8.15%   100 2708s
 144465 51579 149482.129   48 1781 149987.512 137765.568  8.15%   100 2713s
 145030 51891 148750.340   46 1794 149987.512 137791.402  8.13%   100 2718s
 145730 51903     cutoff   49      149987.512 137811.844  8.12%   100 2723s
 145776 52180 141534.000   60 2133 149987.512 137814.071  8.12%   100 2727s
 146353 52378     cutoff   72      149987.512 137830.768  8.11%   100 2732s
 146731 52385 139807.355   47 2162 149987.512 137841.472  8.10%   100 2735s
 146787 52529 142636.192   43 2198 149987.512 137846.509  8.09%   100 2740s
 147218 52564 139389.172   58 2013 149987.512 137862.146  8.08%   100 2745s
 147296 52756 143925.698   45 1943 149987.512 137862.146  8.08%   100 2750s
 147807 53187 142748.335   43 2233 149987.512 137881.330  8.07%   100 2758s
 148818 53228 139735.187   42 2249 149987.512 137900.145  8.06%   100 2761s
 148875 53312     cutoff   50      149987.512 137900.145  8.06%   100 2765s
 149161 53401 145711.723   61 1942 149987.512 137921.218  8.04%   100 2770s
 149793 53674 147208.383   51 2171 149987.512 137940.187  8.03%   100 2778s
 149950 54007 147786.362   43 1874 149987.512 137946.092  8.03%   100 2782s
 150614 54045 138801.498   40 2294 149987.512 137966.419  8.01%   100 2788s
 150703 54253 148895.307   48 1456 149987.512 137968.134  8.01%   100 2792s
 151169 54357 138327.213   60 1690 149987.512 137985.280  8.00%   100 2798s
 151397 54532 infeasible   44      149987.512 137991.372  8.00%   100 2803s
 151741 54564 149347.198   44 1399 149987.512 138000.172  7.99%   100 2807s
 151861 54748 146256.260   52 2169 149987.512 138002.216  7.99%   100 2813s
 152367 54831 139055.449   45 2497 149987.512 138016.968  7.98%   100 2817s
 152576 55038 146999.481   70  674 149987.512 138027.928  7.97%   100 2823s
 152967 55097 148689.082   68  921 149987.512 138034.225  7.97%   100 2829s
 153096 55216     cutoff   44      149987.512 138038.387  7.97%   100 2835s
 153355 55541 140442.478   43 2223 149987.512 138045.641  7.96%   100 2842s
 154054 55566 140515.351   61 1566 149987.512 138060.291  7.95%   100 2850s
 154102 55919 142654.380   65 1504 149987.512 138060.291  7.95%   100 2856s
 154857 56239 143693.055   64 1540 149987.512 138077.580  7.94%   100 2862s
 155489 56266 144699.201   42 2123 149987.512 138089.596  7.93%   100 2868s
 155541 56511 144854.702   49 2083 149987.512 138092.211  7.93%   100 2873s
 156086 56633 143239.135   52 1952 149987.512 138107.398  7.92%   100 2878s
 156313 56765 145876.900   46  896 149987.512 138107.646  7.92%   100 2883s
 156583 56898 147628.107   44 1284 149987.512 138117.244  7.91%   100 2889s
 156863 57262 142676.785   58 1360 149987.512 138128.379  7.91%   100 2894s
 157574 57288 139583.759   45 2064 149987.512 138138.137  7.90%   100 2898s
 157679 57564 144988.683   55 1748 149987.512 138143.067  7.90%   100 2903s
 158244 57687 141278.148   57 1050 149987.512 138154.724  7.89%   100 2910s
 158569 58100 infeasible   45      149987.512 138168.483  7.88%   100 2919s
 159593 58115 149151.731   44 1197 149987.512 138192.175  7.86%   100 2930s
 159667 58363 139022.981   53 2011 149987.512 138202.753  7.86%   100 2936s
 160269 58598 140486.625   57 1424 149987.512 138212.760  7.85%   100 2941s
 160763 58632     cutoff   45      149987.512 138215.603  7.85%   100 2945s
 160842 58971 147397.585   42 2199 149987.512 138225.860  7.84%   100 2951s
 161500 59153 147375.916   42 2095 149987.512 138236.977  7.83%   100 2956s
 161898 59209 infeasible   64      149987.512 138244.666  7.83%   100 2960s
 162056 59606 146236.516   47 1221 149987.512 138249.340  7.83%   100 2966s
 162889 59637     cutoff   61      149987.512 138266.427  7.81%   100 2972s
 162973 60002 infeasible   63      149987.512 138266.427  7.81%   100 2977s
 163738 60121 144572.355   46 2174 149987.512 138295.371  7.80%   100 2982s
 164069 60214 142385.541   41 1925 149987.512 138306.757  7.79%   100 2986s
 164225 60385 145790.399   43 1926 149987.512 138314.004  7.78%   100 2991s
 164712 60444 146269.178   65 1184 149987.512 138320.944  7.78%   100 2997s
 164814 60690     cutoff   47      149987.512 138322.852  7.78%   100 3004s
 165432 61026 140264.622   44 2265 149987.512 138338.004  7.77%   100 3009s
 166147 61043 143884.299   43 2333 149987.512 138351.637  7.76%   100 3015s
 166665 61465 143094.071   62 1461 149987.512 138361.798  7.75%   100 3026s
 167288 61759 146798.825   67 1391 149987.512 138374.213  7.74%   100 3034s
 167768 61792 139815.955   58 1459 149987.512 138388.962  7.73%   100 3040s
 167817 61867 148700.392   71  683 149987.512 138388.962  7.73%   100 3045s
 167983 62081     cutoff   46      149987.512 138388.962  7.73%   100 3050s
 168530 62624 148237.662   75  636 149987.512 138399.486  7.73%   100 3063s
 169621 62648 144291.228   63 1299 149987.512 138419.422  7.71%   100 3070s
 169956 63028     cutoff   73      149987.512 138429.852  7.71%   100 3078s
 170445 63087 149478.530   47 1410 149987.512 138441.005  7.70%   100 3082s
 170534 63371 146872.576   51 2052 149987.512 138441.005  7.70%   100 3085s
 171586 63593 139163.105   55 1432 149987.512 138467.883  7.68%   100 3093s
 171612 63903 144141.634   59 1263 149987.512 138467.883  7.68%   100 3097s
 172204 63941     cutoff   67      149987.512 138480.296  7.67%  99.4 3100s
 172639 64177 149806.542   86  240 149987.512 138488.569  7.67%  99.4 3107s
 172754 64443 139209.279   58 2010 149987.512 138488.569  7.67%  99.5 3111s
 173331 64476 141091.933   42 2345 149987.512 138507.438  7.65%  99.4 3116s
 173397 64778 149223.765   44 1712 149987.512 138510.558  7.65%  99.4 3120s
 174410 64965 147544.082   48  729 149987.512 138531.038  7.64%  99.3 3129s
 174466 65236 139660.887   40 1847 149987.512 138535.124  7.64%  99.4 3133s
 174979 65316 142737.110   58 1249 149987.512 138544.702  7.63%  99.3 3138s
 175173 65540 140208.837   60 1459 149987.512 138553.037  7.62%  99.4 3142s
 175609 65580 139734.962   50 2098 149987.512 138561.335  7.62%  99.4 3148s
 175676 65904 140316.210   51 2094 149987.512 138561.335  7.62%  99.4 3152s
 176303 66044     cutoff   49      149987.512 138571.264  7.61%  99.3 3156s
 176608 66068 145015.100   49 2154 149987.512 138575.237  7.61%  99.3 3160s
 177150 66388 143648.929   41 2477 149987.512 138590.008  7.60%  99.3 3167s
 177361 66493 140188.063   42 2231 149987.512 138594.229  7.60%  99.3 3171s
 177940 66699 infeasible   69      149987.512 138606.411  7.59%  99.3 3178s
 178137 66935 145756.574   43 2136 149987.512 138612.259  7.58%  99.3 3182s
 178621 66944 148928.075   45 1574 149987.512 138616.228  7.58%  99.3 3187s
 178674 67244 145246.072   43 2393 149987.512 138622.614  7.58%  99.3 3191s
 179279 67630 149051.211   46 2095 149987.512 138637.612  7.57%  99.2 3200s
 180035 67700 141817.201   64 1146 149987.512 138646.849  7.56%  99.2 3205s
 180159 68071 139807.471   57 1985 149987.512 138649.559  7.56%  99.1 3211s
 180883 68091 139645.083   62 1416 149987.512 138658.711  7.55%  99.1 3216s
 180942 68451 140339.225   63 1372 149987.512 138665.262  7.55%  99.1 3221s
 181706 68684 148189.275   71  750 149987.512 138672.511  7.54%  99.0 3225s
 182270 68846 141404.960   58 1953 149987.512 138685.613  7.54%  98.9 3233s
 182650 68920 147589.770   67 1345 149987.512 138694.636  7.53%  98.9 3236s
 182814 69114 147529.913   64  966 149987.512 138696.036  7.53%  98.9 3241s
 183352 69146 144906.629   59 1491 149987.512 138711.109  7.52%  98.8 3245s
 183384 69362     cutoff   71      149987.512 138711.109  7.52%  98.8 3250s
 184230 69714 147518.261   56 2218 149987.512 138729.563  7.51%  98.8 3258s
 184716 69726 145725.569   45 2107 149987.512 138738.714  7.50%  98.8 3263s
 184769 69807     cutoff   51      149987.512 138738.714  7.50%  98.8 3267s
 184968 69911 146061.065   68  803 149987.512 138744.221  7.50%  98.7 3272s
 185228 69953 140951.584   41 2490 149987.512 138749.241  7.49%  98.8 3277s
 185317 70152 143596.603   56 1216 149987.512 138755.816  7.49%  98.8 3282s
 185777 70295 148377.291   65  868 149987.512 138761.519  7.48%  98.8 3286s
 186066 70381 148420.017   51 1849 149987.512 138767.733  7.48%  98.9 3290s
 186333 70523 149165.594   50  788 149987.512 138771.492  7.48%  98.9 3295s
 186758 70547 144630.197   41 1737 149987.512 138785.999  7.47%  98.8 3301s
 186819 70930 146217.018   42 2514 149987.512 138790.439  7.47%  98.9 3306s
 187571 71054 139590.114   51 2016 149987.512 138799.717  7.46%  98.8 3310s
 187823 71130 141013.713   56 1565 149987.512 138801.529  7.46%  98.8 3315s
 187999 71463 148508.195   75  591 149987.512 138804.770  7.46%  98.8 3321s
 188678 71506 141821.666   59 1523 149987.512 138820.200  7.45%  98.7 3327s
 188751 71860 147628.630   69  830 149987.512 138820.200  7.45%  98.7 3332s
 189386 72119 149542.626   44 1829 149987.512 138829.644  7.44%  98.7 3337s
 190015 72178 149486.646   73  624 149987.512 138840.460  7.43%  98.6 3344s
 190178 73301 143475.557   56 1375 149987.512 138844.040  7.43%  98.6 3363s
 192734 73371 148918.536   69  785 149987.512 138886.679  7.40%  98.4 3367s
 192851 73392 142991.216   40 1490 149987.512 138889.436  7.40%  98.4 3373s
 192891 73421 140945.462   58 1337 149987.512 138889.436  7.40%  98.4 3376s
 192963 73767 147306.671   70  613 149987.512 138889.436  7.40%  98.4 3381s
 193633 73848     cutoff   76      149987.512 138901.429  7.39%  98.4 3386s
 193812 73939 146781.359   42 1567 149987.512 138904.091  7.39%  98.4 3390s
 194342 74158 140380.600   54 1993 149987.512 138915.771  7.38%  98.4 3397s
 194702 74223 140895.644   56 1977 149987.512 138923.418  7.38%  98.4 3401s
 194791 74481     cutoff   67      149987.512 138923.418  7.38%  98.4 3405s
 195435 74819 139445.223   43 2613 149987.512 138937.860  7.37%  98.3 3410s
 196078 74851 141961.403   58 1869 149987.512 138947.504  7.36%  98.2 3417s
 196152 75181 148994.137   64 1745 149987.512 138947.504  7.36%  98.2 3422s
 196783 75451 147826.897   71  994 149987.512 138957.990  7.35%  98.2 3426s
 197269 75493     cutoff   75      149987.512 138964.940  7.35%  98.1 3430s
 197350 75714 infeasible   57      149987.512 138965.025  7.35%  98.1 3435s
 197942 75976     cutoff   47      149987.512 138975.018  7.34%  98.1 3443s
 198387 76016 143999.727   60 1866 149987.512 138985.227  7.34%  98.1 3446s
 198464 76288 139916.563   60 2056 149987.512 138985.227  7.34%  98.2 3451s
 199043 76311 147413.264   79  731 149987.512 138995.227  7.33%  98.1 3456s
 199095 76553 149157.445   84  795 149987.512 138995.227  7.33%  98.1 3460s
 199808 76918 146337.057   52 2096 149987.512 139007.578  7.32%  98.1 3468s
 200524 77171 147740.861   73  661 149987.512 139015.313  7.32%  98.0 3475s
 200986 77525 141767.700   43 2110 149987.512 139022.520  7.31%  98.0 3482s
 201733 77543 146819.816   64 1363 149987.512 139026.779  7.31%  97.9 3489s
 201774 77811 146819.816   65 1362 149987.512 139030.294  7.31%  97.9 3494s
 202371 78094 146906.425   74  750 149987.512 139052.354  7.29%  97.9 3498s
 202974 78095 145094.789   49 1950 149987.512 139059.004  7.29%  97.9 3503s
 202980 78100 145094.838   50 1929 149987.512 139059.004  7.29%  97.9 3508s
 202997 78313 148161.706   54 1731 149987.512 139059.004  7.29%  97.9 3512s
 203493 78368 143011.698   40 2495 149987.512 139075.092  7.28%  97.9 3516s
 203579 78622 147441.421   48  968 149987.512 139075.389  7.28%  97.9 3520s
 204330 79021 139638.254   57 1473 149987.512 139087.153  7.27%  97.9 3529s
 204940 79060 149861.069   72  915 149987.512 139095.146  7.26%  97.8 3533s
 205016 79217 148179.925   69 1792 149987.512 139095.146  7.26%  97.8 3537s
 205555 79318 148034.573   68  889 149987.512 139104.748  7.26%  97.8 3541s
 205821 79362     cutoff   48      149987.512 139109.261  7.25%  97.8 3545s
 205971 79628 145100.394   75  865 149987.512 139110.228  7.25%  97.9 3550s
 206672 79863 142758.707   59 2100 149987.512 139119.451  7.25%  97.8 3558s
 207084 79959 140260.116   43 2179 149987.512 139123.902  7.24%  97.8 3563s
 207289 80206 147156.334   41 1786 149987.512 139126.925  7.24%  97.9 3568s
 207840 80227 144260.853   47 2197 149987.512 139133.870  7.24%  97.8 3573s
 207898 80471 149165.071   52 2091 149987.512 139133.870  7.24%  97.8 3578s
 208550 80497 145406.581   55 2082 149987.512 139144.595  7.23%  97.8 3582s
 208600 80811 145429.231   56 2096 149987.512 139144.595  7.23%  97.8 3587s
 209220 81028 141004.505   61 1594 149987.512 139150.453  7.23%  97.7 3591s
 209721 81053 149619.924   74  207 149987.512 139163.046  7.22%  97.7 3595s
 209790 81328 143433.093   60 2031 149987.512 139167.349  7.21%  97.7 3600s
 210381 81410 142373.764   61 1594 149987.512 139177.272  7.21%  97.7 3605s
 210782 81787 149691.298   82  270 149987.512 139186.042  7.20%  97.7 3615s
 211383 81797 146383.881   43 2219 149987.512 139195.343  7.20%  97.7 3621s
 211469 82194 141379.111   58 1140 149987.512 139199.542  7.19%  97.7 3628s
 212306 82316 147233.991   76  640 149987.512 139211.200  7.18%  97.6 3632s
 212541 82487 145964.298   42 1910 149987.512 139218.346  7.18%  97.6 3636s
 213049 82521 141881.594   62 1330 149987.512 139226.423  7.17%  97.6 3641s
 213110 82815     cutoff   48      149987.512 139226.423  7.17%  97.6 3646s
 213779 82985 145073.346   67 1335 149987.512 139242.632  7.16%  97.6 3650s
 214433 83151 143720.433   57 1462 149987.512 139249.643  7.16%  97.6 3659s
 214474 83497 144515.777   57 1435 149987.512 139252.531  7.16%  97.6 3665s
 215180 83594 149095.559   73  837 149987.512 139261.736  7.15%  97.5 3670s
 215373 83611 149943.493   74  768 149987.512 139266.372  7.15%  97.5 3675s
 215413 83626     cutoff   74      149987.512 139266.860  7.15%  97.5 3680s
 215456 83737     cutoff   76      149987.512 139266.860  7.15%  97.5 3685s
 215639 83920     cutoff   79      149987.512 139271.794  7.14%  97.5 3690s
 216045 84061 142953.189   58 1581 149987.512 139277.264  7.14%  97.5 3695s
 216301 84276 147703.219   77  664 149987.512 139282.707  7.14%  97.4 3700s
 216800 84620 142583.878   54 2039 149987.512 139287.782  7.13%  97.4 3709s
 217609 84622     cutoff   79      149987.512 139305.611  7.12%  97.3 3715s
 217615 84631 141333.188   53 2046 149987.512 139305.611  7.12%  97.3 3722s
 217629 84780 142069.741   55 1978 149987.512 139306.089  7.12%  97.3 3726s
 217970 84927 146415.055   70  941 149987.512 139310.042  7.12%  97.3 3731s
 218354 85039 147718.471   66 1448 149987.512 139315.615  7.12%  97.3 3736s
 218696 85064 144242.498   69 1673 149987.512 139323.547  7.11%  97.3 3740s
 218751 85335 143664.581   51 2183 149987.512 139323.547  7.11%  97.3 3746s
 219332 85344 148288.883   42 2008 149987.512 139333.886  7.10%  97.3 3752s
 219376 85607 141397.069   59 1357 149987.512 139333.886  7.10%  97.3 3757s
 219915 85658 infeasible   74      149987.512 139342.672  7.10%  97.3 3761s
 220030 85880 145661.049   65  911 149987.512 139343.825  7.10%  97.3 3765s
 220650 85952 145566.934   60 1862 149987.512 139351.537  7.09%  97.3 3770s
 220832 86193 145493.919   55 1877 149987.512 139356.855  7.09%  97.3 3775s
 221408 86290     cutoff   56      149987.512 139369.935  7.08%  97.3 3780s
 221634 87036 139988.552   61 1433 149987.512 139372.164  7.08%  97.3 3793s
 223351 87044 139454.449   52 1977 149987.512 139393.361  7.06%  97.1 3800s
 223713 87191     cutoff   58      149987.512 139402.634  7.06%  97.1 3807s
 223739 87474     cutoff   59      149987.512 139404.033  7.06%  97.1 3812s
 224353 87517 144923.267   44 2382 149987.512 139415.947  7.05%  97.1 3816s
 224504 87778 142767.513   57 1940 149987.512 139417.010  7.05%  97.1 3820s
 225161 88021 140491.446   60 1976 149987.512 139426.682  7.04%  97.1 3828s
 225684 88225 149543.254   68 1874 149987.512 139434.040  7.04%  97.1 3832s
 226123 88234 141558.739   44 2447 149987.512 139440.911  7.03%  97.0 3839s
 226156 88429 140400.265   60 2017 149987.512 139443.029  7.03%  97.0 3843s
 226640 88528     cutoff   49      149987.512 139452.160  7.02%  97.0 3846s
 226851 88796 140006.257   61 1493 149987.512 139456.591  7.02%  97.0 3851s
 227485 88822 146580.763   64  604 149987.512 139463.190  7.02%  96.9 3855s
 228160 89178 146100.428   61 1359 149987.512 139471.801  7.01%  96.9 3863s
 228407 89405 149906.439   75  933 149987.512 139475.463  7.01%  96.9 3868s
 228887 89445 139952.568   57 1880 149987.512 139478.448  7.01%  96.8 3872s
 228972 89761 140037.257   58 1828 149987.512 139480.400  7.01%  96.8 3877s
 229613 90009 145945.280   66 1867 149987.512 139487.172  7.00%  96.8 3882s
 230124 90028 141012.450   58 1929 149987.512 139494.027  7.00%  96.8 3887s
 230166 90195 141719.675   59 1830 149987.512 139494.027  7.00%  96.8 3891s
 230576 90390 146298.101   43 1598 149987.512 139501.652  6.99%  96.8 3896s
 231053 90427 142568.344   62 1437 149987.512 139511.934  6.98%  96.8 3901s
 231125 90717 148141.768   73  652 149987.512 139512.156  6.98%  96.8 3905s
 231796 91066 141844.674   58 1602 149987.512 139516.323  6.98%  96.7 3912s
 232567 91088     cutoff   55      149987.512 139526.265  6.97%  96.7 3920s
 232603 91314 148231.659   64  952 149987.512 139526.265  6.97%  96.7 3925s
 233514 91563 140926.243   51 1791 149987.512 139538.711  6.97%  96.7 3933s
 233636 91851 148336.577   61 1507 149987.512 139539.290  6.97%  96.7 3938s
 234198 91976 141671.680   52 1948 149987.512 139546.837  6.96%  96.7 3941s
 234470 92200 149149.746   67 1779 149987.512 139547.323  6.96%  96.7 3946s
 235010 92227 141858.531   59 1214 149987.512 139554.850  6.96%  96.7 3952s
 235084 92526 145194.916   64 1133 149987.512 139554.850  6.96%  96.7 3956s
 235668 92837 143185.324   56 1508 149987.512 139564.297  6.95%  96.6 3961s
 236248 92879 148847.712   76  687 149987.512 139569.341  6.95%  96.6 3966s
 236325 93092     cutoff   52      149987.512 139569.341  6.95%  96.6 3972s
 236867 93324 145694.099   65  822 149987.512 139574.753  6.94%  96.6 3976s
 237424 93340     cutoff   67      149987.512 139579.431  6.94%  96.6 3983s
 237479 93637 142391.376   56 1974 149987.512 139581.015  6.94%  96.6 3987s
 238091 93844 149546.655   71 1086 149987.512 139585.458  6.94%  96.5 3991s
 238652 93930 146861.171   76  665 149987.512 139592.320  6.93%  96.5 3995s
 238893 94208 142981.977   66 1041 149987.512 139594.474  6.93%  96.5 4000s
 239485 94418 145222.005   64 1208 149987.512 139604.926  6.92%  96.5 4009s
 239952 94737 145993.127   42 2028 149987.512 139611.923  6.92%  96.5 4013s
 240527 94754 149534.556   68  742 149987.512 139621.741  6.91%  96.5 4017s
 240589 95050 141977.866   50 1952 149987.512 139623.257  6.91%  96.5 4022s
 241139 95262 infeasible   57      149987.512 139631.766  6.90%  96.4 4026s
 241694 95526     cutoff   47      149987.512 139637.178  6.90%  96.4 4033s
 242194 95821 143690.164   66 1288 149987.512 139643.206  6.90%  96.4 4041s
 242747 95908 146509.896   59 1349 149987.512 139650.419  6.89%  96.4 4046s
 242921 96326 148789.661   70 1225 149987.512 139651.201  6.89%  96.4 4054s
 243806 96371     cutoff   53      149987.512 139657.575  6.89%  96.4 4058s
 243868 96676 145935.847   67 1105 149987.512 139657.575  6.89%  96.4 4063s
 244480 96819 141335.447   51 1846 149987.512 139666.860  6.88%  96.4 4067s
 244766 96869 148834.498   62 1728 149987.512 139670.176  6.88%  96.4 4071s
 244873 97132 146787.079   70 1227 149987.512 139670.708  6.88%  96.4 4077s
 245469 97134     cutoff   48      149987.512 139674.785  6.88%  96.4 4083s
 245475 97153     cutoff   48      149987.512 139675.962  6.87%  96.4 4092s
 245508 97443 143548.492   45 2324 149987.512 139676.922  6.87%  96.4 4097s
 246091 97477     cutoff   64      149987.512 139682.364  6.87%  96.4 4103s
 246156 97722 146850.234   77  736 149987.512 139682.364  6.87%  96.4 4108s
 246768 97972 146306.841   64 1158 149987.512 139689.008  6.87%  96.4 4113s
 247330 97986 142385.778   63 1340 149987.512 139696.413  6.86%  96.4 4117s
 247391 98272 147069.567   65 1240 149987.512 139696.413  6.86%  96.4 4122s
 247983 98279     cutoff   50      149987.512 139703.694  6.86%  96.4 4126s
 248028 98497 140813.031   62 1333 149987.512 139704.809  6.86%  96.4 4131s
 248612 98795 149583.217   64 1959 149987.512 139711.683  6.85%  96.4 4136s
 249186 98821 141034.742   63 1938 149987.512 139716.860  6.85%  96.4 4141s
 249242 99094 149259.387   55 2172 149987.512 139717.907  6.85%  96.4 4146s
 249755 99177 142715.617   54 1962 149987.512 139724.976  6.84%  96.4 4151s
 249998 99404 143220.009   60 1837 149987.512 139730.299  6.84%  96.4 4155s
 250580 99424 146583.170   48 1956 149987.512 139739.027  6.83%  96.4 4161s
 250624 99665     cutoff   52      149987.512 139740.820  6.83%  96.4 4166s
 251221 99742 140960.224   49 1632 149987.512 139750.080  6.83%  96.4 4170s
 251393 100042 149651.293   75 1138 149987.512 139750.113  6.83%  96.4 4176s
 251996 100086 140589.100   53 1504 149987.512 139756.222  6.82%  96.3 4180s
 252065 100304 148066.852   69  643 149987.512 139756.222  6.82%  96.3 4186s
 252629 100599 140717.526   59 1776 149987.512 139764.599  6.82%  96.3 4192s
 253203 100654 149574.945   67 1326 149987.512 139771.133  6.81%  96.3 4196s
 253341 100997 infeasible   61      149987.512 139774.872  6.81%  96.3 4203s
 254179 101022 146867.741   58 1074 149987.512 139784.845  6.80%  96.3 4209s
 254270 101305 142795.852   59 1227 149987.512 139784.845  6.80%  96.3 4214s
 254892 101327     cutoff   51      149987.512 139796.030  6.79%  96.3 4217s
 254944 101606 144243.657   66 1362 149987.512 139797.165  6.79%  96.3 4221s
 255486 101678 143485.186   57 1857 149987.512 139801.356  6.79%  96.3 4225s
 255881 101966 147794.451   69  801 149987.512 139807.145  6.79%  96.3 4234s
 256355 102022 141354.446   58 1922 149987.512 139813.187  6.78%  96.3 4239s
 256418 102273 148171.056   75 1313 149987.512 139813.387  6.78%  96.3 4244s
 256915 102433 142609.108   56 2003 149987.512 139817.931  6.78%  96.3 4248s
 257376 102461 141067.981   41 2389 149987.512 139825.793  6.78%  96.4 4253s
 257434 102714 145768.725   59 1202 149987.512 139825.793  6.78%  96.4 4258s
 258021 102922 141764.207   52 1838 149987.512 139830.534  6.77%  96.4 4263s
 258647 102935 146165.563   54 1936 149987.512 139838.954  6.77%  96.4 4271s
 258677 103217 145176.931   49 2225 149987.512 139840.162  6.77%  96.4 4275s
 259326 103528 148155.520   64 1260 149987.512 139846.539  6.76%  96.3 4280s
 259955 103554 146118.866   64 1726 149987.512 139853.327  6.76%  96.3 4286s
 260010 103687 144114.629   62 1087 149987.512 139853.327  6.76%  96.3 4290s
 260408 103873     cutoff   49      149987.512 139859.188  6.75%  96.4 4295s
 260834 103893 145994.886   41 2451 149987.512 139868.607  6.75%  96.4 4301s
 260886 104164 144170.234   60 1404 149987.512 139869.360  6.75%  96.4 4306s
 261467 104170     cutoff   49      149987.512 139873.375  6.74%  96.4 4311s
 261510 104408 142015.867   56 1955 149987.512 139874.418  6.74%  96.4 4316s
 262031 104490 140835.251   47 2106 149987.512 139878.414  6.74%  96.4 4320s
 262186 104741 141071.126   55 1662 149987.512 139878.697  6.74%  96.4 4325s
 262797 104744     cutoff   72      149987.512 139884.168  6.74%  96.4 4331s
 262854 105144 140984.267   51 2005 149987.512 139886.165  6.73%  96.4 4338s
 263747 105160 141312.911   53 1787 149987.512 139892.047  6.73%  96.4 4342s
 263793 105396 142834.665   47 2152 149987.512 139892.752  6.73%  96.4 4347s
 264298 105560 147980.076   64 1142 149987.512 139895.751  6.73%  96.4 4351s
 264682 105591 141036.365   50 2021 149987.512 139904.717  6.72%  96.4 4357s
 264717 105731 146818.640   59 1796 149987.512 139905.320  6.72%  96.4 4361s
 265131 105876 142303.418   56 2051 149987.512 139909.220  6.72%  96.4 4365s
 265499 105901 148761.241   67  487 149987.512 139914.322  6.72%  96.4 4371s
 265540 106102 147631.889   49 2214 149987.512 139914.501  6.72%  96.4 4375s
 266041 106292     cutoff   44      149987.512 139918.181  6.71%  96.5 4380s
 266495 106304 147574.623   69 1026 149987.512 139929.413  6.71%  96.5 4385s
 266541 106453 141521.384   48 2260 149987.512 139930.697  6.71%  96.5 4391s
 266961 106768 148453.579   74  807 149987.512 139933.886  6.70%  96.5 4399s
 267589 106774 147614.321   65 1964 149987.512 139938.983  6.70%  96.5 4403s
 267624 106957 149189.819   45 2021 149987.512 139940.229  6.70%  96.5 4408s
 268165 106984 141650.977   44 2166 149987.512 139946.355  6.69%  96.5 4412s
 268240 107201 146985.728   64 1084 149987.512 139946.795  6.69%  96.5 4417s
 268778 107221     cutoff   58      149987.512 139951.461  6.69%  96.5 4420s
 268825 107435     cutoff   71      149987.512 139951.814  6.69%  96.5 4425s
 269650 107568     cutoff   62      149987.512 139962.976  6.68%  96.5 4433s
 269777 107770 148662.383   67  648 149987.512 139963.872  6.68%  96.5 4438s
 270301 107790 149161.928   68  821 149987.512 139970.278  6.68%  96.5 4443s
 270340 108064 149046.289   70 1041 149987.512 139970.278  6.68%  96.5 4448s
 270911 108110 149180.260   66 1283 149987.512 139976.143  6.67%  96.5 4452s
 271048 108320 141718.553   55 1520 149987.512 139979.382  6.67%  96.5 4457s
 271563 108351 infeasible   63      149987.512 139984.892  6.67%  96.5 4460s
 271661 108611     cutoff   75      149987.512 139986.000  6.67%  96.5 4465s
 272263 108786 144957.512   39 2187 149987.512 139992.213  6.66%  96.5 4470s
 272704 109075     cutoff   66      149987.512 139997.607  6.66%  96.5 4480s
 273337 109306 143406.135   59 1635 149987.512 140006.909  6.65%  96.5 4486s
 273957 109328 149284.014   47 1975 149987.512 140013.093  6.65%  96.5 4491s
 274023 109575 infeasible   55      149987.512 140016.928  6.65%  96.5 4497s
 274640 109736 148360.326   69  831 149987.512 140022.596  6.64%  96.5 4502s
 274983 109807 149677.379   61 1992 149987.512 140026.835  6.64%  96.5 4506s
 275192 110125 141964.586   43 1967 149987.512 140029.111  6.64%  96.5 4512s
 275922 110153 149796.636   64 1716 149987.512 140037.586  6.63%  96.5 4517s
 275982 110374 149061.554   48 2235 149987.512 140039.049  6.63%  96.5 4522s
 276535 110454 149215.707   50 1609 149987.512 140045.847  6.63%  96.5 4528s
 276795 110535 141717.843   65 1547 149987.512 140049.150  6.63%  96.5 4531s
 276980 110704 143519.452   54 1735 149987.512 140051.388  6.62%  96.6 4536s
 277404 110715 147762.728   46 1956 149987.512 140053.298  6.62%  96.5 4542s
 277456 111026 143505.089   55 1782 149987.512 140055.103  6.62%  96.6 4547s
 278189 111048 infeasible   53      149987.512 140064.194  6.62%  96.5 4550s
 278292 111318 142749.646   55 1878 149987.512 140064.194  6.62%  96.5 4556s
 278907 111449     cutoff   77      149987.512 140074.128  6.61%  96.5 4560s
 279189 111694 infeasible   50      149987.512 140077.994  6.61%  96.5 4565s
 279822 111709 144541.271   56 2168 149987.512 140084.065  6.60%  96.5 4573s
 279841 111935 149344.099   60 2101 149987.512 140084.370  6.60%  96.5 4578s
 280459 112171 141544.242   53 1579 149987.512 140088.577  6.60%  96.5 4582s
 280991 112286 149811.648   67  732 149987.512 140091.759  6.60%  96.5 4587s
 281267 112390 147521.037   65  561 149987.512 140094.060  6.60%  96.5 4591s
 281539 112403 143043.917   58 1777 149987.512 140098.945  6.59%  96.5 4595s
 281601 112681 149518.449   69  542 149987.512 140099.842  6.59%  96.5 4600s
 282324 112946     cutoff   68      149987.512 140108.174  6.59%  96.5 4608s
 282941 113035     cutoff   42      149987.512 140114.308  6.58%  96.5 4614s
 283148 113298 142464.957   52 2122 149987.512 140116.996  6.58%  96.5 4619s
 283717 113340 141828.614   57 1779 149987.512 140122.775  6.58%  96.5 4625s
 283809 114819 140541.352   58 1312 149987.512 140125.613  6.58%  96.5 4654s
 287321 114835 146735.274   61 1310 149987.512 140163.137  6.55%  96.6 4659s
 287372 114873 142818.653   62 1309 149987.512 140164.368  6.55%  96.6 4663s
 287446 114896     cutoff   75      149987.512 140164.368  6.55%  96.6 4667s
 287511 114933 149857.489   79  528 149987.512 140164.368  6.55%  96.6 4673s
 287600 115180     cutoff   70      149987.512 140164.368  6.55%  96.6 4678s
 288225 115355 142795.658   43 2325 149987.512 140173.931  6.54%  96.6 4683s
 288616 115378 146033.736   61 1240 149987.512 140176.892  6.54%  96.6 4687s
 288688 115648 149223.383   65  769 149987.512 140177.237  6.54%  96.6 4692s
 289326 115842 146589.321   67  852 149987.512 140183.911  6.54%  96.6 4697s
 289839 115887 141048.004   61 1743 149987.512 140190.304  6.53%  96.7 4702s
 289921 116142 146945.593   71 1126 149987.512 140190.800  6.53%  96.7 4707s
 290620 116289     cutoff   80      149987.512 140196.890  6.53%  96.6 4712s
 291046 116337 142971.277   60 1438 149987.512 140200.989  6.52%  96.7 4715s
 291123 116543 149659.380   78  734 149987.512 140201.672  6.52%  96.7 4720s
 291663 116702 147087.028   69  894 149987.512 140206.829  6.52%  96.7 4725s
 292164 116714 142256.091   57 1519 149987.512 140212.374  6.52%  96.7 4731s
 292198 116983     cutoff   45      149987.512 140212.420  6.52%  96.7 4736s
 292793 117162 144346.695   63 1299 149987.512 140219.605  6.51%  96.7 4741s
 293430 117426 145033.552   58 1167 149987.512 140225.825  6.51%  96.7 4749s
 293931 117451 147295.206   67 1247 149987.512 140229.360  6.51%  96.7 4754s
 294019 118883 147435.207   68 1234 149987.512 140229.730  6.51%  96.7 4785s
 297539 118905 144896.991   62 1431 149987.512 140265.437  6.48%  96.7 4791s
 297590 118947     cutoff   72      149987.512 140266.267  6.48%  96.7 4796s
 297661 118974 148416.464   48 1077 149987.512 140268.598  6.48%  96.7 4800s
 297764 119197 144975.829   56 1720 149987.512 140270.498  6.48%  96.7 4805s
 298439 119431 146127.978   63 1051 149987.512 140272.954  6.48%  96.7 4812s
 298907 119452 142028.431   46 2332 149987.512 140280.485  6.47%  96.8 4817s
 298972 119651 149173.420   71 1790 149987.512 140281.862  6.47%  96.8 4822s
 299522 119654 149283.081   45 2128 149987.512 140286.312  6.47%  96.8 4827s
 299543 119841 149011.040   45 2170 149987.512 140286.406  6.47%  96.8 4831s
 300013 120070     cutoff   53      149987.512 140289.510  6.47%  96.8 4836s
 300705 120100 142640.551   55 1842 149987.512 140293.925  6.46%  96.7 4840s
 300746 120327 147213.931   60 1767 149987.512 140293.925  6.46%  96.7 4845s
 301366 120605     cutoff   74      149987.512 140300.442  6.46%  96.7 4854s
 301982 120609 146571.305   56 1699 149987.512 140309.590  6.45%  96.7 4858s
 302054 120845 143401.596   61 1916 149987.512 140310.280  6.45%  96.7 4863s
 302653 120894 147138.464   70 1239 149987.512 140315.844  6.45%  96.7 4867s
 302802 121130 142257.329   50 1047 149987.512 140319.009  6.45%  96.8 4871s
 303330 121140 146171.340   66 1084 149987.512 140321.646  6.44%  96.8 4876s
 303379 121364 147119.864   48 2014 149987.512 140322.642  6.44%  96.8 4881s
 303944 121452 149196.840   45 2181 149987.512 140329.099  6.44%  96.8 4885s
 304199 121627     cutoff   53      149987.512 140330.155  6.44%  96.8 4891s
 304551 121892     cutoff   70      149987.512 140333.641  6.44%  96.8 4899s
 305237 121922 142043.768   52 1939 149987.512 140337.813  6.43%  96.9 4905s
 305287 122180 142226.068   52 1930 149987.512 140339.966  6.43%  96.9 4911s
 305969 122390 143257.048   60 1541 149987.512 140347.771  6.43%  96.9 4916s
 306435 122422 149802.426   73  857 149987.512 140352.479  6.42%  96.9 4920s
 306521 122709 142721.880   54 2150 149987.512 140354.694  6.42%  96.9 4925s
 307548 122986     cutoff   63      149987.512 140365.500  6.42%  96.9 4934s
 307965 122990 148262.132   70 1110 149987.512 140368.355  6.41%  96.9 4940s
 307969 122991 148430.229   71 1058 149987.512 140368.355  6.41%  96.9 4952s
 307986 122990     cutoff   72      149987.512 140368.355  6.41%  96.9 4958s
 308001 122993     cutoff   72      149987.512 140369.235  6.41%  96.9 4962s
 308035 123005 141978.290   43 2243 149987.512 140369.859  6.41%  96.9 4966s
 308076 123040 146232.514   43 2098 149987.512 140369.859  6.41%  96.9 4974s
 308151 123105     cutoff   47      149987.512 140371.072  6.41%  96.9 4979s
 308316 123262 146156.591   58 1380 149987.512 140371.072  6.41%  96.9 4984s
 308726 123289 141471.551   63 1179 149987.512 140371.861  6.41%  96.9 4990s
 308781 123512 144688.351   66 1120 149987.512 140371.861  6.41%  96.9 4996s
 309334 123533 infeasible   74      149987.512 140375.614  6.41%  96.9 5002s
 309398 123653     cutoff   71      149987.512 140375.614  6.41%  96.9 5008s
 309716 123656 148060.661   63 1760 149987.512 140379.988  6.41%  97.0 5014s
 309772 123833 142249.249   50 1919 149987.512 140382.096  6.40%  97.0 5020s
 310233 123839 141678.868   55 1823 149987.512 140383.762  6.40%  97.0 5026s
 310271 124104 146046.841   60 1498 149987.512 140386.284  6.40%  97.0 5033s
 310921 124106 148685.893   70  775 149987.512 140387.407  6.40%  97.0 5038s
 310936 124297 149719.372   71  764 149987.512 140388.583  6.40%  97.0 5044s
 311462 124330 143817.834   60 1409 149987.512 140396.851  6.39%  97.0 5049s
 311554 124598 144284.504   62 1225 149987.512 140399.767  6.39%  97.0 5055s
 312247 124929     cutoff   62      149987.512 140403.734  6.39%  97.0 5062s
 313029 124945 141804.411   60 1698 149987.512 140413.270  6.38%  97.0 5070s
 313110 125207 144672.622   63 1565 149987.512 140413.270  6.38%  97.0 5076s
 313781 125505 148316.833   71  784 149987.512 140421.305  6.38%  97.0 5082s
 314518 125505     cutoff   73      149987.512 140431.168  6.37%  97.0 5090s
 314540 126939 147941.600   68 1864 149987.512 140431.168  6.37%  97.0 5122s
 318540 126934     cutoff   68      149987.512 140472.923  6.34%  96.9 5127s
 318576 126972 148507.507   67 1232 149987.512 140473.218  6.34%  96.9 5130s
 318701 127024 148613.868   74  753 149987.512 140473.600  6.34%  96.9 5136s
 318796 127068 142127.068   56 1713 149987.512 140475.576  6.34%  96.9 5140s
 318897 127283 145283.365   67 1257 149987.512 140476.400  6.34%  96.9 5145s
 319675 127507 142137.796   43 2313 149987.512 140485.942  6.33%  96.9 5153s
 320111 127530 142792.217   44 2269 149987.512 140490.000  6.33%  96.9 5156s
 320162 127717     cutoff   64      149987.512 140492.022  6.33%  96.9 5161s
 320703 127940 148793.230   77  700 149987.512 140496.631  6.33%  96.9 5169s
 321201 128100 148415.199   70  801 149987.512 140504.297  6.32%  96.9 5174s
 321852 128105 142737.599   55 1290 149987.512 140509.826  6.32%  96.9 5180s
 321873 128348 142745.457   56 1287 149987.512 140510.661  6.32%  96.9 5185s
 323098 128673 146708.599   68 1260 149987.512 140519.819  6.31%  96.8 5193s
 323156 128890 149213.616   69 1038 149987.512 140519.819  6.31%  96.8 5197s
 323734 129078 149592.630   60 1503 149987.512 140524.732  6.31%  96.8 5202s
 324195 129089 147000.396   67 1492 149987.512 140527.549  6.31%  96.7 5206s
 324272 129332 141606.501   55 1934 149987.512 140529.225  6.31%  96.7 5210s
 324824 129476 148429.165   59 1851 149987.512 140532.472  6.30%  96.7 5218s
 325184 129488     cutoff   75      149987.512 140534.311  6.30%  96.7 5225s
 325222 129966 149588.470   63 1643 149987.512 140536.125  6.30%  96.7 5235s
 326501 129973 141167.314   45 1983 149987.512 140545.825  6.29%  96.7 5242s
 326521 130183 146706.668   48 1970 149987.512 140545.825  6.29%  96.7 5246s
 327066 130358 147353.241   62 1374 149987.512 140554.415  6.29%  96.7 5251s
 327602 130366     cutoff   70      149987.512 140559.350  6.29%  96.7 5257s
 327646 130440 149954.436   60 1647 149987.512 140559.350  6.29%  96.7 5261s
 327908 130558     cutoff   62      149987.512 140563.656  6.28%  96.7 5266s
 328280 130574     cutoff   63      149987.512 140568.705  6.28%  96.8 5274s
 328337 130840 145024.019   61 1444 149987.512 140570.012  6.28%  96.8 5279s
 328984 131098     cutoff   69      149987.512 140576.171  6.27%  96.8 5285s
 329725 131347 147670.384   53 1962 149987.512 140583.186  6.27%  96.8 5292s
 330302 131456 147634.479   67  715 149987.512 140585.776  6.27%  96.8 5297s
 330578 131744 149783.762   68 1615 149987.512 140586.457  6.27%  96.7 5302s
 331181 131757 145411.443   61 1239 149987.512 140593.865  6.26%  96.7 5306s
 331225 131988 142745.009   54 1920 149987.512 140594.876  6.26%  96.7 5311s
 331809 132095 145568.530   68 1529 149987.512 140597.597  6.26%  96.7 5315s
 332181 132113 148306.500   60 1559 149987.512 140601.702  6.26%  96.7 5320s
 332704 132426 144115.493   61 1063 149987.512 140606.450  6.25%  96.8 5328s
 333133 132441 142539.535   58 1572 149987.512 140611.879  6.25%  96.8 5334s
 333187 132720 144852.225   62 1545 149987.512 140611.971  6.25%  96.8 5340s
 333793 132856 148525.280   67 1128 149987.512 140616.312  6.25%  96.7 5345s
 334281 133078 148722.081   66  857 149987.512 140620.229  6.25%  96.8 5353s
 334877 133224     cutoff   50      149987.512 140624.489  6.24%  96.7 5358s
 335270 133276 148936.829   63 1262 149987.512 140628.508  6.24%  96.8 5363s
 335408 134217     cutoff   71      149987.512 140630.059  6.24%  96.8 5383s
 337878 134255 143811.008   54 1773 149987.512 140649.899  6.23%  96.7 5393s
 337951 134275 148488.670   61 1155 149987.512 140650.274  6.23%  96.7 5396s
 338040 134464 148895.831   73  597 149987.512 140650.274  6.23%  96.7 5400s

Cutting planes:
  Gomory: 140
  Cover: 441
  Implied bound: 396
  Projected implied bound: 581
  Clique: 97
  MIR: 4997
  Mixing: 4
  StrongCG: 5
  Flow cover: 2928
  Flow path: 1
  GUB cover: 2
  Inf proof: 81
  Zero half: 22
  Mod-K: 1
  RLT: 217
  Relax-and-lift: 197
  BQP: 4

Explored 338527 nodes (32753542 simplex iterations) in 5400.22 seconds (4282.72 work units)
Thread count was 8 (of 8 available processors)

Solution count 1: 149988 

Time limit reached
Best objective 1.499875117670e+05, best bound 1.406561935144e+05, gap 6.2214%

======================================================================
Leas=15000 ratio×1.0 LCM×1.0 bi×1.0 SL=0.97
======================================================================
  Status: TIME_LIMIT  |  Costo: 149,987.51€  |  Gap: 6.22%  |  Tempo: 5402.1s

  SKU  T   DC               B    c_CM    c_AM  ratio
  0    AM  [0]            1.0    2000    2632  1.316
  1    AM  [0]            1.0    2373    2937  1.237
  2    AM  [0]            1.0    3218    3367  1.046
  3    AM  [0]            1.0    3774    3589  0.951
  4    AM  [0]            1.0    4004    4333  1.082
  5    AM  [0]            1.0    4384    5135  1.171
  6    AM  [0]            1.0    5677    5387  0.949
  7    AM  [0]            1.0    6073    7633  1.257
  8    AM  [0]            1.0    6785    7633  1.125

  Stampanti: [1. 0. 0.]
  Costi:
    base_stock          :     53307.39€
    EOQ_raw             :       330.56€
    holding_raw         :       312.50€
    leasing_3DP         :     15000.00€
    backorder_LT        :     25506.28€
    trasporto           :      5321.04€
    produzione_AM       :     48843.57€
    holding_inv         :      1366.18€
    TOTALE              :    149987.51€

  DETTAGLIO ASSEGNAZIONE CLIENTI (y_sol):
    SKU 0 (AM) → DC0: clienti=[0, 1, 2, 3, 4, 5], μ_DC=3.00, B=1.0
    SKU 1 (AM) → DC0: clienti=[0, 1, 2, 3, 4, 5], μ_DC=2.70, B=1.0
    SKU 2 (AM) → DC0: clienti=[0, 1, 2, 3, 4, 5], μ_DC=2.50, B=1.0
    SKU 3 (AM) → DC0: clienti=[0, 1, 2, 3, 4, 5], μ_DC=2.10, B=1.0
    SKU 4 (AM) → DC0: clienti=[0, 1, 2, 3, 4, 5], μ_DC=2.00, B=1.0
    SKU 5 (AM) → DC0: clienti=[0, 1, 2, 3, 4, 5], μ_DC=1.80, B=1.0
    SKU 6 (AM) → DC0: clienti=[0, 1, 2, 3, 4, 5], μ_DC=1.60, B=1.0
    SKU 7 (AM) → DC0: clienti=[0, 1, 2, 3, 4, 5], μ_DC=1.40, B=1.0
    SKU 8 (AM) → DC0: clienti=[0, 1, 2, 3, 4, 5], μ_DC=1.30, B=1.0

  DETTAGLIO BASE STOCK PER DC:
    DC0: 9 SKU (9 AM), B_tot=9.0, stampanti=1

  BACKORDER LT ANALITICO PER SKU:
    SKU   Tech      bo_LT        b_i   L_tech  μ_tot    β    E[BO]
    0     AM       707.42      49995   0.0577   3.00    1   0.0141
    1     AM      1195.50      59332   0.0769   2.70    1   0.0201
    2     AM      2149.09      80460   0.0962   2.50    1   0.0267
    3     AM      2559.16      94356   0.1154   2.10    1   0.0271
    4     AM      3323.19     100103   0.1346   2.00    1   0.0332
    5     AM      2972.85     109609   0.1346   1.80    1   0.0271
    6     AM      3967.43     141918   0.1538   1.60    1   0.0280
    7     AM      4118.01     151831   0.1731   1.40    1   0.0271
    8     AM      4513.63     169627   0.1845   1.30    1   0.0266

  METRICHE ANOVA: n_AM=9, n_DC_CM=0.00, H̄=0.000, costo=149,988€

========================================================================
RUN: Leas=15000 ratio×1.0 LCM×1.0 bi×1.0 SL=0.97
========================================================================
  [CG] Precomputazione: 3,402 entry in 2.1s
  [CG] It   0: LB=   181,446  col=   94  new= 40
  [CG] It   5: LB=   181,446  col=  257  new= 24
  [CG] It  10: LB=   181,446  col=  282  new=  0
Set parameter OutputFlag to value 1
Set parameter TimeLimit to value 300
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: 11th Gen Intel(R) Core(TM) i7-1165G7 @ 2.80GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  300

Optimize a model with 57 rows, 285 columns and 995 nonzeros (Min)
Model fingerprint: 0x4cef3762
Model has 285 linear objective coefficients
Variable types: 0 continuous, 285 integer (282 binary)
Coefficient statistics:
  Matrix range     [1e+00, 5e+03]
  Objective range  [5e+03, 3e+04]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]

Found heuristic solution: objective 220480.86327
Presolve removed 45 rows and 255 columns
Presolve time: 0.00s
Presolved: 12 rows, 30 columns, 57 nonzeros
Found heuristic solution: objective 220127.63957
Variable types: 0 continuous, 30 integer (30 binary)
Found heuristic solution: objective 202122.86629

Root relaxation: objective 1.868098e+05, 7 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

*    0     0               0    186809.81261 186809.813  0.00%     -    0s

Explored 1 nodes (7 simplex iterations) in 0.07 seconds (0.00 work units)
Thread count was 8 (of 8 available processors)

Solution count 4: 186810 202123 220128 220481 

Optimal solution found (tolerance 1.00e-04)
Best objective 1.868098126074e+05, best bound 1.868098126074e+05, gap 0.0000%

  CG: 9 AM [0, 1, 2, 3, 4, 5, 6, 7, 8], costo=186,810, n3dp=[1, 0, 0], t=2.3s
Set parameter OutputFlag to value 1
Set parameter MIPGap to value 0.05
Set parameter TimeLimit to value 5400
  [vincolo] B_cap=5 attivo — max 5 unità BS per DC
Set parameter StartNodeLimit to value 100
  [warm start] MINIMALE: 9 AM, n3DP=[1, 0, 0]
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: 11th Gen Intel(R) Core(TM) i7-1165G7 @ 2.80GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  5400
MIPGap  0.05
StartNodeLimit  100

Optimize a model with 10128 rows, 9513 columns and 36207 nonzeros (Min)
Model fingerprint: 0x6dd41e68
Model has 9027 linear objective coefficients
Model has 141 simple general constraints
  141 PWL
Variable types: 9246 continuous, 267 integer (264 binary)
Coefficient statistics:
  Matrix range     [4e-03, 5e+05]
  Objective range  [1e+00, 2e+04]
  Bounds range     [1e+00, 1e+00]
  RHS range        [3e-02, 5e+05]
  PWLCon x range   [4e-02, 1e+02]
  PWLCon y range   [0e+00, 1e+04]

Using variable hints.
User MIP start did not produce a new incumbent solution

Presolve removed 4702 rows and 1842 columns
Presolve time: 0.12s
Presolved: 5426 rows, 7671 columns, 27225 nonzeros
Presolved model has 141 SOS constraint(s)
Variable types: 7386 continuous, 285 integer (258 binary)

Root relaxation: objective 3.419877e+04, 3307 iterations, 0.09 seconds (0.10 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

     0     0 34198.7673    0  219          - 34198.7673      -     -    0s
Another try with MIP start
     0     0 34198.7673    0  221          - 34198.7673      -     -    0s
     0     0 34925.3789    0  254          - 34925.3789      -     -    0s
     0     0 41327.3506    0  257          - 41327.3506      -     -    1s
     0     0 41327.3506    0  257          - 41327.3506      -     -    1s
     0     0 41327.3506    0  257          - 41327.3506      -     -    1s
     0     0 41327.3506    0  284          - 41327.3506      -     -    1s
     0     0 41327.3506    0  314          - 41327.3506      -     -    2s
     0     0 41327.3506    0  327          - 41327.3506      -     -    2s
     0     0 41327.3506    0  328          - 41327.3506      -     -    2s
     0     0 41327.3506    0  302          - 41327.3506      -     -    2s
     0     0 41327.3506    0  302          - 41327.3506      -     -    2s
     0     0 41327.3506    0  266          - 41327.3506      -     -    2s
     0     2 41327.3506    0  253          - 41327.3506      -     -    3s
*  146    84              31    156716.50343 47079.1369  70.0%   126    4s
   269   145 153793.633   20   42 156716.503 47219.8332  69.9%   126    5s
  1448   893 112296.342   10  141 156716.503 57398.3483  63.4%  76.9   10s
  1470   917 64782.9887   14 1723 156716.503 57398.3483  63.4%  89.8   15s
  1708  1042 150188.672   24 1146 156716.503 57684.6903  63.2%  96.8   20s
  2370  1220 139674.160   44  784 156716.503 64815.0798  58.6%  88.6   25s
  2841  1373 infeasible   70      156716.503 70978.6983  54.7%  90.1   30s
  3477  1579 infeasible   69      156716.503 79277.9530  49.4%  88.2   35s
  3908  1743 135606.610   28 1055 156716.503 82675.1431  47.2%  89.8   40s
  4212  1828 143013.830   39  597 156716.503 85283.9771  45.6%  89.5   45s
  4917  2392 142865.223   26  944 156716.503 90425.5557  42.3%  90.1   50s
  5831  2927 140695.305   28  667 156716.503 92965.6279  40.7%  85.9   55s
  6563  3383 148394.352   31 1075 156716.503 96503.2038  38.4%  83.8   60s
  7249  3936 150574.536   31 1058 156716.503 97094.1294  38.0%  83.6   66s
  8341  4376     cutoff   55      156716.503 100395.112  35.9%  80.6   71s
  8983  5072 149817.505   32 1029 156716.503 100843.763  35.7%  79.6   76s
  9938  5426 151861.741   40  277 156716.503 103676.370  33.8%  77.1   80s
 11125  6640 153329.918   47  193 156716.503 104591.618  33.3%  75.5   88s
*12022  6370              69    155014.73356 104684.013  32.5%  74.3   88s
 12260  6334     cutoff   50      155014.734 105422.628  32.0%  74.3   90s
H12272  6336                    155014.73351 105422.628  32.0%  74.3   92s
 12425  6489     cutoff   31      155014.734 106471.838  31.3%  74.6   96s
*12579  6371              70    154578.68648 106912.135  30.8%  74.2   96s
 12817  6818 140522.772   23  926 154578.686 107306.079  30.6%  74.5  100s
 13591  6896 146769.203   29  953 154578.686 108472.977  29.8%  74.4  106s
 13727  7319 125550.132   20 1393 154578.686 109107.383  29.4%  74.7  110s
 14542  7800 142690.765   34 1367 154578.686 110752.953  28.4%  74.2  115s
 16066  8326     cutoff   49      154578.686 112322.272  27.3%  72.9  121s
 16815  8772 133974.400   22 1323 154578.686 113051.940  26.9%  72.4  125s
H16984  8778                    154438.05058 113111.542  26.8%  72.1  128s
 17031  8874 153529.210   66  934 154438.051 113111.542  26.8%  72.0  130s
 17966  9371 129737.143   30 1207 154438.051 114000.366  26.2%  71.1  135s
 18891  9871 143375.294   22 1049 154438.051 114907.975  25.6%  70.4  140s
 19696 10719 infeasible   55      154438.051 115726.736  25.1%  69.8  145s
H21282 11207                    154438.04436 117041.133  24.2%  68.7  185s
H21284 10648                    154438.04316 117041.133  24.2%  68.7  188s
H21286 10117                    154438.03726 117041.133  24.2%  68.7  190s
 21292 10121 154159.837   43 2341 154438.037 117041.133  24.2%  68.7  195s
H21295  9616                    154438.03702 117041.133  24.2%  68.7  200s
 21299  9619 143078.352   59 2371 154438.037 117041.133  24.2%  68.7  206s
 21301  9620 153299.533   33 2346 154438.037 117041.133  24.2%  68.7  210s
 21305  9623 145388.653   33 2364 154438.037 117041.133  24.2%  68.7  217s
 21307  9624 147234.341   28 2362 154438.037 117041.133  24.2%  68.7  221s
 21310  9626 134495.780   26 2325 154438.037 117041.133  24.2%  68.6  225s
 21314  9629 133734.301   21 2360 154438.037 117041.133  24.2%  68.6  231s
 21317  9631 133689.293   21 2352 154438.037 117041.133  24.2%  68.6  235s
 21320  9633 145648.707   41 2393 154438.037 117041.133  24.2%  68.6  241s
 21322  9634 133370.929   23 2398 154438.037 117041.133  24.2%  68.6  245s
 21326  9637 133522.946   25 2408 154438.037 117041.133  24.2%  68.6  252s
 21328  9638 123472.455   21 2397 154438.037 117041.133  24.2%  68.6  255s
 21332  9641 145555.099   43 2400 154438.037 117041.133  24.2%  68.6  261s
 21335  9643 148389.476   24 2404 154438.037 117041.133  24.2%  68.6  265s
 21338  9645 137466.797   27 2406 154438.037 117041.133  24.2%  68.6  270s
 21342  9647 144082.218   31 2383 154438.037 117041.133  24.2%  68.5  276s
 21346  9650 141732.715   27 2385 154438.037 117041.133  24.2%  68.5  282s
 21348  9651 150863.280   33 2388 154438.037 117041.133  24.2%  68.5  285s
 21352  9654 136016.493   23 2399 154438.037 117041.133  24.2%  68.5  290s
 21355  9656 142337.504   36 2398 154438.037 117041.133  24.2%  68.5  296s
 21357  9657 152730.244   35 2400 154438.037 117041.133  24.2%  68.5  300s
 21361  9660 150179.893   37 2399 154438.037 117041.133  24.2%  68.5  305s
 21364  9662 136005.902   24 2386 154438.037 117041.133  24.2%  68.5  310s
 21367  9664 142187.992   27 2391 154438.037 117109.221  24.2%  68.5  315s
 21371  9667 150413.606   38 2392 154438.037 117170.421  24.1%  68.4  321s
 21375  9669 147901.088   23 2395 154438.037 117250.707  24.1%  68.4  326s
 21379  9672 142686.018   28 2393 154438.037 117325.546  24.0%  68.4  332s
 21381  9673 137307.687   21 2393 154438.037 117362.827  24.0%  68.4  335s
 21385  9676 152006.609   23 2393 154438.037 117434.670  24.0%  68.4  341s
 21389  9679 149713.673   42 2388 154438.037 117500.991  23.9%  68.4  347s
 21393  9681 132237.403   21 2388 154438.037 117520.835  23.9%  68.4  350s
 21398  9690 117520.835   22 2069 154438.037 117520.835  23.9%  70.9  355s
 21513  9718 136117.768   26 1761 154438.037 117520.835  23.9%  72.5  360s
 21637  9744 140146.981   27 2259 154438.037 117520.835  23.9%  73.3  365s
 21783  9773 133426.850   29 1257 154438.037 117520.835  23.9%  73.8  370s
 21990  9821 121762.377   33 1564 154438.037 117520.835  23.9%  74.1  375s
 22129  9865 135768.711   37  893 154438.037 117520.835  23.9%  74.6  380s
 22237  9874 132297.881   35 1586 154438.037 117520.835  23.9%  74.9  387s
 22296  9885 139673.220   48 1295 154438.037 117773.964  23.7%  75.0  390s
 22354  9903 129619.958   26 2260 154438.037 117977.288  23.6%  75.3  397s
 22423  9937 137642.914   37 1787 154438.037 117977.288  23.6%  75.4  400s
 22611  9962 137982.190   49 1413 154438.037 117977.288  23.6%  75.6  405s
 22830  9979 146605.110   66 1029 154438.037 118495.506  23.3%  75.9  412s
 22939  9974 119285.524   35 1714 154438.037 118562.926  23.2%  75.9  417s
 22985 10004 134736.073   38 1674 154438.037 118666.863  23.2%  76.1  420s
 23174  9995     cutoff   68      154438.037 118728.049  23.1%  76.0  425s
 23465 10036 infeasible   38      154438.037 118864.800  23.0%  76.4  432s
 23702 10064 141339.522   51 1244 154438.037 119090.724  22.9%  76.8  437s
 23920 10090 151556.800   59  965 154438.037 119309.857  22.7%  77.1  442s
 24044 10125 152602.832   49  871 154438.037 119426.454  22.7%  77.3  445s
 24460 10079 120847.491   46 1513 154438.037 119786.297  22.4%  77.6  452s
 24479 10113 125436.527   47 1497 154438.037 119786.297  22.4%  77.7  455s
 24707 10108 135166.801   43 1424 154438.037 120088.549  22.2%  78.1  462s
 24818 10200 147775.437   37 1353 154438.037 120356.537  22.1%  78.5  466s
 25142 10156 infeasible   45      154438.037 120708.386  21.8%  78.7  470s
 25563 10172 141596.737   43 1419 154438.037 121045.444  21.6%  79.2  478s
 25772 10200 134094.346   37 1609 154438.037 121357.506  21.4%  79.8  482s
 26016 10201 138362.242   41 1631 154438.037 121654.286  21.2%  80.4  487s
 26228 10373 137654.025   36 1129 154438.037 121789.174  21.1%  80.7  492s
 26851 10221 140117.375   51 1447 154438.037 122229.458  20.9%  81.3  496s
 27077 10261 infeasible   41      154438.037 122495.713  20.7%  81.9  501s
 27405 10271 152303.159   41 1192 154438.037 122906.098  20.4%  82.8  505s
 27705 10279 151450.895   56 1076 154438.037 123117.705  20.3%  83.5  510s
 28025 10224 129481.260   41 1434 154438.037 123570.407  20.0%  84.4  515s
 28222 10395 141493.140   46 1318 154438.037 123714.875  19.9%  85.0  520s
 28811 10285 152129.190   43  979 154438.037 124307.129  19.5%  86.0  526s
 29124 10246 138179.936   41 1460 154438.037 124531.638  19.4%  86.9  531s
 29387 10300 135539.062   49 1344 154438.037 124677.763  19.3%  87.9  537s
 29876 10219 126505.285   31 1735 154438.037 125107.931  19.0%  89.0  543s
 30088 10387 142715.252   47 1562 154438.037 125332.322  18.8%  89.4  550s
 30792 10249 140297.573   47 1422 154438.037 125891.686  18.5%  90.6  556s
 31150 10435 137306.065   34 1569 154438.037 126242.366  18.3%  91.3  563s
 31959 10297 infeasible   42      154438.037 126452.029  18.1%  91.8  572s
 32306 10614 135405.857   30 1684 154438.037 126743.962  17.9%  92.2  582s
 33557 10291     cutoff   62      154438.037 127356.688  17.5%  93.7  587s
 33805 10293 151693.898   39  959 154438.037 127501.654  17.4%  94.1  591s
 34068 10218 140621.818   42  671 154438.037 127673.871  17.3%  94.2  597s
 34099 10286 149113.367   46  606 154438.037 127673.871  17.3%  94.3  603s
 34383 10264 141052.348   49 1512 154438.037 127865.973  17.2%  94.5  608s
 34614 10356 140450.185   42 1439 154438.037 127941.097  17.2%  94.6  614s
 35101 10269 148783.715   42 1466 154438.037 128320.614  16.9%  94.9  619s
 35397 10228 146724.293   34 1025 154438.037 128509.974  16.8%  95.3  625s
 35624 10416 145793.174   31 1664 154438.037 128755.301  16.6%  95.8  631s
 36313 10272 146489.767   43 1502 154438.037 128977.930  16.5%  96.2  637s
 36614 10426 151001.291   39 1537 154438.037 129178.793  16.4%  96.5  643s
 37355 10267 141322.839   46 1436 154438.037 129649.299  16.1%  97.1  650s
 37643 10390 143945.495   43 1570 154438.037 129740.707  16.0%  97.4  655s
 38196 10434 135133.620   29 1214 154438.037 129987.379  15.8%  97.3  661s
 38746 10378 150442.189   43  830 154438.037 130192.052  15.7%  97.4  666s
 39131 10417 132879.813   36 1435 154438.037 130411.225  15.6%  97.5  672s
 39699 10315 140266.246   38 1382 154438.037 130566.664  15.5%  97.6  676s
 39864 10631 138640.007   42 1199 154438.037 130671.752  15.4%  97.6  682s
 40836 10400 147747.002   43  583 154438.037 130861.649  15.3%  97.2  687s
 41102 10734 143786.226   36 1309 154438.037 130877.133  15.3%  97.1  693s
 42057 10466 150516.204   49  658 154438.037 131052.704  15.1%  96.8  698s
 42207 10677 140308.894   34  992 154438.037 131077.022  15.1%  96.8  704s
 42803 10590 144568.125   42 1198 154438.037 131155.003  15.1%  96.8  709s
 43154 10637 131260.882   34 1352 154438.037 131205.263  15.0%  96.9  714s
 43518 10873 143763.794   33 1544 154438.037 131247.125  15.0%  96.8  720s
 44430 10618 infeasible   53      154438.037 131338.157  15.0%  96.7  725s
 45151 10663 140979.878   39 1205 154438.037 131464.593  14.9%  96.7  734s
 45331 10887 149062.844   36 1493 154438.037 131483.851  14.9%  96.7  738s
 46151 10668 134288.923   31 1090 154438.037 131572.982  14.8%  96.4  743s
 46314 10820 152853.545   43  881 154438.037 131616.130  14.8%  96.5  747s
 46855 10641 infeasible   42      154438.037 131668.644  14.7%  96.4  751s
 46882 10686 151140.705   39 1031 154438.037 131668.644  14.7%  96.4  755s
 47240 10916 134053.660   32 1462 154438.037 131761.370  14.7%  96.5  763s
 47964 10739 140421.909   28 1586 154438.037 131856.177  14.6%  96.3  769s
 48115 11031 143819.812   33 1391 154438.037 131856.177  14.6%  96.3  773s
 48933 10844 153145.374   49  606 154438.037 131953.752  14.6%  95.9  777s
 49187 11044 138131.940   35 1226 154438.037 131987.772  14.5%  96.0  781s
 49920 10854     cutoff   44      154438.037 132024.003  14.5%  95.7  786s
 50143 11046 141972.291   36 1402 154438.037 132064.150  14.5%  95.7  790s
 50853 11009 146930.967   42 1358 154438.037 132117.970  14.5%  95.5  795s
 51585 11315 147375.884   43 1052 154438.037 132209.130  14.4%  95.2  803s
 51758 11585 143118.588   34 1138 154438.037 132224.305  14.4%  95.2  807s
 52480 11732 144135.247   31 1573 154438.037 132296.852  14.3%  94.8  812s
 52920 12072 150178.938   40 1152 154438.037 132372.665  14.3%  94.7  819s
 53840 12137 149426.589   44  932 154438.037 132459.828  14.2%  94.4  823s
 54019 12311 136526.598   35 1246 154438.037 132495.948  14.2%  94.4  827s
 54500 12398 152257.864   37 1212 154438.037 132541.116  14.2%  94.3  831s
 54757 12663 146623.246   39 1158 154438.037 132566.198  14.2%  94.3  836s
 55625 12913 132801.472   34 1472 154438.037 132641.339  14.1%  94.3  843s
 56219 12946     cutoff   51      154438.037 132669.240  14.1%  94.1  846s
 56388 13142 145149.167   38  943 154438.037 132681.493  14.1%  94.1  850s
 57250 13437 144030.243   41 1063 154438.037 132775.000  14.0%  93.9  857s
 57911 13539 149805.908   37  388 154438.037 132820.493  14.0%  93.7  861s
 58148 13778 142378.512   37 1136 154438.037 132837.269  14.0%  93.6  865s
 59067 14067 141221.389   35 1195 154438.037 132953.221  13.9%  93.4  872s
 59677 14133 139529.426   37 1020 154438.037 133009.229  13.9%  93.2  876s
 59851 14377 144449.883   42  749 154438.037 133030.753  13.9%  93.2  880s
 60797 14625 150088.740   39  426 154438.037 133142.567  13.8%  92.9  887s
 61430 14692 146726.832   36 1368 154438.037 133182.278  13.8%  92.8  890s
 62265 14988 151201.014   39  611 154438.037 133257.179  13.7%  92.7  898s
 62557 15103 133893.994   30 1053 154438.037 133289.589  13.7%  92.6  901s
 62817 15207 145313.101   42  830 154438.037 133289.589  13.7%  92.6  906s
 63210 15554 151382.092   42 1263 154438.037 133332.944  13.7%  92.5  913s
 64281 15593 147121.312   37 1082 154438.037 133418.947  13.6%  92.3  917s
 64420 15658 137341.204   36 1094 154438.037 133431.635  13.6%  92.3  920s
 65304 15963     cutoff   48      154438.037 133532.558  13.5%  92.1  927s
 65435 16074 142195.128   47 1584 154438.037 133562.123  13.5%  92.1  931s
 66034 16354 138562.255   33 1154 154438.037 133620.558  13.5%  92.1  939s
 66592 16395 152921.320   47 1087 154438.037 133644.086  13.5%  92.0  942s
 66777 16433 145821.251   44 1113 154438.037 133663.096  13.5%  91.9  946s
 66926 16731 148377.664   43  847 154438.037 133665.517  13.5%  92.0  951s
 67790 16989 142438.744   38  893 154438.037 133781.331  13.4%  91.8  958s
 68397 16998 135843.178   37 1323 154438.037 133856.173  13.3%  91.6  962s
 68423 17000 141071.633   40  936 154438.037 133856.173  13.3%  91.6  967s
 68429 17004 142355.284   41  917 154438.037 133861.768  13.3%  91.6  971s
 68433 17013 146893.426   42  898 154438.037 133861.768  13.3%  91.6  975s
 68561 17071 141485.899   41 1306 154438.037 133870.790  13.3%  91.6  982s
 68639 17092 146479.123   39  940 154438.037 133872.624  13.3%  91.6  988s
 68754 17151 135968.514   33 1013 154438.037 133892.951  13.3%  91.7  991s
 68987 17233 139027.811   31 1279 154438.037 133926.391  13.3%  91.7  995s
 69446 17324 139408.684   30 1828 154438.037 133959.059  13.3%  91.6 1004s
 69641 17592 141801.887   34 1166 154438.037 133961.974  13.3%  91.6 1009s
 70453 17692 infeasible   36      154438.037 134013.405  13.2%  91.3 1013s
 70705 17893 143224.459   37 1071 154438.037 134048.233  13.2%  91.3 1018s
 71390 17965     cutoff   46      154438.037 134090.201  13.2%  91.2 1022s
 71625 18177 147478.811   45 1057 154438.037 134111.459  13.2%  91.2 1027s
 72353 18227 infeasible   54      154438.037 134178.012  13.1%  91.0 1031s
 72601 18310 149709.390   40  468 154438.037 134192.432  13.1%  91.0 1035s
 73347 18593 143837.615   36  359 154438.037 134249.153  13.1%  90.9 1044s
 73795 18914 146241.750   37  892 154438.037 134303.721  13.0%  90.9 1049s
 74701 18950 143150.056   39 1087 154438.037 134364.700  13.0%  90.7 1052s
 74839 19018 infeasible   50      154438.037 134382.869  13.0%  90.7 1056s
 75083 19226     cutoff   38      154438.037 134406.963  13.0%  90.8 1060s
 75893 19344     cutoff   38      154438.037 134474.138  12.9%  90.8 1067s
 76151 19521 142431.888   39  765 154438.037 134489.655  12.9%  90.7 1071s
 77073 19660 134943.616   33 1168 154438.037 134555.047  12.9%  90.5 1078s
 77411 19776 137616.915   39 1438 154438.037 134592.700  12.9%  90.5 1081s
 77739 19863 145272.780   32  402 154438.037 134615.452  12.8%  90.5 1085s
 77997 20059     cutoff   41      154438.037 134642.372  12.8%  90.4 1090s
 78874 20316 149772.126   43  542 154438.037 134704.279  12.8%  90.2 1098s
 79526 20392 147787.407   37  589 154438.037 134754.367  12.7%  90.1 1103s
 79744 20571 144062.336   38  802 154438.037 134762.900  12.7%  90.2 1108s
 80387 20626 141600.126   39 1161 154438.037 134824.201  12.7%  90.0 1112s
 80538 20778 151235.814   48 1080 154438.037 134842.984  12.7%  90.0 1116s
 81180 21030 145769.140   35 1201 154438.037 134918.651  12.6%  89.9 1123s
 81896 21083 153442.321   40 1002 154438.037 134970.507  12.6%  89.8 1126s
 82082 21229 149294.913   35  416 154438.037 134987.279  12.6%  89.8 1131s
 82681 21367 140097.566   36 1253 154438.037 135068.541  12.5%  89.7 1137s
 83023 21421 143327.143   36 1193 154438.037 135093.086  12.5%  89.8 1141s
 83235 21624 150518.449   43  795 154438.037 135112.515  12.5%  89.8 1145s
 83865 21696 149903.545   46 1570 154438.037 135175.241  12.5%  89.6 1153s
 84102 22303 142962.730   33 1267 154438.037 135185.547  12.5%  89.7 1167s
 86213 22326 150800.157   36 1422 154438.037 135373.384  12.3%  89.5 1173s
 86344 22380 144089.791   37 1067 154438.037 135394.141  12.3%  89.5 1177s
 86512 22440 138608.225   43 1341 154438.037 135397.682  12.3%  89.5 1180s
 86743 22698 143351.818   40 1234 154438.037 135421.575  12.3%  89.5 1185s
 87527 22739 143244.449   31 1370 154438.037 135482.707  12.3%  89.4 1191s
 87709 22940 148961.372   37  840 154438.037 135503.427  12.3%  89.4 1195s
 88498 23064 145539.662   35 1112 154438.037 135571.447  12.2%  89.4 1203s
 88727 23118 142333.970   37  980 154438.037 135583.857  12.2%  89.4 1208s
 88871 23379 150452.041   42  895 154438.037 135596.430  12.2%  89.4 1212s
 89569 23453 138061.661   47 1337 154438.037 135636.271  12.2%  89.4 1216s
 90193 23631 146062.829   49  754 154438.037 135682.921  12.1%  89.4 1223s
 90495 23851 153593.343   42  460 154438.037 135718.453  12.1%  89.4 1227s
 91133 23903 138626.796   34  871 154438.037 135740.147  12.1%  89.4 1232s
 91369 24087 148462.067   39 1269 154438.037 135752.128  12.1%  89.4 1237s
 92098 24168 151096.313   59  509 154438.037 135820.145  12.1%  89.3 1241s
 92399 24336 145591.267   29 1638 154438.037 135842.135  12.0%  89.2 1245s
 93077 24549 144408.208   43 1081 154438.037 135897.907  12.0%  89.2 1253s
 93696 24597 138219.916   45 1444 154438.037 135932.104  12.0%  89.2 1258s
 93874 24816 147694.357   33 1449 154438.037 135938.614  12.0%  89.2 1262s
 94565 24869 142252.782   45 1444 154438.037 135999.103  11.9%  89.1 1268s
 94806 25049 138220.912   35 1494 154438.037 136034.547  11.9%  89.1 1277s
 95598 25398 141476.752   32 1071 154438.037 136071.404  11.9%  89.1 1284s
 96487 25398 149482.228   48  789 154438.037 136081.430  11.9%  89.0 1285s
 97079 25531 147428.871   35 1650 154438.037 136152.182  11.8%  89.0 1291s
 97257 25653 154235.301   50  574 154438.037 136166.069  11.8%  89.0 1295s
 98058 25879 140430.612   42 1016 154438.037 136216.625  11.8%  88.9 1303s
 98563 25959 infeasible   45      154438.037 136242.887  11.8%  88.9 1307s
 98841 26046 149686.766   43  910 154438.037 136242.887  11.8%  88.9 1311s
 99064 26225 147490.764   40  786 154438.037 136242.887  11.8%  88.9 1316s
 99657 26287 infeasible   33      154438.037 136322.541  11.7%  88.8 1320s
 100186 26494 147602.032   39  459 154438.037 136364.216  11.7%  88.9 1328s
 100583 26609 153189.302   52  653 154438.037 136387.494  11.7%  88.9 1332s
 100884 26838 138515.823   35 1582 154438.037 136404.893  11.7%  88.8 1337s
 101691 26931 infeasible   39      154438.037 136439.467  11.7%  88.7 1340s
 102426 27189 153982.469   49  609 154438.037 136487.049  11.6%  88.7 1348s
 102750 27431 infeasible   45      154438.037 136513.019  11.6%  88.7 1353s
 103569 27483 150183.221   42 1006 154438.037 136536.716  11.6%  88.5 1356s
 103779 27606 infeasible   46      154438.037 136544.720  11.6%  88.5 1360s
 104480 27753 152395.822   44  277 154438.037 136600.415  11.6%  88.5 1367s
 104752 27999 142796.557   43 1217 154438.037 136632.114  11.5%  88.6 1372s
*105023 27772              38    154097.57065 136632.114  11.3%  88.5 1372s
 105480 27833 137384.343   34 1370 154097.571 136675.844  11.3%  88.5 1377s
 105728 27935 infeasible   37      154097.571 136688.854  11.3%  88.4 1384s
 106182 28240 146697.972   49  950 154097.571 136694.590  11.3%  88.5 1392s
 107265 28306 148057.112   42  863 154097.571 136769.883  11.2%  88.4 1397s
 107531 28374 145219.318   36 1511 154097.571 136794.141  11.2%  88.4 1402s
 107789 28487 138080.592   33 1080 154097.571 136817.401  11.2%  88.4 1406s
 108112 28708     cutoff   43      154097.571 136838.028  11.2%  88.4 1411s
 108873 28773 151138.018   51  787 154097.571 136870.308  11.2%  88.3 1417s
 109128 28895 144875.010   48  957 154097.571 136893.670  11.2%  88.3 1422s
 109739 28954     cutoff   40      154097.571 136935.237  11.1%  88.4 1427s
 110014 29027 140258.663   32  811 154097.571 136945.774  11.1%  88.4 1431s
 110348 29114 147810.244   47  844 154097.571 136951.539  11.1%  88.4 1436s
 110577 29363 144788.198   36 1153 154097.571 136969.173  11.1%  88.4 1441s
 111406 29453 152113.263   52  755 154097.571 137034.624  11.1%  88.3 1445s
 111776 29532     cutoff   37      154097.571 137063.225  11.1%  88.3 1450s
 112068 29652 143029.863   43 1549 154097.571 137076.635  11.0%  88.3 1455s
 112596 29710 147659.699   41 1251 154097.571 137116.888  11.0%  88.4 1460s
 112825 29778 144755.467   36  801 154097.571 137127.884  11.0%  88.4 1465s
 113252 30031     cutoff   51      154097.571 137137.364  11.0%  88.4 1474s
 113986 30076 144512.426   40 1105 154097.571 137193.828  11.0%  88.5 1480s
 114224 30126 148531.839   42  914 154097.571 137210.835  11.0%  88.5 1485s
 114483 30392 infeasible   36      154097.571 137222.497  11.0%  88.5 1490s
 115754 30576 140104.944   31 1504 154097.571 137288.001  10.9%  88.4 1499s
 116174 30789     cutoff   37      154097.571 137308.874  10.9%  88.4 1505s
 116977 30830 147586.925   35  800 154097.571 137344.249  10.9%  88.3 1510s
 117451 31130 143425.422   36 1325 154097.571 137371.575  10.9%  88.3 1519s
 118256 31165 151274.148   39  588 154097.571 137413.676  10.8%  88.2 1525s
H118300 31165                    154097.56922 137413.676  10.8%  88.2 1525s
 118698 31432 139607.879   37 1108 154097.569 137436.431  10.8%  88.2 1532s
H119323 31456                    154097.56891 137457.662  10.8%  88.2 1536s
 119444 31519 149805.031   40  845 154097.569 137465.588  10.8%  88.2 1542s
 119641 31698 150240.141   42  363 154097.569 137476.248  10.8%  88.2 1546s
 120266 31738 infeasible   42      154097.569 137520.246  10.8%  88.2 1551s
 120390 31902 143110.894   39 1536 154097.569 137520.881  10.8%  88.2 1555s
 121231 32117 147760.335   39  820 154097.569 137564.402  10.7%  88.2 1564s
H121513 32117                    154097.56884 137564.402  10.7%  88.2 1564s
 121785 32180 151910.837   41  856 154097.569 137582.974  10.7%  88.2 1569s
 122036 32213 146610.642   45  858 154097.569 137601.789  10.7%  88.2 1573s
 122230 32340 148813.117   45  705 154097.569 137623.323  10.7%  88.2 1577s
*122534 31516              41    153379.22435 137623.323  10.3%  88.2 1578s
 122740 31602 148071.419   37  865 153379.224 137657.534  10.3%  88.2 1581s
 123067 31705 147432.222   45  904 153379.224 137681.306  10.2%  88.2 1585s
 123604 31781 139407.933   43 1463 153379.224 137715.310  10.2%  88.2 1590s
 124226 31990 143906.522   33 1492 153379.224 137752.245  10.2%  88.2 1598s
 124712 32032 151317.081   41  419 153379.224 137770.159  10.2%  88.3 1602s
 124909 32223 142705.734   48 1228 153379.224 137791.082  10.2%  88.2 1607s
 125669 32335 147967.992   39  461 153379.224 137811.872  10.1%  88.1 1611s
 126074 32448 151042.613   37 1427 153379.224 137831.379  10.1%  88.1 1617s
 126370 32758 150581.274   44 1023 153379.224 137843.052  10.1%  88.1 1625s
 127587 32793     cutoff   41      153379.224 137909.998  10.1%  88.1 1630s
 127717 32808 150540.268   40 1424 153379.224 137920.008  10.1%  88.1 1635s
 128041 33027 147156.675   45 1009 153379.224 137951.703  10.1%  88.2 1643s
 128697 33067 146940.501   39  736 153379.224 137991.294  10.0%  88.2 1647s
 128899 33142 147694.954   33 1601 153379.224 138000.028  10.0%  88.2 1651s
 129207 33211 140392.969   41 1504 153379.224 138019.537  10.0%  88.2 1655s
 130014 33410     cutoff   35      153379.224 138052.866  10.0%  88.2 1664s
 130477 33510 140313.664   39 1248 153379.224 138083.841  10.0%  88.2 1669s
 130854 33737 148590.177   51 1128 153379.224 138094.845  10.0%  88.2 1673s
 131689 33797     cutoff   54      153379.224 138133.131  9.94%  88.1 1679s
 131982 33858 infeasible   36      153379.224 138150.919  9.93%  88.1 1683s
 132252 34033 152455.015   45  859 153379.224 138152.193  9.93%  88.1 1687s
 132902 34090 139513.649   37 1135 153379.224 138200.225  9.90%  88.1 1693s
 133167 34165     cutoff   45      153379.224 138205.342  9.89%  88.1 1697s
 133576 34235 147954.017   35 1372 153379.224 138227.343  9.88%  88.1 1701s
 133871 34301 infeasible   46      153379.224 138247.524  9.87%  88.1 1706s
 134059 34516 151840.782   38 1323 153379.224 138251.938  9.86%  88.1 1710s
 134854 34571 148821.923   40  773 153379.224 138279.885  9.84%  88.0 1715s
 135042 34758 150860.454   38  655 153379.224 138280.676  9.84%  88.1 1720s
 135928 34855 148448.157   37 1022 153379.224 138340.072  9.81%  88.0 1727s
 136249 34917 152694.917   43 1231 153379.224 138354.884  9.80%  88.1 1733s
 136529 35005 148057.658   44  692 153379.224 138359.455  9.79%  88.1 1739s
 136913 35238 152483.659   38 1341 153379.224 138383.463  9.78%  88.1 1745s
 138253 35353 146043.680   51 1288 153379.224 138446.096  9.74%  88.0 1752s
 138521 35395     cutoff   37      153379.224 138455.923  9.73%  88.0 1756s
 139134 35611     cutoff   48      153379.224 138475.097  9.72%  88.0 1763s
 139683 35649 141328.074   38 1204 153379.224 138510.029  9.69%  88.0 1768s
 139898 35861 141320.511   43  847 153379.224 138512.355  9.69%  87.9 1773s
 140781 35942     cutoff   45      153379.224 138549.225  9.67%  87.8 1777s
 141095 36024 infeasible   42      153379.224 138566.455  9.66%  87.8 1781s
 141727 36301 infeasible   40      153379.224 138594.978  9.64%  87.7 1788s
 142382 36355 149216.401   37  864 153379.224 138620.095  9.62%  87.7 1791s
 142565 36415 142807.293   36  869 153379.224 138622.123  9.62%  87.7 1795s
 143335 36615     cutoff   45      153379.224 138658.181  9.60%  87.7 1802s
 143623 36696 148080.858   46 1130 153379.224 138665.818  9.59%  87.7 1806s
 144181 36730 148090.756   45 1092 153379.224 138686.607  9.58%  87.7 1811s
 144352 36933 141235.983   45  868 153379.224 138689.519  9.58%  87.7 1815s
 145456 37010 143473.837   42 1419 153379.224 138743.284  9.54%  87.5 1822s
 145751 37128     cutoff   43      153379.224 138752.387  9.54%  87.5 1825s
 146550 37215     cutoff   40      153379.224 138777.171  9.52%  87.5 1832s
 146806 37305 140232.992   34 1034 153379.224 138792.622  9.51%  87.5 1836s
 147256 37398 141596.767   41  910 153379.224 138805.207  9.50%  87.4 1841s
 147805 37458 144802.062   29 1484 153379.224 138832.931  9.48%  87.5 1846s
 148065 37618 141519.697   42 1092 153379.224 138844.514  9.48%  87.5 1851s
 148848 37668 150626.912   43 1146 153379.224 138869.465  9.46%  87.4 1857s
 149052 37767 141668.892   38 1117 153379.224 138880.356  9.45%  87.5 1861s
 149588 37811 150640.640   36 1186 153379.224 138897.282  9.44%  87.4 1866s
 149844 37972 143964.259   38  835 153379.224 138915.768  9.43%  87.5 1871s
 150607 38026 142309.350   33 1085 153379.224 138946.875  9.41%  87.4 1876s
 150902 38081 142775.156   38 1087 153379.224 138952.696  9.41%  87.4 1881s
 151177 38189 149518.520   44 1166 153379.224 138961.885  9.40%  87.4 1885s
 151785 38236 142468.391   35 1119 153379.224 138984.993  9.38%  87.4 1891s
 152023 38361 142647.590   38 1495 153379.224 138991.750  9.38%  87.4 1895s
 152736 38415 141575.480   35 1367 153379.224 139020.709  9.36%  87.4 1900s
 153354 38618 144627.434   34 1202 153379.224 139042.849  9.35%  87.3 1908s
 154015 38633 140839.040   37 1178 153379.224 139062.835  9.33%  87.3 1913s
 154208 38764 145394.499   41 1096 153379.224 139062.835  9.33%  87.3 1917s
 154833 38799     cutoff   38      153379.224 139095.016  9.31%  87.2 1921s
 155005 38867     cutoff   39      153379.224 139096.929  9.31%  87.3 1925s
 155708 38994 143183.742   43 1219 153379.224 139121.086  9.30%  87.3 1933s
 156038 39075 146391.602   34  706 153379.224 139137.814  9.29%  87.3 1937s
 156446 39127 143132.538   35  925 153379.224 139157.716  9.27%  87.3 1940s
 157167 39303 infeasible   43      153379.224 139184.264  9.25%  87.3 1948s
 157457 39473 148945.543   36 1378 153379.224 139196.197  9.25%  87.4 1956s
 158501 39509 141069.217   42 1455 153379.224 139243.261  9.22%  87.4 1961s
 158732 39536 146769.634   42 1223 153379.224 139252.222  9.21%  87.4 1965s
 158964 39691 145550.108   40 1427 153379.224 139259.482  9.21%  87.4 1970s
 159794 39755 infeasible   38      153379.224 139282.956  9.19%  87.3 1976s
 160101 39805 144474.807   39  746 153379.224 139297.366  9.18%  87.3 1980s
 161028 40005 149522.298   40 1089 153379.224 139325.699  9.16%  87.2 1990s
 161662 40112 146526.886   30 2035 153379.224 139354.364  9.14%  87.3 1997s
 161920 40272 146429.797   42  871 153379.224 139361.027  9.14%  87.2 2001s
 162595 40312 146528.138   41  798 153379.224 139380.596  9.13%  87.2 2005s
 162794 40342 148999.354   34 1197 153379.224 139388.342  9.12%  87.2 2010s
 162996 40373 146335.940   37 1109 153379.224 139399.647  9.11%  87.3 2015s
 163463 40485 147504.993   47  960 153379.224 139426.983  9.10%  87.3 2022s
 163609 40608 140638.048   47 1072 153379.224 139431.453  9.09%  87.3 2027s
 164296 40665 143357.801   43  521 153379.224 139447.705  9.08%  87.3 2031s
 164641 40728 144819.692   35 1143 153379.224 139466.104  9.07%  87.3 2035s
 165346 40868     cutoff   48      153379.224 139489.386  9.06%  87.3 2043s
*165387 40335              43    153102.29708 139489.386  8.89%  87.3 2043s
H165849 40370                    153102.29701 139500.961  8.88%  87.3 2047s
 166046 40440 146838.199   39 1010 153102.297 139507.772  8.88%  87.3 2052s
 166489 40472 145199.420   36  791 153102.297 139522.998  8.87%  87.3 2056s
 166652 40538     cutoff   44      153102.297 139528.875  8.87%  87.3 2061s
 167043 40639 150467.624   41 1250 153102.297 139541.009  8.86%  87.3 2066s
 167556 40714     cutoff   42      153102.297 139558.671  8.85%  87.3 2074s
 167903 40982 148486.489   42  731 153102.297 139564.123  8.84%  87.3 2083s
 169190 41034     cutoff   43      153102.297 139613.149  8.81%  87.3 2086s
 169443 41089 151136.410   42  663 153102.297 139614.659  8.81%  87.3 2090s
 169712 41112     cutoff   43      153102.297 139623.698  8.80%  87.3 2096s
 169968 41191     cutoff   38      153102.297 139637.380  8.79%  87.4 2100s
 170343 41316 141045.489   44 1126 153102.297 139649.213  8.79%  87.3 2105s
 171309 41449 143824.972   40 1212 153102.297 139682.094  8.77%  87.3 2114s
 171692 41477 143304.308   48  961 153102.297 139693.164  8.76%  87.3 2118s
 172023 41524     cutoff   44      153102.297 139697.323  8.76%  87.4 2122s
 172376 41625 150993.353   41 1405 153102.297 139714.480  8.74%  87.4 2126s
 172900 41687 150604.572   39  450 153102.297 139733.287  8.73%  87.3 2131s
 173204 41745 142449.396   41 1189 153102.297 139738.052  8.73%  87.3 2136s
 173497 41809     cutoff   43      153102.297 139752.578  8.72%  87.3 2140s
 173820 41876 140376.869   34 1036 153102.297 139766.132  8.71%  87.3 2146s
 174237 41911 148337.299   45  797 153102.297 139776.523  8.70%  87.3 2151s
 174422 41947 150883.583   42 1037 153102.297 139776.523  8.70%  87.3 2158s
 174677 42030 144161.400   40 1269 153102.297 139785.787  8.70%  87.3 2164s
 175165 42101 141891.392   38  986 153102.297 139807.315  8.68%  87.3 2168s
 175645 42178 infeasible   43      153102.297 139820.182  8.68%  87.2 2174s
 176134 42254 infeasible   34      153102.297 139846.292  8.66%  87.2 2178s
 176495 42400 144665.574   34  971 153102.297 139859.292  8.65%  87.2 2184s
 177346 42498 149787.996   38  705 153102.297 139884.755  8.63%  87.1 2188s
 177750 42561     cutoff   41      153102.297 139897.999  8.62%  87.1 2193s
 178171 42641 150388.526   47 1094 153102.297 139918.710  8.61%  87.2 2199s
 178614 42708 144126.808   36  832 153102.297 139932.050  8.60%  87.2 2207s
 179087 42772 144378.133   39  970 153102.297 139946.574  8.59%  87.2 2212s
 179472 42853     cutoff   39      153102.297 139958.381  8.59%  87.2 2219s
 180000 43032 146817.140   40 1059 153102.297 139976.974  8.57%  87.2 2226s
 180995 43089     cutoff   52      153102.297 140017.214  8.55%  87.1 2232s
 181320 43113 146139.285   34 1707 153102.297 140018.481  8.55%  87.1 2236s
 181501 43250 146248.254   36 1590 153102.297 140019.359  8.55%  87.1 2241s
 182235 43300     cutoff   49      153102.297 140047.778  8.53%  87.1 2246s
 182507 43356     cutoff   50      153102.297 140055.315  8.52%  87.1 2251s
 182908 43478 147986.336   36 1165 153102.297 140068.443  8.51%  87.1 2256s
 183578 43520 148599.611   40  919 153102.297 140091.596  8.50%  87.1 2262s
 183828 43669 147212.854   37 1661 153102.297 140103.976  8.49%  87.1 2267s
 184599 43696 152577.465   45  598 153102.297 140122.331  8.48%  87.0 2273s
 184846 43823 infeasible   37      153102.297 140131.126  8.47%  87.1 2279s
 185427 43853 145920.597   36  959 153102.297 140147.930  8.46%  87.0 2285s
 186342 44025     cutoff   39      153102.297 140168.812  8.45%  87.0 2295s
 186982 44203 infeasible   45      153102.297 140185.994  8.44%  87.0 2304s
 187573 44283     cutoff   40      153102.297 140209.347  8.42%  87.0 2307s
 187930 44344 141381.328   41 1084 153102.297 140216.684  8.42%  87.0 2312s
 188254 44426 151505.463   44  580 153102.297 140224.718  8.41%  87.0 2316s
 188892 44467 147821.844   43  876 153102.297 140249.741  8.39%  87.0 2321s
 189303 44536 141971.775   38  990 153102.297 140264.394  8.39%  87.0 2328s
 189602 44665 143616.413   42 1229 153102.297 140268.091  8.38%  87.0 2335s
 190830 44744     cutoff   43      153102.297 140298.865  8.36%  87.0 2344s
 191159 44845 143431.094   42  998 153102.297 140308.487  8.36%  87.0 2349s
 191838 44904 147364.508   43  943 153102.297 140329.236  8.34%  87.0 2353s
 192173 44954 147237.649   44 1178 153102.297 140337.635  8.34%  87.0 2357s
 192553 44987 147605.397   50 1279 153102.297 140345.289  8.33%  87.0 2361s
 192869 45084 143008.539   35 1201 153102.297 140352.567  8.33%  87.0 2366s
 193544 45112 149797.623   44 1272 153102.297 140363.813  8.32%  87.0 2372s
 193768 45204 142041.597   37 1021 153102.297 140370.146  8.32%  87.0 2376s
 194431 45244     cutoff   41      153102.297 140385.331  8.31%  87.0 2380s
 194986 45332     cutoff   44      153102.297 140402.618  8.29%  87.0 2389s
 195501 45382 infeasible   37      153102.297 140419.760  8.28%  87.0 2394s
 195683 45433 146931.060   41  842 153102.297 140419.760  8.28%  87.0 2399s
 195962 45484 147806.877   45 1361 153102.297 140422.820  8.28%  87.0 2405s
 196743 45612     cutoff   40      153102.297 140450.200  8.26%  87.0 2414s
 197098 45665 143809.300   36 1094 153102.297 140457.617  8.26%  87.0 2418s
 197642 45707 142173.206   39 1595 153102.297 140473.745  8.25%  87.0 2423s
 197842 45826 144785.091   40 1603 153102.297 140473.745  8.25%  87.0 2428s
 198544 45851 148446.061   42 1267 153102.297 140495.128  8.23%  87.1 2433s
 198888 45906 140960.341   33 1708 153102.297 140513.701  8.22%  87.1 2438s
 199334 45936 144223.115   36  558 153102.297 140525.567  8.21%  87.1 2442s
 199752 46019 141838.791   41  878 153102.297 140532.976  8.21%  87.0 2449s
 200318 46068 143813.184   42  986 153102.297 140546.828  8.20%  87.0 2453s
 200576 46204 151009.971   41  602 153102.297 140556.388  8.19%  87.1 2458s
 201423 46235 147897.108   40 1179 153102.297 140574.849  8.18%  87.0 2462s
 201733 46268 143289.124   42 1144 153102.297 140584.383  8.18%  87.0 2468s
 202068 46355 146471.219   38 1413 153102.297 140591.903  8.17%  87.0 2473s
 202604 46372 infeasible   43      153102.297 140605.966  8.16%  87.0 2477s
 202813 46415 146147.274   39 1066 153102.297 140611.716  8.16%  87.0 2482s
 203264 46451 infeasible   35      153102.297 140624.591  8.15%  87.0 2487s
 203510 46545 141616.040   35 1292 153102.297 140627.473  8.15%  87.0 2491s
 204191 46581 141832.797   40  766 153102.297 140640.871  8.14%  87.0 2497s
 204415 46714 144693.811   43  889 153102.297 140644.577  8.14%  87.0 2502s
 205161 46733 152787.382   51  392 153102.297 140661.244  8.13%  87.0 2508s
 205367 46818 infeasible   40      153102.297 140664.796  8.12%  87.0 2512s
 205919 46861 152080.595   39  941 153102.297 140673.623  8.12%  87.0 2518s
 206296 46886 142508.995   34 1552 153102.297 140683.539  8.11%  87.0 2523s
 206408 46929 149739.285   46  922 153102.297 140683.936  8.11%  87.0 2528s
 206752 47014 142004.602   41  885 153102.297 140689.566  8.11%  87.0 2532s
 207226 47042 144107.356   37 1362 153102.297 140704.327  8.10%  87.0 2537s
 207491 47194 148252.886   41 1319 153102.297 140710.168  8.09%  87.0 2542s
 208314 47218 153083.643   43  657 153102.297 140724.518  8.08%  86.9 2549s
 208602 47304 146310.243   41 1039 153102.297 140732.133  8.08%  86.9 2554s
 209216 47334 150899.086   47  921 153102.297 140738.114  8.08%  87.0 2559s
 209421 47419 146077.855   39 1027 153102.297 140743.689  8.07%  87.0 2564s
 210161 47429 150245.744   47 1351 153102.297 140766.508  8.06%  86.9 2571s
 210324 47827 147010.694   36 1039 153102.297 140771.319  8.05%  86.9 2591s
 212936 47875 145449.470   44  978 153102.297 140821.133  8.02%  86.9 2597s
 213273 47900 142145.999   39 1131 153102.297 140828.376  8.02%  86.9 2600s
 213698 47954 147274.105   47  693 153102.297 140834.603  8.01%  86.9 2609s
 213889 48015     cutoff   47      153102.297 140841.978  8.01%  86.9 2614s
 214344 48050 151498.520   43  972 153102.297 140850.553  8.00%  86.9 2620s
 215418 48177     cutoff   50      153102.297 140870.347  7.99%  86.8 2632s
 215712 48282 149718.069   41  580 153102.297 140876.680  7.99%  86.8 2637s
 216489 48315 147186.053   43 1017 153102.297 140891.634  7.98%  86.8 2644s
 216838 48400 141111.764   40 1248 153102.297 140904.616  7.97%  86.8 2649s
 217587 48423 146631.089   45 1180 153102.297 140920.410  7.96%  86.7 2654s
 217934 48448 145728.045   50 1223 153102.297 140923.669  7.95%  86.8 2662s
 218138 48523 144926.050   40 1127 153102.297 140937.885  7.95%  86.8 2666s
 218799 48528     cutoff   41      153102.297 140950.937  7.94%  86.8 2671s
 219168 48602 147115.441   38  624 153102.297 140957.882  7.93%  86.8 2678s
 219541 48632 153084.235   40  878 153102.297 140964.739  7.93%  86.7 2683s
 219891 48716 142433.905   40 1080 153102.297 140971.907  7.92%  86.7 2689s
 220592 48744 152201.207   39  686 153102.297 140980.624  7.92%  86.7 2694s
 220736 48744 143708.728   40 1311 153102.297 140980.624  7.92%  86.7 2695s
 220932 48783 142261.132   38 1230 153102.297 140981.999  7.92%  86.7 2702s
 221383 48934     cutoff   37      153102.297 140997.981  7.91%  86.7 2709s
 222468 48982 143387.392   42 1136 153102.297 141022.604  7.89%  86.7 2714s
 222768 49020 146437.483   45  735 153102.297 141022.604  7.89%  86.6 2719s
 223054 49079 142745.150   42 1137 153102.297 141031.903  7.88%  86.6 2724s
 223613 49123 148681.909   49 1000 153102.297 141044.177  7.88%  86.6 2729s
 223964 49201 141247.294   39 1217 153102.297 141048.064  7.87%  86.6 2734s
 224561 49244 151085.576   48  972 153102.297 141063.886  7.86%  86.6 2738s
 224901 49310 148655.172   40  989 153102.297 141068.151  7.86%  86.6 2742s
 225410 49327     cutoff   40      153102.297 141081.271  7.85%  86.6 2747s
 225766 49372 144445.351   42 1373 153102.297 141088.178  7.85%  86.6 2751s
 226147 49451 143092.882   41 1168 153102.297 141100.707  7.84%  86.6 2756s
 226685 49479 151401.811   38 1008 153102.297 141109.456  7.83%  86.6 2764s
 227013 49580 141433.502   37  780 153102.297 141116.684  7.83%  86.6 2768s
 227696 49614 infeasible   43      153102.297 141132.311  7.82%  86.6 2776s
 227980 49669 infeasible   42      153102.297 141138.358  7.81%  86.5 2781s
 228698 49688     cutoff   44      153102.297 141153.333  7.80%  86.5 2785s
 229238 49763 149955.862   43 1200 153102.297 141163.020  7.80%  86.5 2793s
 229592 49851 infeasible   44      153102.297 141170.921  7.79%  86.5 2798s
 230160 49883 151936.828   38  564 153102.297 141180.449  7.79%  86.5 2802s
 230481 49885 148488.077   37 1197 153102.297 141183.151  7.79%  86.5 2808s
 230514 49908 145402.691   39 1127 153102.297 141183.733  7.78%  86.5 2815s
 230689 50043 145426.927   40 1163 153102.297 141183.733  7.78%  86.5 2820s
 231485 50101 152622.532   43  818 153102.297 141206.227  7.77%  86.4 2825s
 231991 50133 146928.515   45 1077 153102.297 141215.674  7.76%  86.4 2830s
 232511 50155 144033.070   39 1057 153102.297 141231.987  7.75%  86.4 2836s
 232715 50186 144616.357   41 1054 153102.297 141232.650  7.75%  86.4 2842s
 233145 50298     cutoff   47      153102.297 141244.983  7.74%  86.4 2847s
 234048 50367 151492.286   43  426 153102.297 141261.838  7.73%  86.3 2851s
 234422 50390     cutoff   41      153102.297 141268.010  7.73%  86.3 2857s
 234688 50459 147119.174   39 1623 153102.297 141269.589  7.73%  86.3 2861s
 235170 50497 141578.998   37 1264 153102.297 141280.350  7.72%  86.3 2865s
 235949 50620 147026.395   41 1066 153102.297 141294.963  7.71%  86.3 2875s
 236244 50724 147419.343   43 1085 153102.297 141296.849  7.71%  86.3 2880s
 236933 50746 145207.996   41  818 153102.297 141315.379  7.70%  86.3 2886s
 237207 50802 142897.253   38 1347 153102.297 141319.246  7.70%  86.3 2891s
 237795 50840 143229.973   31 1668 153102.297 141330.430  7.69%  86.3 2897s
 238095 50921 145061.881   42  811 153102.297 141331.208  7.69%  86.2 2902s
 238754 50953 infeasible   37      153102.297 141343.318  7.68%  86.2 2906s
 238957 51017 144337.506   43  666 153102.297 141348.830  7.68%  86.2 2910s
 239460 51043 151363.559   38 1031 153102.297 141357.166  7.67%  86.2 2916s
 239740 51121 143706.287   44 1097 153102.297 141362.131  7.67%  86.2 2921s
 240410 51150 147102.357   43 1278 153102.297 141373.817  7.66%  86.2 2925s
 241004 51268 142656.735   40  776 153102.297 141384.059  7.65%  86.2 2934s
 241700 51327     cutoff   43      153102.297 141400.723  7.64%  86.1 2941s
 242079 51425     cutoff   44      153102.297 141404.867  7.64%  86.1 2948s
 243131 51449 infeasible   33      153102.297 141424.273  7.63%  86.1 2952s
 243339 51514 148055.496   43 1301 153102.297 141426.115  7.63%  86.1 2956s
 243741 51526 148665.757   43  921 153102.297 141435.281  7.62%  86.1 2960s
 244079 51575     cutoff   41      153102.297 141439.280  7.62%  86.1 2965s
 244451 51628 147679.041   39  838 153102.297 141445.863  7.61%  86.1 2970s
 244838 51691 143647.109   44  882 153102.297 141452.255  7.61%  86.1 2975s
 245575 51688     cutoff   47      153102.297 141465.571  7.60%  86.0 2980s
 245774 51762 149552.500   37 1205 153102.297 141471.237  7.60%  86.0 2985s
H246420 51774                    153102.28294 141481.141  7.59%  86.0 2989s
 246626 51781 150585.292   41 1094 153102.283 141483.937  7.59%  86.0 2993s
 246654 51793     cutoff   47      153102.283 141484.799  7.59%  86.0 2999s
 246814 51812     cutoff   44      153102.283 141486.187  7.59%  86.0 3005s
 247125 51881 149593.278   44 1055 153102.283 141491.623  7.58%  86.0 3010s
 247736 51920     cutoff   44      153102.283 141499.611  7.58%  86.0 3018s
 248148 51971 143867.817   33 1494 153102.283 141508.147  7.57%  86.0 3022s
 248480 52059     cutoff   44      153102.283 141513.645  7.57%  85.9 3027s
 249196 52080 143847.273   43 1189 153102.283 141523.356  7.56%  85.9 3034s
 249447 52188 147708.270   39 1128 153102.283 141525.866  7.56%  85.9 3039s
 250178 52196 149055.963   36  778 153102.283 141545.948  7.55%  85.9 3043s
 250431 52212     cutoff   41      153102.283 141548.732  7.55%  85.9 3048s
 250608 52327     cutoff   42      153102.283 141549.211  7.55%  85.9 3053s
 251274 52378 149908.950   41 1047 153102.283 141564.793  7.54%  85.9 3057s
 251649 52387 142863.358   36  840 153102.283 141569.194  7.53%  85.9 3062s
 252058 52427 149956.989   45 1115 153102.283 141574.627  7.53%  85.9 3067s
 252443 52485 151278.015   49 1261 153102.283 141582.043  7.52%  85.9 3073s
 253033 52532 150685.371   43  885 153102.283 141591.485  7.52%  85.9 3077s
 253458 52596 143195.252   43  911 153102.283 141597.033  7.51%  85.8 3082s
 254015 52657 149270.290   41  621 153102.283 141610.284  7.51%  85.8 3087s
 254450 52675     cutoff   45      153102.283 141615.178  7.50%  85.8 3092s
 254771 52745 147725.855   37 1013 153102.283 141624.141  7.50%  85.8 3097s
 255515 52780 infeasible   40      153102.283 141635.418  7.49%  85.8 3101s
 255827 52788 147720.636   46  775 153102.283 141638.316  7.49%  85.8 3105s
 255993 52847 143727.842   38 1132 153102.283 141641.377  7.49%  85.8 3110s
 256626 52883 146680.969   41  927 153102.283 141654.309  7.48%  85.7 3116s
 256906 52909     cutoff   47      153102.283 141658.516  7.47%  85.7 3121s
 257392 52957 143877.422   41 1167 153102.283 141667.467  7.47%  85.8 3125s
 257786 53040 143455.687   45 1076 153102.283 141673.275  7.46%  85.7 3130s
 258671 53160 152043.281   39 1383 153102.283 141685.595  7.46%  85.7 3139s
 259394 53170 145570.614   44  951 153102.283 141697.197  7.45%  85.7 3143s
 259690 53199 145071.491   37 1273 153102.283 141699.787  7.45%  85.7 3149s
 259870 53297 147265.248   44  898 153102.283 141699.787  7.45%  85.7 3154s
 260613 53326 152433.971   46 1217 153102.283 141710.982  7.44%  85.6 3160s
 261451 53418     cutoff   39      153102.283 141726.888  7.43%  85.6 3169s
 261848 53462     cutoff   39      153102.283 141733.384  7.43%  85.6 3174s
 262165 53530 146113.389   47 1227 153102.283 141739.819  7.42%  85.6 3179s
 262751 53563 142635.745   37 1661 153102.283 141748.314  7.42%  85.6 3185s
 263054 53599 147253.645   41  652 153102.283 141752.013  7.41%  85.6 3193s
 263397 53657 144779.975   42 1080 153102.283 141758.991  7.41%  85.6 3201s
 263858 53842     cutoff   48      153102.283 141764.089  7.41%  85.6 3210s
 265197 53875 143706.574   39  804 153102.283 141784.605  7.39%  85.6 3218s
 265526 53895 143836.434   43 1102 153102.283 141791.327  7.39%  85.6 3222s
 265872 53943     cutoff   52      153102.283 141792.421  7.39%  85.6 3228s
 266266 53991     cutoff   42      153102.283 141803.186  7.38%  85.6 3233s
 266752 54059 149638.074   34 1511 153102.283 141809.786  7.38%  85.6 3239s
 267503 54110     cutoff   57      153102.283 141821.101  7.37%  85.6 3244s
 268008 54122 152871.159   52  922 153102.283 141828.136  7.36%  85.6 3252s
 268318 54282 144550.038   35 1388 153102.283 141834.083  7.36%  85.6 3257s
 269209 54307 149544.824   43  276 153102.283 141848.629  7.35%  85.5 3264s
 269407 54424 144065.656   38 1250 153102.283 141850.260  7.35%  85.5 3269s
 270126 54458     cutoff   51      153102.283 141859.447  7.34%  85.5 3274s
 270449 54495 149952.786   43  525 153102.283 141863.710  7.34%  85.5 3278s
 270784 54536 146024.307   44  935 153102.283 141866.900  7.34%  85.5 3283s
 271333 54563 143044.917   38  926 153102.283 141879.514  7.33%  85.4 3287s
 271596 54626 144684.739   40 1484 153102.283 141881.734  7.33%  85.4 3292s
 272399 54644 142954.997   36 1052 153102.283 141893.759  7.32%  85.4 3296s
 272748 54655 infeasible   43      153102.283 141898.753  7.32%  85.4 3300s
 273115 54699 152454.543   38 1340 153102.283 141902.856  7.31%  85.4 3306s
 273557 54770 151978.797   44  794 153102.283 141910.525  7.31%  85.4 3311s
 274168 54801     cutoff   47      153102.283 141920.861  7.30%  85.4 3316s
 274390 54879 151321.078   43  998 153102.283 141921.998  7.30%  85.4 3321s
 275144 54898 151767.789   41  581 153102.283 141934.680  7.29%  85.3 3325s
 275730 54947 143824.760   43  910 153102.283 141941.760  7.29%  85.3 3333s
 275889 55045     cutoff   47      153102.283 141942.972  7.29%  85.3 3338s
 276661 55067 infeasible   49      153102.283 141953.409  7.28%  85.3 3343s
 276951 55102 150469.064   41  630 153102.283 141957.003  7.28%  85.3 3347s
 277309 55104     cutoff   52      153102.283 141962.986  7.28%  85.3 3352s
 277348 55121 142268.675   36 1462 153102.283 141963.033  7.28%  85.3 3356s
 277580 55162 152741.182   48  828 153102.283 141966.050  7.27%  85.3 3361s
 277855 55199 142693.109   39  747 153102.283 141969.716  7.27%  85.3 3365s
 278154 55248 infeasible   38      153102.283 141973.620  7.27%  85.3 3370s
 279068 55326     cutoff   42      153102.283 141985.211  7.26%  85.3 3379s
 279584 55355 146528.550   42 1241 153102.283 141994.755  7.25%  85.2 3385s
 279869 55484 147472.524   45  874 153102.283 141998.274  7.25%  85.2 3390s
 281000 55566 148179.274   37 1344 153102.283 142012.886  7.24%  85.2 3400s
 281860 55649     cutoff   42      153102.283 142023.949  7.24%  85.2 3411s
 282252 55698 infeasible   48      153102.283 142029.860  7.23%  85.2 3415s
 282892 55722 145813.613   39 1493 153102.283 142039.413  7.23%  85.1 3420s
 283330 55734 151450.980   47  625 153102.283 142042.792  7.22%  85.1 3425s
 283702 55778 152443.895   41 1191 153102.283 142046.656  7.22%  85.1 3431s
 284337 55802 144772.610   36 1051 153102.283 142057.472  7.21%  85.1 3437s
 284590 55882 145603.930   39 1120 153102.283 142062.545  7.21%  85.1 3444s
 285382 55892 149557.882   37  617 153102.283 142074.005  7.20%  85.1 3449s
 285622 55928 148048.577   43 1000 153102.283 142074.005  7.20%  85.1 3453s
 285953 56004 144074.624   42 1245 153102.283 142079.078  7.20%  85.1 3458s
 286598 56025     cutoff   40      153102.283 142090.026  7.19%  85.1 3462s
 286912 56067 143851.319   41 1158 153102.283 142093.913  7.19%  85.1 3466s
 287197 56074 151178.343   40  666 153102.283 142098.392  7.19%  85.1 3470s
 287639 56124 152661.002   47  618 153102.283 142099.152  7.19%  85.1 3479s
 287928 56157     cutoff   38      153102.283 142108.027  7.18%  85.1 3484s
 288210 56177     cutoff   51      153102.283 142110.128  7.18%  85.1 3489s
 288474 56199 infeasible   40      153102.283 142111.209  7.18%  85.1 3494s
 289076 56213     cutoff   42      153102.283 142125.073  7.17%  85.0 3499s
 289494 56246 148728.787   39  858 153102.283 142129.533  7.17%  85.0 3503s
 289961 56280 146370.917   45 1242 153102.283 142134.316  7.16%  85.0 3508s
 290439 56285 147617.037   42 1345 153102.283 142144.782  7.16%  85.0 3513s
 290668 56402 148056.244   43 1365 153102.283 142146.280  7.16%  85.0 3518s
 291500 56418 142436.884   41 1135 153102.283 142156.010  7.15%  85.0 3526s
 291750 56527 149151.271   44 1150 153102.283 142160.926  7.15%  85.0 3531s
 292546 56554 149185.703   41  929 153102.283 142171.384  7.14%  84.9 3537s
 292861 56577     cutoff   44      153102.283 142175.924  7.14%  85.0 3543s
 293184 56672 147017.444   48 1113 153102.283 142181.761  7.13%  85.0 3548s
 293891 56685 145509.056   42  644 153102.283 142189.038  7.13%  85.0 3553s
 294091 56737 147795.511   48  986 153102.283 142192.599  7.13%  84.9 3563s
 294750 56748     cutoff   45      153102.283 142201.164  7.12%  84.9 3572s
 295185 56803     cutoff   45      153102.283 142205.814  7.12%  84.9 3580s
 295785 56837     cutoff   51      153102.283 142215.457  7.11%  84.9 3588s
 296476 56922 infeasible   44      153102.283 142222.859  7.11%  84.9 3595s
 297398 56951 143390.937   45  867 153102.283 142231.474  7.10%  84.9 3604s
 297705 57056 145106.744   43 1089 153102.283 142234.857  7.10%  84.9 3610s
 298641 57092 147555.278   39  917 153102.283 142246.784  7.09%  84.9 3615s
 299031 57132 145836.537   48 1225 153102.283 142249.501  7.09%  84.8 3620s
 299479 57173     cutoff   45      153102.283 142256.401  7.08%  84.8 3625s
 300028 57206     cutoff   43      153102.283 142267.025  7.08%  84.8 3630s
 300517 57280 150973.274   45  660 153102.283 142273.687  7.07%  84.8 3635s
 301130 57282 150669.648   40 1317 153102.283 142280.822  7.07%  84.8 3640s
 301581 57319 150896.762   43  886 153102.283 142287.453  7.06%  84.7 3645s
 302338 57390 145452.759   37 1538 153102.283 142298.683  7.06%  84.7 3654s
 302973 57428 143874.017   43  835 153102.283 142305.441  7.05%  84.7 3658s
 303437 57457 148174.250   38 1352 153102.283 142310.525  7.05%  84.7 3663s
 303833 57483 146492.679   44  868 153102.283 142313.842  7.05%  84.7 3670s
 304230 57542 infeasible   45      153102.283 142317.745  7.04%  84.7 3677s
 304939 57572 151810.414   41  597 153102.283 142327.933  7.04%  84.7 3682s
 305144 57593     cutoff   48      153102.283 142329.183  7.04%  84.7 3686s
 305409 57651 145841.721   39 1463 153102.283 142331.142  7.04%  84.7 3691s
 305917 57669 147586.376   44 1356 153102.283 142338.462  7.03%  84.6 3695s
 306292 57689 144780.002   40  836 153102.283 142344.264  7.03%  84.6 3700s
 306687 57726 145662.937   47 1274 153102.283 142350.613  7.02%  84.6 3705s
 307584 57773 145603.047   38 1397 153102.283 142361.222  7.02%  84.6 3714s
 307854 57796 infeasible   47      153102.283 142365.159  7.01%  84.6 3718s
 308260 57857 145808.417   45  859 153102.283 142373.756  7.01%  84.6 3724s
 308823 57878 146570.408   46 1032 153102.283 142381.699  7.00%  84.6 3729s
 309147 57897 145056.900   34 1636 153102.283 142383.838  7.00%  84.6 3734s
 309239 57897 145782.070   43 1097 153102.283 142383.838  7.00%  84.6 3735s
 309459 57937 148873.604   40  762 153102.283 142390.761  7.00%  84.6 3740s
 309882 57979 149647.489   37  890 153102.283 142395.552  6.99%  84.5 3748s
 310346 57989 148222.361   42 1058 153102.283 142400.765  6.99%  84.5 3754s
 310663 58024 145286.484   43  997 153102.283 142405.476  6.99%  84.5 3759s
 311190 58036 152185.228   43  772 153102.283 142412.702  6.98%  84.5 3768s
 311549 58073 146952.237   43 1434 153102.283 142416.041  6.98%  84.5 3774s
 312239 58098 148228.041   39 1315 153102.283 142427.732  6.97%  84.5 3780s
 312468 58166 143218.119   38  805 153102.283 142429.270  6.97%  84.5 3785s
 312887 58195 146908.007   42 1287 153102.283 142438.812  6.96%  84.5 3790s
 313348 58257 151527.092   46  943 153102.283 142444.917  6.96%  84.5 3795s
 313736 58343     cutoff   49      153102.283 142448.208  6.96%  84.5 3801s
 314411 58375 148480.517   43 1296 153102.283 142454.764  6.95%  84.4 3807s
 314682 58389     cutoff   45      153102.283 142459.106  6.95%  84.4 3813s
 315262 58411     cutoff   48      153102.283 142468.739  6.95%  84.4 3816s
 315511 58440 146062.079   41  983 153102.283 142473.490  6.94%  84.4 3821s
 316291 58461 144706.049   43  859 153102.283 142484.079  6.94%  84.4 3825s
 316818 58495 144516.934   35 1435 153102.283 142490.082  6.93%  84.4 3834s
 316949 58522     cutoff   52      153102.283 142490.082  6.93%  84.4 3837s
 317205 58575 145672.036   42 1156 153102.283 142495.524  6.93%  84.4 3842s
 317771 58607 142735.747   38 1640 153102.283 142504.872  6.92%  84.4 3847s
 318189 58639     cutoff   45      153102.283 142508.695  6.92%  84.4 3851s
 318410 58715 149082.036   36 1563 153102.283 142510.014  6.92%  84.4 3856s
 319202 58737     cutoff   38      153102.283 142515.226  6.92%  84.4 3861s
 319562 58748 142565.351   37  864 153102.283 142520.251  6.91%  84.4 3865s
 320227 58807 144354.681   42 1187 153102.283 142529.742  6.91%  84.3 3873s
 320686 58849     cutoff   43      153102.283 142536.556  6.90%  84.3 3877s
 321131 58897 150274.639   46  663 153102.283 142540.804  6.90%  84.3 3881s
 321584 58924     cutoff   43      153102.283 142545.994  6.89%  84.3 3885s
 322317 58981     cutoff   48      153102.283 142554.561  6.89%  84.3 3894s
 322673 59043     cutoff   38      153102.283 142557.492  6.89%  84.3 3899s
 323378 59066 infeasible   42      153102.283 142569.205  6.88%  84.3 3905s
 324097 59156 147323.788   36  738 153102.283 142578.182  6.87%  84.3 3914s
 324594 59183 infeasible   45      153102.283 142585.751  6.87%  84.3 3918s
 324929 59242     cutoff   49      153102.283 142588.187  6.87%  84.3 3923s
 325681 59255     cutoff   43      153102.283 142601.053  6.86%  84.2 3928s
 325888 59306 143192.321   42 1056 153102.283 142602.184  6.86%  84.2 3933s
 326583 59339 149183.840   39 1107 153102.283 142611.156  6.85%  84.2 3937s
 326921 59340 145036.571   43  827 153102.283 142618.608  6.85%  84.2 3940s
 327665 59389 148938.336   42 1066 153102.283 142627.550  6.84%  84.1 3947s
 327943 59407     cutoff   36      153102.283 142630.498  6.84%  84.1 3951s
 328369 59425 144888.266   41 1352 153102.283 142637.317  6.84%  84.1 3955s
 328580 59506 infeasible   45      153102.283 142640.240  6.83%  84.1 3960s
 329505 59531     cutoff   44      153102.283 142650.184  6.83%  84.1 3970s
 330025 59563 144473.927   40  883 153102.283 142659.300  6.82%  84.1 3977s
 330207 59565 152651.527   42  948 153102.283 142661.790  6.82%  84.1 3982s
 330514 59591 143059.588   34 1870 153102.283 142663.995  6.82%  84.1 3986s
 330844 59632 147881.239   47 1234 153102.283 142668.167  6.82%  84.1 3991s
 331556 59647     cutoff   45      153102.283 142675.450  6.81%  84.0 3995s
 331860 59681 143105.724   40 1410 153102.283 142679.136  6.81%  84.0 4000s
 332146 59710 152850.045   42  614 153102.283 142682.314  6.81%  84.0 4005s
 332925 59759 144498.904   40 1391 153102.283 142693.507  6.80%  84.0 4013s
 333278 59842 147564.510   43  555 153102.283 142697.402  6.80%  84.0 4017s
 333928 59851     cutoff   44      153102.283 142705.585  6.79%  84.0 4022s
 334276 59867     cutoff   53      153102.283 142708.402  6.79%  84.0 4026s
 334624 59911 145519.557   40  609 153102.283 142714.185  6.79%  84.0 4030s
 335171 59936 143264.811   37 1386 153102.283 142720.582  6.78%  84.0 4037s
 335507 59992 144817.890   38 1068 153102.283 142721.699  6.78%  84.0 4046s
 336386 60000 147273.839   35 1312 153102.283 142736.843  6.77%  84.0 4052s
 336903 60003 150153.749   43  850 153102.283 142743.221  6.77%  84.0 4058s
 337301 60024 145304.760   41 1290 153102.283 142747.883  6.76%  84.0 4063s
 337836 60047 149280.321   40 1047 153102.283 142752.304  6.76%  84.0 4068s
 338273 60082 147985.413   44 1207 153102.283 142759.358  6.76%  84.0 4073s
 338858 60131 144464.019   45 1163 153102.283 142766.116  6.75%  83.9 4079s
 339149 60158 144965.307   44 1164 153102.283 142770.637  6.75%  83.9 4084s
 339792 60210     cutoff   42      153102.283 142778.506  6.74%  83.9 4089s
 340300 60231 151305.005   43  477 153102.283 142786.212  6.74%  83.9 4093s
 340701 60249 144232.692   41 1003 153102.283 142792.643  6.73%  83.9 4097s
 341125 60260 145217.477   47 1225 153102.283 142795.586  6.73%  83.9 4103s
 341409 60323 143021.901   40 1338 153102.283 142797.318  6.73%  83.9 4108s
 342180 60343 144763.854   40 1123 153102.283 142810.606  6.72%  83.9 4112s
 342468 60376 151653.619   39  564 153102.283 142811.418  6.72%  83.9 4117s
 342881 60431 147719.099   45 1316 153102.283 142815.682  6.72%  83.9 4122s
 343449 60442 144027.228   37 1280 153102.283 142822.703  6.71%  83.9 4126s
 343639 60474 infeasible   42      153102.283 142825.238  6.71%  83.9 4131s
 344017 60481 151848.462   40  898 153102.283 142828.533  6.71%  83.9 4135s
 344316 60511 148379.527   47  707 153102.283 142829.879  6.71%  83.9 4140s
 344959 60521     cutoff   41      153102.283 142843.244  6.70%  83.9 4145s
 345284 60539 145710.508   45  899 153102.283 142847.097  6.70%  83.9 4150s
 346070 60673 146486.321   46 1180 153102.283 142858.048  6.69%  83.8 4166s
 347933 60698     cutoff   45      153102.283 142876.571  6.68%  83.8 4174s
 348177 60702 144332.827   41  914 153102.283 142881.076  6.68%  83.8 4178s
 348181 60706 150465.690   42  917 153102.283 142881.076  6.68%  83.8 4183s
 348185 60715 152506.770   43  916 153102.283 142881.076  6.68%  83.8 4188s
 348242 60715 infeasible   44      153102.283 142881.076  6.68%  83.8 4192s
 348315 60742 144922.224   37 1507 153102.283 142881.406  6.68%  83.8 4197s
 348578 60756 144836.963   47 1037 153102.283 142883.227  6.67%  83.8 4201s
 348885 60761 150611.071   37  371 153102.283 142885.781  6.67%  83.8 4206s
 349231 60774     cutoff   50      153102.283 142889.030  6.67%  83.8 4210s
 349579 60782 146058.500   38 1438 153102.283 142895.442  6.67%  83.8 4217s
 349920 60798 infeasible   42      153102.283 142900.849  6.66%  83.8 4221s
 350124 60813 144795.217   39 1191 153102.283 142900.849  6.66%  83.8 4227s
 350330 60844 148271.033   43 1134 153102.283 142901.921  6.66%  83.8 4237s
 350867 60884 144267.255   34 1471 153102.283 142911.716  6.66%  83.8 4242s
 351275 60935 147983.829   36 1417 153102.283 142917.221  6.65%  83.7 4249s
 352074 60972     cutoff   46      153102.283 142931.016  6.64%  83.7 4256s
 352480 61030 145713.402   35 1336 153102.283 142933.331  6.64%  83.7 4261s
 353226 61053 149803.505   39 1021 153102.283 142942.496  6.64%  83.7 4270s
 353696 61097     cutoff   47      153102.283 142949.362  6.63%  83.7 4275s
 354294 61119     cutoff   41      153102.283 142960.172  6.62%  83.7 4281s
 354625 61164 150519.294   39 1252 153102.283 142962.224  6.62%  83.7 4287s
 355490 61193 144692.971   42 1063 153102.283 142971.846  6.62%  83.7 4294s
 355770 61221 144903.410   48 1449 153102.283 142974.896  6.61%  83.7 4299s
 356257 61245 infeasible   35      153102.283 142979.858  6.61%  83.7 4305s
 356786 61267 147373.646   35 1423 153102.283 142985.456  6.61%  83.7 4310s
 357078 61378 147524.880   40 1046 153102.283 142990.754  6.60%  83.7 4316s
 358008 61384     cutoff   40      153102.283 143000.296  6.60%  83.6 4322s
 358353 61394 150693.939   45  780 153102.283 143004.971  6.60%  83.6 4325s
 358841 61442 152663.646   41 1334 153102.283 143009.760  6.59%  83.6 4333s
 359194 61449 152149.628   47  785 153102.283 143012.111  6.59%  83.6 4338s
 359608 61493 144250.945   39 1276 153102.283 143016.286  6.59%  83.6 4343s
 360100 61501 147776.540   42  837 153102.283 143024.473  6.58%  83.6 4347s
 360523 61539 143097.848   35 1683 153102.283 143032.748  6.58%  83.6 4351s
 360974 61578     cutoff   41      153102.283 143037.169  6.57%  83.6 4356s
 361332 61599 infeasible   49      153102.283 143041.984  6.57%  83.6 4362s
 361805 61620 146192.308   47 1165 153102.283 143046.598  6.57%  83.6 4366s
 362274 61620     cutoff   47      153102.283 143051.410  6.56%  83.6 4371s
 362469 61676 144195.536   39 1161 153102.283 143053.222  6.56%  83.6 4376s
 363279 61697 152843.491   36  638 153102.283 143063.763  6.56%  83.6 4380s
 363704 61703 144605.509   37 1017 153102.283 143066.804  6.55%  83.6 4386s
 363994 61737 148260.448   44 1435 153102.283 143068.741  6.55%  83.6 4391s
 364707 61747 145659.719   34 1045 153102.283 143075.098  6.55%  83.6 4395s
 364967 61747 148119.224   45  657 153102.283 143078.719  6.55%  83.6 4401s
 365230 61803 148256.852   39  550 153102.283 143079.201  6.55%  83.6 4406s
 365829 61816 145179.444   44 1183 153102.283 143089.727  6.54%  83.6 4410s
 366492 61901 148024.845   41 1288 153102.283 143096.878  6.54%  83.6 4419s
 367111 61909     cutoff   39      153102.283 143104.617  6.53%  83.6 4422s
 367287 61929 151093.488   39 1209 153102.283 143104.815  6.53%  83.6 4426s
 367625 61944 143335.454   42  983 153102.283 143109.996  6.53%  83.6 4431s
 368160 61960 148164.022   41  721 153102.283 143115.565  6.52%  83.5 4436s
 368574 61970     cutoff   45      153102.283 143118.016  6.52%  83.5 4440s
 368862 61984 144647.784   42 1085 153102.283 143122.291  6.52%  83.5 4445s
 369829 62025 147167.008   45 1074 153102.283 143135.029  6.51%  83.5 4453s
 370005 62098 146981.641   45 1392 153102.283 143136.164  6.51%  83.5 4458s
 370672 62087 151737.272   41  908 153102.283 143145.141  6.50%  83.5 4462s
 371057 62106 infeasible   41      153102.283 143151.807  6.50%  83.5 4470s
 371435 62173 143407.300   41 1068 153102.283 143152.390  6.50%  83.5 4475s
 372292 62218 143613.895   44  950 153102.283 143162.423  6.49%  83.5 4484s
 372911 62240 infeasible   35      153102.283 143169.400  6.49%  83.5 4490s
 373174 62274 149059.750   44  935 153102.283 143171.537  6.49%  83.5 4495s
 373572 62320 151702.206   41 1084 153102.283 143175.149  6.48%  83.5 4500s
 374334 62365 145955.905   44  794 153102.283 143182.920  6.48%  83.4 4509s
 375026 62382 147392.615   36 1262 153102.283 143195.532  6.47%  83.4 4513s
 375315 62391 infeasible   50      153102.283 143196.451  6.47%  83.5 4518s
 375644 62411     cutoff   40      153102.283 143201.045  6.47%  83.4 4522s
 375924 62449 145615.215   42  712 153102.283 143203.694  6.47%  83.4 4527s
 376607 62458     cutoff   48      153102.283 143209.095  6.46%  83.5 4532s
 376857 62539 144501.252   38  866 153102.283 143210.244  6.46%  83.5 4537s
 377393 62545 149983.156   51  358 153102.283 143217.691  6.46%  83.4 4543s
 377732 62600 146045.477   47 1076 153102.283 143220.614  6.45%  83.4 4549s
 378495 62613 146350.089   38 1038 153102.283 143228.447  6.45%  83.4 4552s
 378800 62635 146541.210   31 1452 153102.283 143231.163  6.45%  83.4 4558s
 379073 62671     cutoff   43      153102.283 143232.857  6.45%  83.4 4562s
 379615 62679 148196.697   49 1257 153102.283 143238.950  6.44%  83.4 4567s
 379979 62688     cutoff   39      153102.283 143243.965  6.44%  83.4 4571s
 380402 62703 146396.404   44 1131 153102.283 143249.773  6.44%  83.4 4576s
 380805 62720 152691.152   43 1368 153102.283 143253.392  6.43%  83.4 4580s
 381620 62729 infeasible   32      153102.283 143264.362  6.43%  83.4 4589s
 381980 62746 145499.534   48 1162 153102.283 143268.116  6.42%  83.4 4594s
 382599 62762 150321.603   43 1009 153102.283 143274.206  6.42%  83.4 4600s
 382892 62757 145312.571   45 1068 153102.283 143277.742  6.42%  83.4 4607s
 383257 62761 149617.779   44 1146 153102.283 143283.364  6.41%  83.3 4613s
 383852 62780 146524.173   45 1264 153102.283 143289.715  6.41%  83.3 4619s
 384157 62835 151672.569   44  990 153102.283 143292.368  6.41%  83.3 4624s
 384834 62852     cutoff   45      153102.283 143299.061  6.40%  83.3 4631s
 385237 62893 145522.300   42 1145 153102.283 143303.758  6.40%  83.3 4635s
 385540 62910 151429.272   43 1166 153102.283 143304.402  6.40%  83.3 4640s
 386064 62926     cutoff   41      153102.283 143312.302  6.39%  83.3 4645s
 386480 62956 152153.480   46  596 153102.283 143316.301  6.39%  83.3 4650s
 386929 62974 152780.685   45  840 153102.283 143319.309  6.39%  83.3 4656s
 387376 62988 145766.181   55 1079 153102.283 143327.832  6.38%  83.3 4662s
 388219 62985     cutoff   43      153102.283 143338.556  6.38%  83.3 4666s
 388428 62989 147448.532   45  853 153102.283 143338.998  6.38%  83.3 4670s
 389147 63036     cutoff   40      153102.283 143349.000  6.37%  83.3 4677s
 389501 63069 146235.627   38 1361 153102.283 143354.572  6.37%  83.3 4681s
 389811 63082     cutoff   45      153102.283 143356.830  6.37%  83.3 4686s
 390047 63093 148123.679   38  959 153102.283 143358.124  6.36%  83.3 4690s
 390386 63093     cutoff   41      153102.283 143364.233  6.36%  83.2 4695s
 390712 63104 145934.929   44 1007 153102.283 143366.422  6.36%  83.2 4704s
 390994 63133 147757.098   45 1020 153102.283 143368.918  6.36%  83.2 4708s
 391293 63137 146964.053   38 1493 153102.283 143368.937  6.36%  83.2 4716s
 391554 63171 146514.710   41  815 153102.283 143375.373  6.35%  83.2 4721s
 392230 63184 144841.496   40 1271 153102.283 143382.522  6.35%  83.2 4726s
 392556 63205 143449.163   40 1698 153102.283 143385.163  6.35%  83.2 4731s
 393138 63211 150682.498   42  692 153102.283 143393.443  6.34%  83.2 4735s
 393358 63215 144218.716   32 1250 153102.283 143395.634  6.34%  83.2 4741s
 394216 63218     cutoff   44      153102.283 143406.720  6.33%  83.2 4746s
 394454 63215 149023.060   41  938 153102.283 143408.503  6.33%  83.2 4751s
 394922 63235 148174.205   46  974 153102.283 143417.525  6.33%  83.2 4756s
 395457 63252 infeasible   34      153102.283 143422.598  6.32%  83.1 4760s
 395885 63267 151578.166   44  803 153102.283 143427.837  6.32%  83.2 4766s
 396248 63289     cutoff   46      153102.283 143431.574  6.32%  83.2 4771s
 396948 63311 150876.297   42 1285 153102.283 143437.962  6.31%  83.1 4777s
 397276 63384     cutoff   45      153102.283 143441.308  6.31%  83.1 4782s
 398007 63384 infeasible   43      153102.283 143446.913  6.31%  83.1 4787s
 398403 63386 infeasible   47      153102.283 143449.993  6.30%  83.1 4793s
 398710 63411 143591.344   42 1006 153102.283 143453.496  6.30%  83.1 4800s
 399858 63444     cutoff   50      153102.283 143468.258  6.29%  83.1 4812s
 400099 63475 144611.685   39 1226 153102.283 143470.806  6.29%  83.1 4818s
 401010 63478 152256.948   39 1293 153102.283 143481.170  6.28%  83.1 4822s
 401324 63505 151121.828   43 1095 153102.283 143481.235  6.28%  83.1 4829s
 401755 63546 147800.286   43  940 153102.283 143489.321  6.28%  83.1 4833s
 402345 63544     cutoff   48      153102.283 143496.107  6.27%  83.1 4837s
 402584 63560 145797.259   41 1214 153102.283 143497.846  6.27%  83.1 4842s
 403280 63568     cutoff   47      153102.283 143505.605  6.27%  83.0 4847s
 403524 63577 146752.497   41  841 153102.283 143506.240  6.27%  83.0 4851s
 403886 63589 150823.171   45 1208 153102.283 143508.009  6.27%  83.0 4856s
 404279 63619 147468.672   34 1551 153102.283 143514.526  6.26%  83.0 4861s
 404903 63639 148878.232   45 1103 153102.283 143521.594  6.26%  83.0 4865s
 405674 63665 144030.510   40  796 153102.283 143529.285  6.25%  83.0 4874s
 406130 63692 143910.418   38 1347 153102.283 143533.603  6.25%  83.0 4879s
 406368 63753 145500.899   44  645 153102.283 143535.540  6.25%  83.0 4884s
 407096 63760     cutoff   47      153102.283 143542.684  6.24%  83.0 4889s
 407359 63786 143968.613   42 1293 153102.283 143545.681  6.24%  83.0 4894s
 408003 63785 145200.643   31 1457 153102.283 143551.617  6.24%  83.0 4898s
 408287 63804 148833.901   35 1557 153102.283 143554.006  6.24%  83.0 4904s
 408761 63796 infeasible   53      153102.283 143559.215  6.23%  83.0 4908s
 409185 63817 143843.667   44 1128 153102.283 143567.716  6.23%  83.0 4914s
 409477 63831     cutoff   52      153102.283 143569.328  6.23%  82.9 4919s
 409895 63875 148025.104   44 1216 153102.283 143574.769  6.22%  82.9 4924s
 410492 63890 146682.373   48 1176 153102.283 143582.262  6.22%  82.9 4929s
 410843 63903     cutoff   48      153102.283 143584.602  6.22%  82.9 4934s
 411560 63925 144461.859   41  998 153102.283 143591.574  6.21%  82.9 4939s
 411835 63941     cutoff   45      153102.283 143595.157  6.21%  82.9 4944s
 412457 63943 152476.615   42  616 153102.283 143601.342  6.21%  82.9 4948s
 412934 63945 147136.514   43  907 153102.283 143606.081  6.20%  82.9 4954s
 413220 63984     cutoff   52      153102.283 143607.881  6.20%  82.9 4959s
 413698 63998 148024.863   43 1140 153102.283 143611.692  6.20%  82.9 4963s
 413987 64003 infeasible   38      153102.283 143615.006  6.20%  82.9 4968s
 414683 64015 147649.477   44 1393 153102.283 143621.431  6.19%  82.9 4973s
 414949 64091 148195.919   46 1174 153102.283 143622.376  6.19%  82.9 4978s
 415729 64087     cutoff   39      153102.283 143634.749  6.18%  82.9 4982s
 415928 64101 147929.816   46 1013 153102.283 143636.965  6.18%  82.9 4986s
 416376 64131     cutoff   42      153102.283 143643.554  6.18%  82.9 4993s
 416661 64204 145173.521   34  783 153102.283 143645.317  6.18%  82.9 4998s
 417410 64211     cutoff   40      153102.283 143652.758  6.17%  82.8 5002s
 417737 64202 146770.834   41  861 153102.283 143658.517  6.17%  82.8 5006s
 418152 64222 infeasible   36      153102.283 143661.553  6.17%  82.8 5011s
 418441 64267 143932.541   43 1290 153102.283 143664.426  6.16%  82.8 5016s
 419135 64297     cutoff   40      153102.283 143671.717  6.16%  82.8 5023s
 419506 64321 145661.221   45 1202 153102.283 143675.118  6.16%  82.8 5029s
 420185 64330 144452.971   35 1508 153102.283 143681.019  6.15%  82.8 5034s
 420656 64355 145676.961   45  967 153102.283 143685.273  6.15%  82.8 5039s
 421158 64367     cutoff   42      153102.283 143693.733  6.15%  82.8 5043s
 421432 64376 146067.386   53 1199 153102.283 143695.727  6.14%  82.8 5049s
 422260 64409 149785.795   44 1391 153102.283 143705.041  6.14%  82.8 5056s
 422429 64436 146474.390   34 1445 153102.283 143705.953  6.14%  82.8 5061s
 423217 64438     cutoff   43      153102.283 143712.069  6.13%  82.8 5066s
 423446 64470 148174.839   48 1216 153102.283 143713.527  6.13%  82.8 5070s
 424043 64467 infeasible   44      153102.283 143723.118  6.13%  82.8 5075s
 424712 64471     cutoff   43      153102.283 143730.259  6.12%  82.7 5084s
 425339 64459     cutoff   43      153102.283 143736.990  6.12%  82.7 5088s
 425619 64496 144939.149   51 1189 153102.283 143740.345  6.11%  82.7 5093s
 426164 64503     cutoff   52      153102.283 143746.824  6.11%  82.7 5099s
 426473 64528     cutoff   48      153102.283 143747.150  6.11%  82.7 5104s
 427286 64528 146496.663   40  840 153102.283 143758.666  6.10%  82.7 5110s
 427600 64546 145561.463   41 1176 153102.283 143760.173  6.10%  82.7 5115s
 428265 64542 infeasible   48      153102.283 143765.448  6.10%  82.7 5122s
 428639 64572 145052.032   43  678 153102.283 143771.259  6.09%  82.7 5126s
 429063 64581     cutoff   35      153102.283 143776.384  6.09%  82.7 5131s
 429499 64580 152310.439   45  797 153102.283 143779.383  6.09%  82.6 5136s
 429988 64591 147846.329   48 1168 153102.283 143783.782  6.09%  82.7 5140s
 430864 64597 infeasible   39      153102.283 143793.983  6.08%  82.6 5151s
 431154 64606 infeasible   49      153102.283 143796.459  6.08%  82.6 5156s
 431507 64615     cutoff   52      153102.283 143801.436  6.07%  82.6 5163s
 431839 64655     cutoff   45      153102.283 143803.369  6.07%  82.6 5169s
 432694 64679 146139.848   39  960 153102.283 143810.523  6.07%  82.6 5174s
 432923 64699 147596.823   42  791 153102.283 143810.523  6.07%  82.6 5179s
 433251 64759 144338.288   41  723 153102.283 143816.829  6.06%  82.6 5184s
 434063 64742     cutoff   43      153102.283 143824.439  6.06%  82.6 5188s
 434515 64763 147376.112   36 1262 153102.283 143827.805  6.06%  82.6 5194s
 434811 64770     cutoff   47      153102.283 143829.412  6.06%  82.6 5199s
 435213 64786     cutoff   41      153102.283 143836.179  6.05%  82.6 5203s
 435558 64809 151971.175   41  512 153102.283 143839.998  6.05%  82.6 5207s
 435975 64837 147348.731   46  634 153102.283 143843.500  6.05%  82.6 5212s
 436455 64831     cutoff   44      153102.283 143847.959  6.04%  82.6 5216s
 436779 64840 147999.848   44  827 153102.283 143849.228  6.04%  82.6 5221s
 437201 64854 151699.021   39  585 153102.283 143854.590  6.04%  82.6 5225s
 438200 64911     cutoff   40      153102.283 143863.934  6.03%  82.6 5235s
 439063 64934 144202.523   41 1364 153102.283 143872.398  6.03%  82.6 5243s
 439333 64964     cutoff   49      153102.283 143875.724  6.03%  82.6 5248s
 440123 64995 146709.331   44  875 153102.283 143883.870  6.02%  82.6 5253s
 440403 64991 147722.986   45  809 153102.283 143883.870  6.02%  82.6 5261s
 440908 65057 149335.295   42 1468 153102.283 143888.442  6.02%  82.6 5269s
 442195 65059     cutoff   44      153102.283 143900.794  6.01%  82.5 5274s
 442386 65065 149801.910   35 1675 153102.283 143905.832  6.01%  82.5 5280s
 442967 65099     cutoff   38      153102.283 143910.398  6.00%  82.5 5290s
 443843 65110     cutoff   37      153102.283 143917.112  6.00%  82.5 5299s
 443961 65114     cutoff   43      153102.283 143918.996  6.00%  82.5 5305s
 444247 65173     cutoff   48      153102.283 143921.295  6.00%  82.5 5310s
 445376 65203 148498.756   46  549 153102.283 143929.205  5.99%  82.5 5319s
 445655 65211 149792.518   41  718 153102.283 143930.462  5.99%  82.5 5323s
 446117 65208 146629.272   44  830 153102.283 143937.857  5.99%  82.5 5328s
 446552 65221 148984.480   38 1166 153102.283 143941.501  5.98%  82.5 5332s
 447082 65234 150301.648   39 1304 153102.283 143949.064  5.98%  82.5 5338s
 447555 65242 148255.244   45  798 153102.283 143953.691  5.98%  82.5 5342s
 447843 65274 146208.132   44  759 153102.283 143955.025  5.97%  82.5 5347s
 448600 65291 144887.475   45 1036 153102.283 143966.829  5.97%  82.4 5354s
 449027 65298 149348.841   38  476 153102.283 143970.354  5.96%  82.4 5359s
 449387 65300     cutoff   48      153102.283 143971.792  5.96%  82.4 5364s
 449929 65315 148220.854   42 1131 153102.283 143978.424  5.96%  82.4 5367s
 450190 65333 147899.811   49  862 153102.283 143980.331  5.96%  82.4 5373s
 451000 65315     cutoff   44      153102.283 143988.434  5.95%  82.4 5381s
 451395 65350 145870.335   46 1208 153102.283 143993.215  5.95%  82.4 5390s
 452589 65354 145454.003   43 1210 153102.283 144003.086  5.94%  82.4 5395s
 453090 65364 146623.961   36 1325 153102.283 144008.164  5.94%  82.4 5400s

Cutting planes:
  Gomory: 154
  Cover: 57
  Implied bound: 178
  Projected implied bound: 244
  Clique: 54
  MIR: 1278
  StrongCG: 1
  Flow cover: 1016
  Flow path: 18
  Inf proof: 2
  Zero half: 21
  RLT: 151
  Relax-and-lift: 52

Explored 453156 nodes (37339870 simplex iterations) in 5400.10 seconds (4544.71 work units)
Thread count was 8 (of 8 available processors)

Solution count 7: 153102 153379 154098 ... 156717

Time limit reached
Warning: max constraint violation (2.6603e-06) exceeds tolerance
Warning: max general constraint violation (2.6603e-06) exceeds tolerance
Best objective 1.531022829437e+05, best bound 1.440110343376e+05, gap 5.9380%

======================================================================
Leas=15000 ratio×1.0 LCM×1.0 bi×1.0 SL=0.97
======================================================================
  Status: TIME_LIMIT  |  Costo: 153,102.30€  |  Gap: 5.94%  |  Tempo: 5402.2s

  SKU  T   DC               B    c_CM    c_AM  ratio
  0    CM  [1]            3.0    2000    2632  1.316
  1    CM  [2]            3.0    2373    2937  1.237
  2    AM  [0]            1.0    3218    3367  1.046
  3    AM  [0]            1.0    3774    3589  0.951
  4    AM  [0]            1.0    4004    4333  1.082
  5    CM  [1]            2.0    4384    5135  1.171
  6    AM  [0]            1.0    5677    5387  0.949
  7    CM  [2]            2.0    6073    7633  1.257
  8    AM  [0]            1.0    6785    7633  1.125

  Stampanti: [1. 0. 0.]
  Costi:
    base_stock          :     72928.66€
    EOQ_CM              :      1692.75€
    EOQ_raw             :       244.46€
    holding_raw         :       187.50€
    leasing_3DP         :     15000.00€
    backorder_LT        :     21292.38€
    trasporto           :      5994.41€
    produzione_CM       :      6175.50€
    produzione_AM       :     26397.03€
    holding_inv         :      3189.60€
    TOTALE              :    153102.30€

  DETTAGLIO ASSEGNAZIONE CLIENTI (y_sol):
    SKU 0 (CM) → DC1: clienti=[0, 1, 2, 3, 4, 5], μ_DC=3.00, B=3.0
    SKU 1 (CM) → DC2: clienti=[0, 1, 2, 3, 4, 5], μ_DC=2.70, B=3.0
    SKU 2 (AM) → DC0: clienti=[0, 1, 2, 3, 4, 5], μ_DC=2.50, B=1.0
    SKU 3 (AM) → DC0: clienti=[0, 1, 2, 3, 4, 5], μ_DC=2.10, B=1.0
    SKU 4 (AM) → DC0: clienti=[0, 1, 2, 3, 4, 5], μ_DC=2.00, B=1.0
    SKU 5 (CM) → DC1: clienti=[0, 1, 2, 3, 4, 5], μ_DC=1.80, B=2.0
    SKU 6 (AM) → DC0: clienti=[0, 1, 2, 3, 4, 5], μ_DC=1.60, B=1.0
    SKU 7 (CM) → DC2: clienti=[0, 1, 2, 3, 4, 5], μ_DC=1.40, B=2.0
    SKU 8 (AM) → DC0: clienti=[0, 1, 2, 3, 4, 5], μ_DC=1.30, B=1.0

  DETTAGLIO BASE STOCK PER DC:
    DC0: 5 SKU (5 AM), B_tot=5.0, stampanti=1
    DC1: 2 SKU (0 AM), B_tot=5.0, stampanti=0
    DC2: 2 SKU (0 AM), B_tot=5.0, stampanti=0

  BACKORDER LT ANALITICO PER SKU:
    SKU   Tech      bo_LT        b_i   L_tech  μ_tot    β    E[BO]
    0     CM       425.09      49995   0.2500   3.00    3   0.0085
    1     CM       345.47      59332   0.2500   2.70    3   0.0058
    2     AM      2149.09      80460   0.0962   2.50    1   0.0267
    3     AM      2559.16      94356   0.1154   2.10    1   0.0271
    4     AM      3323.19     100103   0.1346   2.00    1   0.0332
    5     CM      2370.95     109609   0.3077   1.80    2   0.0216
    6     AM      3967.43     141918   0.1538   1.60    1   0.0280
    7     CM      1638.41     151831   0.3077   1.40    2   0.0108
    8     AM      4513.63     169627   0.1845   1.30    1   0.0266

  METRICHE ANOVA: n_AM=5, n_DC_CM=1.00, H̄=0.000, costo=153,102€

========================================================================
RUN: Leas=30000 ratio×1.0 LCM×1.0 bi×1.0 SL=0.97
========================================================================
  [CG] Precomputazione: 3,402 entry in 2.1s
  [CG] It   0: LB=   190,935  col=   90  new= 36
  [CG] It   5: LB=   190,935  col=  245  new= 11
  [CG] It  10: LB=   190,935  col=  259  new=  0
Set parameter OutputFlag to value 1
Set parameter TimeLimit to value 300
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: 11th Gen Intel(R) Core(TM) i7-1165G7 @ 2.80GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  300

Optimize a model with 57 rows, 262 columns and 922 nonzeros (Min)
Model fingerprint: 0x79da0699
Model has 262 linear objective coefficients
Variable types: 0 continuous, 262 integer (259 binary)
Coefficient statistics:
  Matrix range     [1e+00, 5e+03]
  Objective range  [5e+03, 3e+04]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]

Presolve removed 45 rows and 232 columns
Presolve time: 0.01s
Presolved: 12 rows, 30 columns, 57 nonzeros
Variable types: 0 continuous, 30 integer (30 binary)
Found heuristic solution: objective 209715.68220

Root relaxation: objective 1.984963e+05, 9 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

     0     0 198496.264    0    1 209715.682 198496.264  5.35%     -    0s
H    0     0                    208574.91781 198496.264  4.83%     -    0s
H    0     0                    201809.81261 198496.264  1.64%     -    0s
     0     0 198496.264    0    1 201809.813 198496.264  1.64%     -    0s

Explored 1 nodes (9 simplex iterations) in 0.06 seconds (0.00 work units)
Thread count was 8 (of 8 available processors)

Solution count 3: 201810 208575 209716 

Optimal solution found (tolerance 1.00e-04)
Best objective 2.018098126074e+05, best bound 2.018098126074e+05, gap 0.0000%

  CG: 9 AM [0, 1, 2, 3, 4, 5, 6, 7, 8], costo=201,810, n3dp=[1, 0, 0], t=2.3s
Set parameter OutputFlag to value 1
Set parameter MIPGap to value 0.05
Set parameter TimeLimit to value 5400
Set parameter StartNodeLimit to value 100
  [warm start] MINIMALE: 9 AM, n3DP=[1, 0, 0]
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: 11th Gen Intel(R) Core(TM) i7-1165G7 @ 2.80GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  5400
MIPGap  0.05
StartNodeLimit  100

Optimize a model with 10125 rows, 9513 columns and 36180 nonzeros (Min)
Model fingerprint: 0xe0e1c578
Model has 9027 linear objective coefficients
Model has 141 simple general constraints
  141 PWL
Variable types: 9246 continuous, 267 integer (264 binary)
Coefficient statistics:
  Matrix range     [4e-03, 5e+05]
  Objective range  [1e+00, 3e+04]
  Bounds range     [1e+00, 1e+00]
  RHS range        [3e-02, 5e+05]
  PWLCon x range   [4e-02, 1e+02]
  PWLCon y range   [0e+00, 1e+04]

Using variable hints.
User MIP start produced solution with objective 164988 (0.03s)
User MIP start produced solution with objective 164988 (0.03s)
Loaded user MIP start with objective 164988

Presolve removed 4702 rows and 1842 columns
Presolve time: 0.10s
Presolved: 5423 rows, 7671 columns, 27198 nonzeros
Presolved model has 141 SOS constraint(s)
Variable types: 7386 continuous, 285 integer (258 binary)

Root relaxation: objective 2.027407e+04, 1372 iterations, 0.03 seconds (0.02 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

     0     0 20274.0738    0  212 164987.512 20274.0738  87.7%     -    0s
     0     0 20274.0738    0  212 164987.512 20274.0738  87.7%     -    0s
     0     0 27993.9126    0  252 164987.512 27993.9126  83.0%     -    0s
     0     0 28047.6935    0  252 164987.512 28047.6935  83.0%     -    1s
     0     0 45646.7872    0  276 164987.512 45646.7872  72.3%     -    1s
     0     0 45787.2292    0  277 164987.512 45787.2292  72.2%     -    1s
     0     0 45806.3349    0  283 164987.512 45806.3349  72.2%     -    1s
     0     0 45807.6672    0  281 164987.512 45807.6672  72.2%     -    1s
     0     0 51863.3328    0  301 164987.512 51863.3328  68.6%     -    1s
     0     0 52177.4368    0  319 164987.512 52177.4368  68.4%     -    1s
     0     0 52179.1050    0  323 164987.512 52179.1050  68.4%     -    1s
     0     0 52975.6152    0  326 164987.512 52975.6152  67.9%     -    1s
     0     0 53000.2210    0  334 164987.512 53000.2210  67.9%     -    1s
     0     0 53015.6114    0  332 164987.512 53015.6114  67.9%     -    1s
     0     0 53017.6732    0  333 164987.512 53017.6732  67.9%     -    2s
     0     0 55392.3527    0  313 164987.512 55392.3527  66.4%     -    2s
     0     0 55425.7394    0  332 164987.512 55425.7394  66.4%     -    2s
     0     0 55517.8197    0  352 164987.512 55517.8197  66.4%     -    2s
     0     0 55518.5921    0  356 164987.512 55518.5921  66.3%     -    2s
H    0     0                    150498.39514 55518.5921  63.1%     -    2s
     0     0 57269.1522    0  351 150498.395 57269.1522  61.9%     -    2s
     0     0 59408.2444    0  353 150498.395 59408.2444  60.5%     -    3s
     0     0 59456.6277    0  367 150498.395 59456.6277  60.5%     -    3s
     0     0 59460.2182    0  373 150498.395 59460.2182  60.5%     -    3s
     0     0 74868.3360    0  346 150498.395 74868.3360  50.3%     -    3s
     0     0 75701.7113    0  351 150498.395 75701.7113  49.7%     -    3s
     0     0 75914.3644    0  357 150498.395 75914.3644  49.6%     -    3s
     0     0 75943.6357    0  362 150498.395 75943.6357  49.5%     -    3s
     0     0 75952.7745    0  364 150498.395 75952.7745  49.5%     -    3s
     0     0 86134.5690    0  354 150498.395 86134.5690  42.8%     -    4s
     0     0 87089.0624    0  367 150498.395 87089.0624  42.1%     -    4s
     0     0 87170.6761    0  374 150498.395 87170.6761  42.1%     -    4s
     0     0 87181.1353    0  371 150498.395 87181.1353  42.1%     -    4s
     0     0 90509.7457    0  362 150498.395 90509.7457  39.9%     -    4s
     0     0 91483.1543    0  359 150498.395 91483.1543  39.2%     -    4s
     0     0 91492.7379    0  370 150498.395 91492.7379  39.2%     -    4s
     0     0 92490.2933    0  362 150498.395 92490.2933  38.5%     -    4s
H    0     0                    150432.41356 92492.8977  38.5%     -    4s
     0     0 92774.3431    0  358 150432.414 92774.3431  38.3%     -    5s
     0     0 92802.0508    0  365 150432.414 92802.0508  38.3%     -    5s
     0     0 93462.4562    0  364 150432.414 93462.4562  37.9%     -    5s
     0     0 93485.7064    0  378 150432.414 93485.7064  37.9%     -    5s
     0     0 93782.9814    0  373 150432.414 93782.9814  37.7%     -    5s
     0     0 93786.2300    0  373 150432.414 93786.2300  37.7%     -    5s
     0     0 93786.2300    0  351 150432.414 93786.2300  37.7%     -    5s
     0     2 93788.1436    0  351 150432.414 93788.1436  37.7%     -    6s
   435   264 125422.844    9  263 150432.414 105047.827  30.2%   188   10s
  1634   934 147943.789   24  210 150432.414 116896.772  22.3%  80.8   15s
  1641   939 139406.257    7 2530 150432.414 116896.772  22.3%  80.4   20s
  1646   942 146964.937   12 2583 150432.414 116896.772  22.3%  80.2   25s
  1651   945 121742.968   32 2737 150432.414 116896.772  22.3%  79.9   30s
  1684   958 135665.232   19 2036 150432.414 116896.772  22.3%   122   35s
  1876   981 149942.785   38 1050 150432.414 117558.081  21.9%   127   40s
  2101  1145 121378.168   49 1666 150432.414 117954.506  21.6%   119   45s
  2701  1226 124468.986   33 1686 150432.414 117954.506  21.6%  98.1   50s
  3240  1394 129737.578   41 2039 150432.414 119018.120  20.9%  87.4   55s
  3917  1625 138241.582   54 1335 150432.414 119201.706  20.8%  79.6   60s
  4559  1939 132932.405   47 2027 150432.414 119216.580  20.8%  74.6   65s
  5189  2219 infeasible   53      150432.414 119306.546  20.7%  70.0   70s
  5651  2688 123393.529   31 2254 150432.414 119401.321  20.6%  67.5   75s
  6206  3090 142637.464   64 1238 150432.414 119484.714  20.6%  65.2   80s
  7205  3840 142203.647   54 1687 150432.414 119577.442  20.5%  61.7   86s
  8003  4171 142659.588   52 1534 150432.414 119634.353  20.5%  59.5   90s
  8729  4581 146450.109   51 1262 150432.414 119691.138  20.4%  58.1   95s
  9201  5176 148356.365   56 1769 150432.414 119754.535  20.4%  57.3  100s
  9739  5527 148211.715   56 1216 150432.414 119825.185  20.3%  56.2  106s
 10923  5977 124782.273   29 1559 150432.414 119984.902  20.2%  54.5  113s
 10948  6303 133758.934   37 1513 150432.414 119984.902  20.2%  54.5  115s
 11904  6742 122810.403   41 1693 150432.414 120089.587  20.2%  53.7  122s
 12173  7184 146904.946   55 1214 150432.414 120095.243  20.2%  53.7  127s
 12883  7273 149448.093   72  724 150432.414 120118.443  20.2%  53.2  133s
 12897  7301 149994.856   74  658 150432.414 120127.379  20.1%  53.2  136s
 13487  7975 infeasible   41      150432.414 120154.358  20.1%  52.5  142s
 14138  8391 121910.329   33 2240 150432.414 120192.208  20.1%  52.5  147s
 14980  9017 149329.060   55 1404 150432.414 120248.161  20.1%  52.3  151s
 15678  9466 123260.957   34 1678 150432.414 120313.014  20.0%  52.1  157s
 17093  9992     cutoff   50      150432.414 120417.439  20.0%  51.0  162s
 17145 10482 121634.311   33 2163 150432.414 120442.046  19.9%  51.0  165s
 18640 11426 129478.795   33 1733 150432.414 120513.810  19.9%  50.3  171s
 19898 11849     cutoff   54      150432.414 120569.479  19.9%  49.9  175s
 20579 12259 133736.394   41  210 150432.414 120613.818  19.8%  49.7  243s
 20581 12260 147792.386   61 1782 150432.414 120613.818  19.8%  49.7  245s
 20585 12263 126671.480   38 2843 150432.414 120613.818  19.8%  49.7  250s
 20588 12265 134151.891   42 2585 150432.414 120613.818  19.8%  49.7  258s
 20589 12266 139755.370   48 2791 150432.414 120613.818  19.8%  49.7  262s
 20590 12266 130815.429   40 2680 150432.414 120613.818  19.8%  49.7  268s
 20591 12267 144792.104   40 2758 150432.414 120613.818  19.8%  49.7  270s
 20592 12268 141825.867   67 2819 150432.414 120613.818  19.8%  49.7  275s
 20593 12268 137450.867   56 2682 150432.414 120613.818  19.8%  49.7  280s
 20594 12269 141090.544   45 2791 150432.414 120613.818  19.8%  49.7  285s
 20595 12270 141453.101   54 2758 150432.414 120613.818  19.8%  49.7  290s
 20596 12270 124466.717   39 2801 150432.414 120613.818  19.8%  49.7  296s
 20597 12271 123908.233   39 2652 150432.414 120613.818  19.8%  49.7  302s
 20598 12272 121468.358   31 2838 150432.414 120613.818  19.8%  49.7  305s
 20599 12272 124793.927   37 2851 150432.414 120613.818  19.8%  49.7  310s
 20600 12273 133807.548   46 2727 150432.414 120613.818  19.8%  49.7  316s
 20601 12274 148878.225   62 2875 150432.414 120613.818  19.8%  49.7  322s
 20602 12274 148199.272   47 2699 150432.414 120613.818  19.8%  49.7  328s
 20603 12275 147029.913   66 2835 150432.414 120613.818  19.8%  49.6  334s
 20604 12276 129135.772   41 2715 150432.414 120613.818  19.8%  49.6  340s
 20605 12276 139528.153   73 2777 150432.414 120613.818  19.8%  49.6  346s
 20606 12277 143778.623   52 2719 150432.414 120613.818  19.8%  49.6  352s
 20607 12278 142472.712   47 2832 150432.414 120613.818  19.8%  49.6  359s
 20608 12278 146613.847   70 2802 150432.414 120613.818  19.8%  49.6  365s
 20609 12279 142866.341   57 2846 150432.414 120613.818  19.8%  49.6  370s
 20610 12280 142061.171   51 2726 150432.414 120613.818  19.8%  49.6  376s
 20611 12280 149105.473   40 2857 150432.414 120613.818  19.8%  49.6  384s
 20612 12281 148199.408   59 2785 150432.414 120613.818  19.8%  49.6  390s
 20613 12282 122301.072   32 2885 150432.414 120613.818  19.8%  49.6  397s
 20614 12282 144229.569   62 2800 150432.414 120613.818  19.8%  49.6  402s
 20615 12283 130270.045   44 2854 150432.414 120613.818  19.8%  49.6  407s
 20616 12284 149538.756   74 2740 150432.414 120613.818  19.8%  49.6  414s
 20617 12284 134811.254   40 2926 150432.414 120613.818  19.8%  49.6  420s
 20618 12285 135098.813   49 2791 150432.414 120613.818  19.8%  49.6  426s
 20619 12286 126651.616   37 2863 150432.414 120613.818  19.8%  49.6  431s
 20620 12286 123931.569   31 2801 150432.414 120613.818  19.8%  49.6  437s
 20621 12287 137329.654   55 2869 150432.414 120613.818  19.8%  49.6  442s
 20622 12288 128684.463   33 2810 150432.414 120613.818  19.8%  49.6  447s
 20623 12288 149975.467   62 2887 150432.414 120613.818  19.8%  49.6  451s
 20624 12289 146989.615   77 2813 150432.414 120613.818  19.8%  49.6  456s
 20625 12290 131062.364   39 2910 150432.414 120613.818  19.8%  49.6  460s
 20626 12290 121065.057   27 2820 150432.414 120613.818  19.8%  49.6  465s
 20627 12291 133649.718   42 2908 150432.414 120613.818  19.8%  49.6  470s
 20628 12292 141540.969   50 2890 150432.414 120613.818  19.8%  49.6  475s
 20629 12292 143450.541   45 2939 150432.414 120613.818  19.8%  49.6  480s
 20631 12294 135113.377   48 2902 150432.414 120613.818  19.8%  49.6  488s
 20632 12294 135633.066   39 2860 150432.414 120613.818  19.8%  49.6  491s
 20635 12296 149860.007   62 2881 150432.414 120613.818  19.8%  49.6  495s
 20638 12298 140457.638   47 2873 150432.414 120613.818  19.8%  49.6  500s
 20641 12300 127935.889   34 2929 150432.414 120613.818  19.8%  49.6  505s
 20644 12302 144849.750   50 2833 150432.414 120613.818  19.8%  49.6  510s
 20648 12305 140129.932   45 2846 150432.414 120613.818  19.8%  49.5  516s
 20651 12307 150091.295   47 2921 150432.414 120613.818  19.8%  49.5  520s
 20654 12309 143015.831   54 2836 150432.414 120613.818  19.8%  49.5  525s
 20658 12312 133669.815   40 2850 150432.414 120613.818  19.8%  49.5  531s
 20661 12314 146523.600   46 2926 150432.414 120613.818  19.8%  49.5  535s
 20664 12316 124346.301   38 2845 150432.414 120613.818  19.8%  49.5  540s
 20668 12318 144569.033   53 2889 150432.414 120613.818  19.8%  49.5  546s
 20670 12320 142081.442   48 2899 150432.414 120613.818  19.8%  49.5  550s
 20673 12322 132541.124   44 2918 150432.414 120613.818  19.8%  49.5  555s
 20676 12324 144703.473   49 2919 150432.414 120613.818  19.8%  49.5  560s
 20679 12326 133736.394   41 2951 150432.414 120613.818  19.8%  49.5  565s
 20682 12328 149589.043   55 2881 150432.414 120613.818  19.8%  49.5  571s
 20685 12330 126671.480   38 2925 150432.414 120613.818  19.8%  49.5  576s
 20686 12330 123905.760   32 2899 150432.414 120613.818  19.8%  49.4  580s
 20689 12332 139755.370   48 2960 150432.414 120613.818  19.8%  49.4  585s
 20692 12334 141825.867   67 2864 150432.414 120613.818  19.8%  49.4  591s
 20695 12336 141453.101   54 2951 150432.414 120613.818  19.8%  49.4  596s
 20698 12338 121468.358   31 2874 150432.414 120613.818  19.8%  49.4  603s
 20700 12340 133807.548   46 2910 150432.414 120613.818  19.8%  49.4  606s
 20702 12341 148199.272   47 2857 150432.414 120613.818  19.8%  49.4  610s
 20705 12343 139528.153   73 2925 150432.414 120613.818  19.8%  49.4  615s
 20708 12345 146613.847   70 2878 150432.414 120613.818  19.8%  49.4  622s
 20710 12346 142061.171   51 2888 150432.414 120613.818  19.8%  49.4  625s
 20713 12348 122301.072   32 2944 150432.414 120613.818  19.8%  49.4  631s
 20715 12350 130270.045   44 2961 150432.414 120613.818  19.8%  49.4  635s
 20718 12352 135098.813   49 2898 150432.414 120613.818  19.8%  49.4  641s
 20720 12353 123931.569   31 2857 150432.414 120613.818  19.8%  49.4  645s
 20724 12356 146989.615   77 2881 150432.414 120613.818  19.8%  49.4  652s
 20726 12357 121065.057   27 2894 150432.414 120613.818  19.8%  49.4  655s
 20729 12359 143450.541   45 2913 150432.414 120613.818  19.8%  49.3  660s
 20730 12360 126036.608   38 2881 150432.414 120613.818  19.8%  49.3  666s
 20732 12361 135633.066   39 2912 150432.414 120613.818  19.8%  49.3  670s
 20735 12363 149860.007   62 2915 150432.414 120613.818  19.8%  49.3  675s
 20738 12365 140457.638   47 2881 150432.414 120613.818  19.8%  49.3  681s
 20740 12366 133619.351   35 2898 150432.414 120613.818  19.8%  49.3  685s
 20744 12369 144849.750   50 2869 150432.414 120613.818  19.8%  49.3  692s
 20746 12370 145471.146   52 2906 150432.414 120613.818  19.8%  49.3  695s
 20749 12372 146161.351   56 2954 150432.414 120613.818  19.8%  49.3  700s
 20752 12374 126029.319   42 2918 150432.414 120613.818  19.8%  49.3  707s
 20754 12376 143015.831   54 2896 150432.414 120613.818  19.8%  49.3  710s
 20757 12378 146310.310   50 2957 150432.414 120613.818  19.8%  49.3  715s
 20760 12380 133258.013   66 2938 150432.414 120613.818  19.8%  49.3  721s
 20762 12381 141127.737   50 2898 150432.414 120613.818  19.8%  49.3  725s
 20766 12384 145258.001   46 2828 150432.414 120613.818  19.8%  49.3  731s
 20768 12385 144569.033   53 2900 150432.414 120613.818  19.8%  49.3  735s
 20771 12387 141068.978   44 2929 150432.414 120613.818  19.8%  49.2  740s
 20773 12388 132541.124   44 2937 150432.414 120613.818  19.8%  49.2  745s
 20776 12390 144703.473   49 2887 150432.414 120613.818  19.8%  49.2  751s
 20779 12392 133736.394   41 2945 150432.414 120613.818  19.8%  49.2  756s
 20782 12394 149589.043   55 2903 150432.414 120613.818  19.8%  49.2  760s
 20785 12396 126671.480   38 2955 150432.414 120613.818  19.8%  49.2  765s
 20787 12398 144207.203   51 2898 150432.414 120613.818  19.8%  49.2  771s
 20788 12401 120613.818   24 2859 150432.414 120613.818  19.8%  70.3  776s
 20798 12414 120613.818   27 2635 150432.414 120613.818  19.8%  71.8  780s
 20844 12414 141561.495   35 2184 150432.414 120613.818  19.8%  72.9  786s
 20876 12413 148887.265   43 2168 150432.414 120613.818  19.8%  72.9  791s
 20895 12416 122036.806   30 2339 150432.414 120613.818  19.8%  73.7  795s
 20921 12414 125237.171   35 1916 150432.414 120613.818  19.8%  74.4  802s
 20932 12412 127500.281   39 1693 150432.414 120613.818  19.8%  74.5  805s
 20948 12417 128571.815   42 1654 150432.414 120613.818  19.8%  74.7  810s
 20990 12435 134455.319   55 1505 150432.414 120613.818  19.8%  75.0  815s
 21092 12467 138624.988   50 1496 150432.414 121428.513  19.3%  75.4  822s
 21172 12493 143822.224   58 1507 150432.414 121428.513  19.3%  75.7  825s
 21326 12518 124911.478   33 2277 150432.414 121627.291  19.1%  75.8  830s
 21432 12568 136656.166   55 1865 150432.414 121779.429  19.0%  76.2  835s
 21586 12587 125889.350   43 1617 150432.414 121779.429  19.0%  76.8  843s
 21689 12590 123621.269   33 2385 150432.414 122212.962  18.8%  77.1  848s
 21732 12607 124658.850   36 1914 150432.414 122212.962  18.8%  77.2  851s
 21845 12626 142295.800   60 1395 150432.414 122212.962  18.8%  77.3  856s
 21960 12650 137919.233   44 1882 150432.414 122212.962  18.8%  77.7  863s
 22020 12673 149967.818   64 1439 150432.414 122212.962  18.8%  77.9  865s
 22175 12693 137571.854   52 1609 150432.414 122212.962  18.8%  77.9  871s
 22342 12709 126624.302   37 1922 150432.414 123205.394  18.1%  78.5  876s
 22468 12752 142365.380   54 1947 150432.414 123286.385  18.0%  79.1  881s
 22664 12772 infeasible   51      150432.414 123286.385  18.0%  79.0  887s
 22737 12785 124427.250   37 2413 150432.414 123286.385  18.0%  79.1  891s
 22822 12776 126476.543   38 2374 150432.414 123286.385  18.0%  78.9  895s
 22984 12820 126462.753   40 2240 150432.414 123286.385  18.0%  79.2  903s
 23082 12838 infeasible   61      150432.414 123450.082  17.9%  79.5  907s
 23188 12868 infeasible   42      150432.414 123450.675  17.9%  79.7  910s
 23301 12870     cutoff   61      150432.414 123450.675  17.9%  79.8  915s
 23378 12871 infeasible   39      150432.414 123579.190  17.9%  80.0  921s
 23425 12913 144483.225   48 2094 150432.414 123579.190  17.9%  80.2  925s
 23665 12960 138944.957   42 1962 150432.414 123893.214  17.6%  80.7  934s
 23785 13057 124417.229   38 2286 150432.414 123965.042  17.6%  80.9  940s
 24077 13021 130403.452   39 2167 150432.414 124036.386  17.5%  81.0  945s
 24170 13134 infeasible   68      150432.414 124036.386  17.5%  81.7  950s
 24478 13114 148020.404   47 2086 150432.414 124178.675  17.5%  81.7  956s
 24635 13260     cutoff   60      150432.414 124319.558  17.4%  81.8  962s
 24981 13186 130999.536   39 2192 150432.414 124450.789  17.3%  81.7  969s
 25054 13302 148711.855   49 2154 150432.414 124450.789  17.3%  81.7  975s
 25295 13322 129506.513   35 1809 150432.414 124513.054  17.2%  82.1  981s
 25496 13478     cutoff   65      150432.414 124513.054  17.2%  82.1  988s
 25928 13418 129026.225   36 1721 150432.414 124728.897  17.1%  81.9  997s
 26082 13635 142462.740   63 2008 150432.414 124728.897  17.1%  82.0 1004s
 26646 13606 150146.997   83 1929 150432.414 124910.760  17.0%  82.2 1012s
 27020 14056 129906.615   37 2215 150432.414 125380.248  16.7%  82.6 1021s
 28018 13797 141203.476   45 2120 150432.414 125783.203  16.4%  82.8 1028s
 28201 14463 149448.436   49 1997 150432.414 125859.210  16.3%  83.1 1037s
 29511 14163 139110.179   43 2210 150432.414 126226.839  16.1%  83.2 1044s
 29737 14631     cutoff   56      150432.414 126242.174  16.1%  83.5 1052s
 30818 14570 132917.528   38 2142 150432.414 126672.061  15.8%  83.8 1063s
 31355 15063 143618.996   51 1997 150432.414 126843.255  15.7%  84.1 1073s
 32649 14708 132321.229   44 1912 150432.414 127222.566  15.4%  84.8 1077s
 32752 14937 139483.089   57 1301 150432.414 127239.382  15.4%  85.0 1082s
 33320 14789 150171.979   51 1745 150432.414 127436.100  15.3%  85.4 1085s
 33414 15169 147134.452   52 1563 150432.414 127470.947  15.3%  85.5 1091s
 34326 15138 infeasible   55      150432.414 127756.640  15.1%  85.7 1100s
 34688 15062 146489.473   52 1562 150432.414 128023.265  14.9%  86.0 1105s
 34782 15400 149111.750   54 1726 150432.414 128023.265  14.9%  86.2 1110s
 35530 15183 139431.926   29 2410 150432.414 128380.647  14.7%  86.0 1116s
 35624 15523 137507.822   38 1739 150432.414 128479.613  14.6%  86.3 1121s
 36374 15345 infeasible   61      150432.414 128681.060  14.5%  86.1 1125s
 37177 15457 138579.806   50 1726 150432.414 129139.424  14.2%  86.3 1133s
 37359 15726 136446.551   41 2350 150432.414 129281.207  14.1%  86.5 1137s
 38047 15550 136404.534   50 1780 150432.414 129802.885  13.7%  86.4 1141s
 38230 15780 142949.364   45 1846 150432.414 130011.115  13.6%  86.5 1145s
 38900 15904     cutoff   51      150432.414 130258.012  13.4%  86.6 1153s
 39554 15736     cutoff   67      150432.414 130562.306  13.2%  86.6 1158s
 39663 15720 136870.743   44 1726 150432.414 130674.487  13.1%  86.7 1162s
 39686 15801     cutoff   55      150432.414 130674.487  13.1%  86.7 1166s
 39885 16045 147096.939   54 1650 150432.414 130700.925  13.1%  86.7 1170s
 40572 16213 140831.147   47 1241 150432.414 130926.908  13.0%  86.7 1177s
 41206 16055 147221.898   59 1356 150432.414 131056.367  12.9%  86.7 1186s
 41357 16646 133961.461   44 1996 150432.414 131108.221  12.8%  86.8 1213s
 42553 16318 144372.535   40 1926 150432.414 131199.515  12.8%  86.9 1223s
 42674 16365 148648.747   64 1329 150432.414 131199.515  12.8%  86.9 1233s
 42851 17045 149534.235   60 1527 150432.414 131199.515  12.8%  87.0 1246s
 44398 16565 infeasible   44      150432.414 131609.300  12.5%  87.2 1253s
 44496 17096 149592.814   56 1706 150432.414 131609.300  12.5%  87.2 1260s
 45540 16840 142567.989   28 2533 150432.414 132223.844  12.1%  87.4 1268s
 45682 17232 148335.016   55 1455 150432.414 132250.599  12.1%  87.5 1273s
 46487 17158 infeasible   45      150432.414 132372.565  12.0%  87.6 1279s
 46777 17150 143231.579   46 1491 150432.414 132405.688  12.0%  87.9 1283s
 46953 17363 140872.267   42 1901 150432.414 132431.141  12.0%  88.0 1289s
 47430 17236 147223.399   53 1363 150432.414 132533.951  11.9%  88.1 1296s
 47517 17655 136273.280   43 2223 150432.414 132601.304  11.9%  88.2 1301s
 48320 17435 133635.043   41 2247 150432.414 132775.698  11.7%  88.3 1307s
 48421 17752 138342.970   38 2376 150432.414 132784.344  11.7%  88.4 1312s
 49101 17571 146283.176   54 1875 150432.414 132860.989  11.7%  88.6 1317s
 49209 17845 143037.430   56 1903 150432.414 132912.127  11.6%  88.7 1322s
 49772 17713 infeasible   40      150432.414 132989.977  11.6%  89.0 1328s
 49859 18079 150105.564   56 1192 150432.414 132989.977  11.6%  89.1 1333s
 50590 17877 148539.866   51 1685 150432.414 133079.821  11.5%  89.1 1337s
 50669 18190     cutoff   53      150432.414 133099.898  11.5%  89.2 1343s
 51344 18043     cutoff   58      150432.414 133225.722  11.4%  89.3 1347s
 51504 18466 141707.259   41 2115 150432.414 133279.886  11.4%  89.4 1354s
 51988 18304 150034.310   58 1345 150432.414 133280.278  11.4%  89.5 1355s
 52424 18212 141758.913   64 1744 150432.414 133412.228  11.3%  89.7 1361s
 52517 18442 145611.816   55 1530 150432.414 133469.146  11.3%  89.8 1365s
 53268 18599 147769.341   54 1256 150432.414 133669.419  11.1%  90.2 1374s
 53862 18430 144974.513   51 1568 150432.414 133737.978  11.1%  90.1 1378s
 53911 18727 146812.073   52 1566 150432.414 133737.978  11.1%  90.2 1383s
 54435 18589 146324.109   58 1158 150432.414 133805.849  11.1%  90.3 1389s
 54516 18896 142860.986   47 1864 150432.414 133838.655  11.0%  90.4 1394s
 55103 18765 146148.089   53 1611 150432.414 133892.582  11.0%  90.6 1397s
 55221 19011 136151.425   46 2226 150432.414 133902.798  11.0%  90.6 1402s
 55754 18915 139211.036   48 1884 150432.414 133964.713  10.9%  90.8 1407s
 55898 18952 136211.529   43 1898 150432.414 133987.890  10.9%  90.9 1412s
 56056 19027 145489.717   45 2140 150432.414 133998.953  10.9%  90.8 1417s
 56293 19153 138591.214   47 1911 150432.414 134011.412  10.9%  90.9 1422s
 56672 19068 144047.908   51 1706 150432.414 134070.559  10.9%  91.1 1426s
 56760 19376     cutoff   60      150432.414 134102.084  10.9%  91.2 1431s
 57346 19236 145232.554   49 1665 150432.414 134152.160  10.8%  91.3 1438s
 57459 19502 135352.041   46 2006 150432.414 134186.487  10.8%  91.4 1443s
 58092 19391 147939.633   59 1635 150432.414 134251.791  10.8%  91.5 1447s
 58196 19689 infeasible   58      150432.414 134257.540  10.8%  91.5 1451s
 58793 19749 146413.518   57 1057 150432.414 134302.301  10.7%  91.6 1457s
 58884 20093 143234.264   53 1744 150432.414 134338.629  10.7%  91.7 1462s
 59540 20155     cutoff   66      150432.414 134386.676  10.7%  91.7 1466s
 59650 20459 149927.807   71  416 150432.414 134391.403  10.7%  91.8 1470s
 60239 20502 141791.836   47 1736 150432.414 134421.352  10.6%  91.8 1475s
 60329 20819     cutoff   62      150432.414 134439.064  10.6%  91.9 1480s
 61035 21113     cutoff   71      150432.414 134486.880  10.6%  92.0 1489s
 61540 21224     cutoff   65      150432.414 134534.826  10.6%  92.1 1493s
 61756 21721 141759.534   50 1292 150432.414 134561.556  10.6%  92.2 1500s
 62682 21748 148454.527   65 1698 150432.414 134599.789  10.5%  92.3 1507s
 62717 22005 144806.491   52 1772 150432.414 134599.789  10.5%  92.4 1511s
 63355 22366 138439.471   47 1961 150432.414 134660.310  10.5%  92.5 1519s
 63831 22371 139876.793   48 1986 150432.414 134688.716  10.5%  92.7 1524s
 63883 22616 infeasible   56      150432.414 134691.269  10.5%  92.7 1529s
 64376 22918 137480.096   47 1783 150432.414 134717.529  10.4%  92.9 1535s
 64892 22983 141773.758   51 1524 150432.414 134746.510  10.4%  93.0 1540s
 65016 23235 137564.197   46 1895 150432.414 134769.819  10.4%  93.0 1545s
 65752 23644 149598.263   58 1577 150432.414 134846.765  10.4%  93.2 1554s
 66350 23660 147345.196   64 1005 150432.414 134877.058  10.3%  93.3 1560s
 66381 23844 147847.599   65  913 150432.414 134877.058  10.3%  93.3 1565s
 66915 24146 infeasible   52      150432.414 134987.915  10.3%  93.3 1574s
 67389 24198 149507.051   63 1519 150432.414 135029.166  10.2%  93.5 1578s
 67470 24536 135562.516   47 1847 150432.414 135031.991  10.2%  93.5 1583s
 68145 24578 147781.879   57 1665 150432.414 135078.337  10.2%  93.4 1587s
 68232 24928 150400.748   78 1140 150432.414 135078.337  10.2%  93.5 1591s
 68790 24972 149051.452   65  217 150432.414 135105.069  10.2%  93.6 1598s
 68895 25285 147732.591   54 1642 150432.414 135134.205  10.2%  93.6 1603s
 69465 25548 146088.110   57 1483 150432.414 135154.253  10.2%  93.6 1608s
 69972 25614 145658.136   64 1247 150432.414 135182.549  10.1%  93.7 1614s
 70089 25867 infeasible   48      150432.414 135182.549  10.1%  93.7 1618s
 70534 26195 142667.972   45 2282 150432.414 135219.145  10.1%  93.8 1623s
 71170 26257 140149.372   50 1804 150432.414 135281.976  10.1%  93.8 1628s
 71266 26520 141225.534   51 1806 150432.414 135289.974  10.1%  93.9 1633s
 71741 26716 148083.954   58 1703 150432.414 135327.314  10.0%  94.0 1637s
 72128 26939 146288.538   56 1503 150432.414 135345.491  10.0%  94.0 1644s
 72538 27243     cutoff   61      150432.414 135383.526  10.0%  94.1 1651s
 73160 27354     cutoff   30      150432.414 135404.065  10.0%  94.4 1656s
 73343 27804 137908.449   43 2261 150432.414 135420.551  10.0%  94.5 1663s
 74203 27980 146227.072   59 1345 150432.414 135473.843  9.94%  94.5 1668s
 74585 28138 139973.357   46 1948 150432.414 135504.808  9.92%  94.6 1673s
 74874 28444 137005.006   44 2015 150432.414 135528.023  9.91%  94.8 1678s
 75473 28512 149706.246   56  859 150432.414 135553.438  9.89%  94.7 1683s
 75588 28780 150432.336   51 1805 150432.414 135559.853  9.89%  94.8 1689s
 76172 28996 140911.835   49 1871 150432.414 135591.879  9.87%  94.9 1694s
 76617 29071 140435.694   44 1871 150432.414 135619.773  9.85%  94.9 1702s
 76745 29365 140698.948   51 1961 150432.414 135624.345  9.84%  94.9 1707s
 77388 29670 142542.766   52 1146 150432.414 135649.880  9.83%  94.9 1713s
 77993 29707 149246.140   71 1125 150432.414 135670.281  9.81%  95.0 1718s
 78065 29909 148334.796   56 1301 150432.414 135670.281  9.81%  95.1 1723s
 78576 29939 143760.231   51 1407 150432.414 135709.012  9.79%  95.1 1727s
 78665 30200 150110.661   55 1663 150432.414 135709.012  9.79%  95.1 1732s
 79322 30253 136843.676   44 1819 150432.414 135745.725  9.76%  95.2 1737s
 79428 30531 146258.684   53 1240 150432.414 135745.725  9.76%  95.2 1742s
 80004 30563 136200.505   42 1817 150432.414 135776.057  9.74%  95.3 1746s
 80093 30763 143887.878   46 1771 150432.414 135776.057  9.74%  95.3 1752s
 80479 30962 147809.818   64 1547 150432.414 135787.865  9.73%  95.4 1757s
 80847 30998 149230.828   53 1665 150432.414 135804.761  9.72%  95.5 1762s
 80936 31277 145701.144   46 1700 150432.414 135804.761  9.72%  95.5 1767s
 81536 31464 137472.510   43 1892 150432.414 135837.544  9.70%  95.6 1773s
 81882 31530 145966.349   53 1704 150432.414 135852.685  9.69%  95.6 1777s
 82032 31824 146045.560   54 1772 150432.414 135852.685  9.69%  95.7 1783s
 82668 31906 139206.299   40 1879 150432.414 135878.906  9.67%  95.7 1790s
 82834 33023 150265.287   53 1009 150432.414 135878.906  9.67%  95.8 1809s
 85120 33047 149384.053   57 1598 150432.414 135943.911  9.63%  95.9 1813s
 85184 33110 149403.500   58 1589 150432.414 135946.646  9.63%  95.9 1817s
 85290 33353     cutoff   57      150432.414 135959.324  9.62%  95.9 1821s
 85813 33385 147981.190   54 1612 150432.414 135979.606  9.61%  96.0 1827s
 85854 33440 148376.036   55 1559 150432.414 135982.355  9.61%  96.0 1831s
 85979 33703 140847.073   49 1891 150432.414 135985.303  9.60%  96.0 1836s
 86497 33797     cutoff   56      150432.414 135998.643  9.59%  96.1 1840s
 86721 34069 137682.213   39 2196 150432.414 136019.864  9.58%  96.2 1845s
 87244 34097     cutoff   56      150432.414 136033.244  9.57%  96.2 1853s
 87300 34395 144477.879   47 1703 150432.414 136033.244  9.57%  96.3 1858s
 87852 34626 148363.164   58 1748 150432.414 136053.824  9.56%  96.3 1864s
 88310 34662 147767.927   58 1500 150432.414 136071.203  9.55%  96.5 1870s
 88386 34953 145286.531   51 1677 150432.414 136071.203  9.55%  96.5 1875s
 88955 34985 148333.177   50 1674 150432.414 136092.145  9.53%  96.5 1880s
 89030 35248 141748.816   48 1724 150432.414 136094.291  9.53%  96.6 1885s
 89799 35616 148362.005   64 1605 150432.414 136121.716  9.51%  96.7 1895s
 90383 35930 146287.339   52 1849 150432.414 136149.957  9.49%  96.9 1904s
 91036 35997     cutoff   69      150432.414 136161.555  9.49%  96.7 1908s
 91170 36250 145679.522   55 1197 150432.414 136171.972  9.48%  96.8 1913s
 91678 36320 149897.690   53 1671 150432.414 136186.746  9.47%  96.8 1917s
 91797 36589 141736.908   48 1779 150432.414 136200.869  9.46%  96.9 1921s
 92311 36729 145002.596   49 1906 150432.414 136216.976  9.45%  96.9 1926s
 92600 36771 147217.376   61 1744 150432.414 136221.047  9.45%  96.9 1930s
 92709 37012 145790.081   50 1412 150432.414 136232.892  9.44%  97.0 1935s
 93205 37080 142350.057   46 2146 150432.414 136239.602  9.43%  97.1 1941s
 93394 37803 145831.597   55 1276 150432.414 136244.789  9.43%  97.1 1955s
 94917 37831 147735.080   52 1587 150432.414 136278.955  9.41%  97.4 1963s
 95002 38133 143106.870   48 1798 150432.414 136288.119  9.40%  97.4 1968s
 95607 38183 148718.663   57 1609 150432.414 136311.134  9.39%  97.5 1973s
 95699 38469 149103.456   58 1624 150432.414 136314.146  9.39%  97.5 1979s
 96231 38647 infeasible   46      150432.414 136329.025  9.38%  97.6 1983s
 96630 38736 149254.286   54 1756 150432.414 136360.542  9.35%  97.6 1988s
 96792 38991 144455.314   57 1556 150432.414 136360.542  9.35%  97.7 1993s
 97370 39020 138036.788   45 2134 150432.414 136383.826  9.34%  97.7 2000s
 97436 39335 138240.734   46 2095 150432.414 136383.826  9.34%  97.7 2005s
 98036 39594     cutoff   56      150432.414 136397.268  9.33%  97.8 2011s
 98628 39616 147185.131   54 1834 150432.414 136416.824  9.32%  97.8 2016s
 98694 39884 138974.469   50 1894 150432.414 136418.724  9.32%  97.8 2021s
 99267 39976 143906.744   47 2235 150432.414 136433.513  9.31%  97.7 2025s
 99480 40225 148511.101   49 1720 150432.414 136435.066  9.30%  97.8 2030s
 100035 40240     cutoff   59      150432.414 136468.521  9.28%  97.8 2038s
 100107 40498 144260.677   49 1875 150432.414 136470.462  9.28%  97.8 2043s
 100602 40783 149917.811   76  656 150432.414 136481.794  9.27%  97.9 2048s
 101199 40820 143410.570   54 1877 150432.414 136504.456  9.26%  97.9 2053s
 101269 41093 148001.158   54 1936 150432.414 136506.575  9.26%  97.9 2058s
 101817 41314     cutoff   66      150432.414 136514.627  9.25%  98.0 2064s
 102339 41334 143848.618   44 1954 150432.414 136525.839  9.24%  98.0 2070s
 102386 41339 148174.407   47 1781 150432.414 136525.839  9.24%  98.0 2077s
 102405 41397 139305.628   50 1788 150432.414 136530.567  9.24%  98.0 2082s
 102515 41667 149182.802   48 1798 150432.414 136534.780  9.24%  98.1 2088s
 103101 41681 150152.533   59 1611 150432.414 136543.104  9.23%  98.1 2094s
 103173 41932 142896.649   51 1646 150432.414 136544.384  9.23%  98.2 2099s
 103713 42139 147896.089   58 1666 150432.414 136565.322  9.22%  98.1 2107s
 104122 42435 149330.362   50 1920 150432.414 136589.483  9.20%  98.2 2114s
 104768 42447     cutoff   59      150432.414 136611.001  9.19%  98.2 2120s
 104819 42793 139436.398   45 1811 150432.414 136611.001  9.19%  98.3 2127s
 105538 42872 145326.126   53 1587 150432.414 136629.682  9.18%  98.3 2131s
 105735 43176 143519.008   48 2242 150432.414 136643.551  9.17%  98.4 2138s
 106476 43208 142737.047   52 1624 150432.414 136659.030  9.16%  98.4 2144s
 106548 43498 140517.603   46 2220 150432.414 136661.084  9.15%  98.4 2150s
 107155 43613     cutoff   50      150432.414 136696.796  9.13%  98.4 2155s
 107398 43639 147374.483   61 1201 150432.414 136708.863  9.12%  98.5 2161s
 107483 43887 148331.603   59 1627 150432.414 136716.178  9.12%  98.5 2167s
 108015 44058 150303.584   66  717 150432.414 136725.661  9.11%  98.6 2172s
 108395 44127     cutoff   57      150432.414 136732.889  9.11%  98.6 2176s
 108523 44403 145246.725   54 1183 150432.414 136735.645  9.10%  98.7 2181s
 109142 44409 infeasible   53      150432.414 136753.179  9.09%  98.7 2188s
 109170 44650     cutoff   52      150432.414 136754.020  9.09%  98.7 2193s
 109813 44874 145013.068   49 1650 150432.414 136770.341  9.08%  98.7 2198s
 110273 44927     cutoff   61      150432.414 136785.654  9.07%  98.8 2204s
 110486 45190 142046.064   49 1926 150432.414 136792.785  9.07%  98.8 2210s
 111127 45368     cutoff   64      150432.414 136807.835  9.06%  98.7 2215s
 111584 45401 148683.750   57 1054 150432.414 136820.697  9.05%  98.8 2222s
 111705 45708 149949.238   48 1872 150432.414 136828.712  9.04%  98.8 2227s
 112308 45772 137652.930   42 1973 150432.414 136843.304  9.03%  98.8 2234s
 112479 45810 139969.314   43 1942 150432.414 136843.304  9.03%  98.8 2240s
 112578 45863 149150.357   57 1658 150432.414 136855.298  9.03%  98.8 2246s
 112723 45992 150355.336   70 1665 150432.414 136860.401  9.02%  98.8 2251s
 113018 46148     cutoff   57      150432.414 136866.798  9.02%  98.9 2256s
 113395 46245 147043.661   56 1545 150432.414 136877.435  9.01%  98.9 2261s
 113675 46548 139981.721   52 2224 150432.414 136890.377  9.00%  99.0 2267s
 114293 46571 146731.224   61 1531 150432.414 136901.075  8.99%  99.0 2275s
 114386 48214 142469.314   44 2067 150432.414 136910.545  8.99%  99.0 2312s
 118093 48237 150335.405   54 1657 150432.414 137019.409  8.92%  99.1 2320s
 118178 48277 148699.032   58 1483 150432.414 137023.650  8.91%  99.1 2325s
 118299 48322 144898.991   59 2051 150432.414 137033.689  8.91%  99.1 2330s
 118455 48635 146971.004   58 1298 150432.414 137042.630  8.90%  99.2 2336s
 119129 48715     cutoff   57      150432.414 137059.396  8.89%  99.1 2342s
 119305 48997 142604.486   46 1873 150432.414 137062.203  8.89%  99.1 2348s
 119896 49051     cutoff   58      150432.414 137084.705  8.87%  99.2 2352s
 120033 49313 149965.601   57 1737 150432.414 137088.067  8.87%  99.3 2358s
 120674 49340 infeasible   55      150432.414 137112.508  8.85%  99.2 2364s
 120746 49628     cutoff   58      150432.414 137113.326  8.85%  99.2 2369s
 121396 49859 infeasible   51      150432.414 137128.588  8.84%  99.2 2375s
 121869 49896 147318.819   57 1667 150432.414 137140.455  8.84%  99.2 2380s
 121970 50148 149223.822   53 1290 150432.414 137147.134  8.83%  99.3 2386s
 122534 50298 140734.644   51 2194 150432.414 137153.679  8.83%  99.3 2391s
 122889 50328     cutoff   67      150432.414 137173.919  8.81%  99.4 2395s
 122991 50382 142869.267   47 2277 150432.414 137185.756  8.81%  99.4 2402s
 123138 50439 139588.622   45 1943 150432.414 137190.324  8.80%  99.4 2406s
 123252 50664 150334.534   56 1370 150432.414 137192.777  8.80%  99.5 2411s
 123838 50701 148573.129   55 1740 150432.414 137208.809  8.79%  99.5 2416s
 123922 50982 148125.594   56 1641 150432.414 137208.809  8.79%   100 2422s
 124553 51134 147876.773   53 1751 150432.414 137229.459  8.78%   100 2428s
 124914 51320 149769.522   60 1238 150432.414 137248.811  8.76%   100 2436s
 125477 51438 149625.075   51 1582 150432.414 137262.737  8.75%   100 2442s
 125779 51844 143208.428   43 2181 150432.414 137277.731  8.74%   100 2450s
 126598 51867 149187.007   60 1250 150432.414 137294.705  8.73%   100 2457s
 126680 52185 149645.361   67 1251 150432.414 137294.705  8.73%   100 2464s
 127388 52396 138154.893   49 1906 150432.414 137322.999  8.71%   100 2470s
 128063 52705 infeasible   68      150432.414 137353.675  8.69%   100 2480s
 128696 52927 147336.075   55 1134 150432.414 137376.736  8.68%   100 2485s
 129237 52962 149300.325   51 1867 150432.414 137394.678  8.67%   100 2491s
 129308 53178     cutoff   47      150432.414 137399.783  8.66%   100 2496s
 129910 53271     cutoff   57      150432.414 137416.507  8.65%   100 2501s
 130139 53506 138919.604   40 2272 150432.414 137424.376  8.65%   100 2506s
 130662 53532 143514.763   47 1892 150432.414 137449.122  8.63%   100 2512s
 130735 53740 144060.910   48 1869 150432.414 137449.122  8.63%   100 2517s
 131264 53767 144291.087   45 1991 150432.414 137466.180  8.62%   100 2521s
 131332 54029     cutoff   57      150432.414 137469.517  8.62%   100 2527s
 131899 54150 infeasible   58      150432.414 137482.817  8.61%   100 2531s
 132129 54320 139379.762   41 2185 150432.414 137497.626  8.60%   100 2537s
 132601 54370     cutoff   56      150432.414 137511.208  8.59%   100 2541s
 132690 54573 146833.972   55 1131 150432.414 137511.312  8.59%   100 2546s
 133218 54635 141274.694   54 1742 150432.414 137529.481  8.58%   100 2551s
 133353 54838 149159.946   57 1632 150432.414 137532.156  8.58%   100 2556s
 133900 54879     cutoff   60      150432.414 137548.808  8.56%   100 2562s
 134033 55098 147911.381   50 1387 150432.414 137556.940  8.56%   100 2567s
 134596 55206     cutoff   61      150432.414 137571.206  8.55%   100 2572s
 134851 55307     cutoff   59      150432.414 137575.664  8.55%   100 2578s
 135136 55621 140169.214   46 1611 150432.414 137586.109  8.54%   100 2587s
 135905 55635     cutoff   55      150432.414 137607.043  8.53%   100 2596s
 135926 55952 infeasible   49      150432.414 137611.115  8.52%   100 2603s
 136677 55974 141943.860   42 1706 150432.414 137633.348  8.51%   101 2611s
 136747 56270 138760.919   50 1731 150432.414 137633.348  8.51%   101 2618s
 137443 56430     cutoff   55      150432.414 137656.272  8.49%   101 2622s
 137782 56431 145217.019   49 1627 150432.414 137662.683  8.49%   101 2629s
 137789 56435 145488.900   50 1640 150432.414 137666.144  8.49%   101 2638s
 137793 56439 146069.349   51 1577 150432.414 137666.144  8.49%   101 2644s
 137797 56442 147335.932   51 1602 150432.414 137666.144  8.49%   101 2650s
 137802 56446 146933.147   52 1543 150432.414 137666.144  8.49%   101 2657s
 137806 56449 147341.414   52 1579 150432.414 137666.144  8.49%   101 2664s
 137811 56453 147504.171   52 1287 150432.414 137666.144  8.49%   101 2670s
 137815 56457 147516.793   53 1290 150432.414 137666.144  8.49%   101 2677s
 137819 56461 147565.326   54 1284 150432.414 137666.144  8.49%   101 2683s
 137823 56465 147969.097   55 1524 150432.414 137666.144  8.49%   101 2690s
 137827 56465 148309.775   55 1279 150432.414 137666.144  8.49%   101 2704s
 137835 56468 147638.946   53 1259 150432.414 137666.144  8.49%   101 2709s
 137839 56482 147989.914   54 1251 150432.414 137666.144  8.49%   101 2715s
 137869 56493     cutoff   58      150432.414 137667.187  8.49%   101 2721s
 137905 56494 144710.646   46 1644 150432.414 137667.187  8.49%   101 2726s
 137945 56537 138534.415   47 1585 150432.414 137673.951  8.48%   101 2734s
 138038 56578 144135.020   49 1469 150432.414 137677.266  8.48%   101 2742s
 138141 56609 147410.614   46 1647 150432.414 137679.715  8.48%   101 2748s
 138227 56653 146370.934   48 1664 150432.414 137683.093  8.48%   101 2756s
 138317 56690 147112.024   54 1442 150432.414 137683.710  8.47%   101 2764s
 138431 56757 145955.926   48 1580 150432.414 137683.710  8.47%   101 2774s
 138582 57092 147750.283   55 1666 150432.414 137683.710  8.47%   101 2783s
 139561 57151 139728.275   40 1855 150432.414 137724.102  8.45%   101 2790s
 139664 57481 149523.679   62  895 150432.414 137724.102  8.45%   101 2797s
 140406 57681 150240.376   61 1685 150432.414 137749.282  8.43%   101 2804s
 140939 57763 148131.427   51 1530 150432.414 137761.866  8.42%   101 2810s
 141193 58002 149312.174   58 1241 150432.414 137775.035  8.41%   101 2817s
 141875 58026 146613.707   48 1586 150432.414 137790.687  8.40%   101 2825s
 141978 58342 148450.628   50 1593 150432.414 137792.358  8.40%   101 2831s
 142629 58351 149506.181   62 1255 150432.414 137812.357  8.39%   101 2839s
 142654 58576 139976.266   47 1930 150432.414 137815.542  8.39%   101 2846s
 143317 58601 139849.938   51 1625 150432.414 137828.969  8.38%   101 2854s
 143368 58824     cutoff   64      150432.414 137828.969  8.38%   101 2861s
 144024 58943 139436.160   46 1607 150432.414 137842.665  8.37%   101 2866s
 144319 59059 146783.249   55 1596 150432.414 137853.309  8.36%   101 2871s
 144642 59214     cutoff   50      150432.414 137866.478  8.35%   101 2877s
 145044 59438 150020.958   61 1608 150432.414 137877.089  8.35%   101 2883s
 145602 59476     cutoff   49      150432.414 137890.301  8.34%   101 2889s
 145734 60423 148617.084   57 1690 150432.414 137890.301  8.34%   101 2911s
 148094 60429 148304.118   61 1599 150432.414 137953.875  8.30%   101 2916s
 148141 60464 140896.553   49 1580 150432.414 137956.959  8.29%   101 2920s
 148213 60491 147947.625   61 1191 150432.414 137958.887  8.29%   101 2926s
 148283 60732 145890.801   49 1552 150432.414 137965.290  8.29%   101 2931s
 148840 60933 138898.776   44 1799 150432.414 137986.986  8.27%   101 2935s
 149219 60993 146699.666   58 1186 150432.414 137996.820  8.27%   101 2940s
 149379 61276     cutoff   60      150432.414 137997.408  8.27%   101 2945s
 149965 61326 140369.227   46 1420 150432.414 138015.226  8.25%   101 2951s
 150076 61599 147034.497   51 1584 150432.414 138016.968  8.25%   101 2956s
 150716 61856 140361.972   53 1782 150432.414 138032.820  8.24%   101 2961s
 151324 61887     cutoff   58      150432.414 138042.697  8.24%   101 2968s
 151396 62070 148015.776   65 1245 150432.414 138045.447  8.23%   101 2972s
 151938 62308     cutoff   63      150432.414 138056.990  8.23%   101 2978s
 152531 62322 149235.645   53 1503 150432.414 138077.642  8.21%   101 2983s
 152606 62534 147684.541   58 1471 150432.414 138080.984  8.21%   101 2988s
 153199 62675 149509.769   53 1571 150432.414 138096.416  8.20%   101 2992s
 153486 62763     cutoff   54      150432.414 138100.762  8.20%   101 2997s
 153781 62853 150118.611   52 1640 150432.414 138108.550  8.19%   101 3001s
 154008 63048 150040.746   56 1474 150432.414 138113.713  8.19%   101 3005s
 154528 63073     cutoff   64      150432.414 138131.140  8.18%   101 3012s
 154616 63288 144829.922   45 1416 150432.414 138133.003  8.18%   101 3017s
 155195 63297 150364.704   55 1264 150432.414 138142.081  8.17%   101 3023s
 155217 63330 147309.630   56 1721 150432.414 138145.016  8.17%   101 3027s
 155338 63553 139715.492   46 1751 150432.414 138151.185  8.16%   101 3032s
 155886 63663 infeasible   52      150432.414 138162.823  8.16%   101 3039s
 156147 64085 150246.220   59 1707 150432.414 138170.546  8.15%   101 3050s
 157516 64228 149519.150   57 1158 150432.414 138213.113  8.12%   101 3059s
 157863 64302     cutoff   47      150432.414 138217.917  8.12%   101 3063s
 158183 64446 144890.100   47 1559 150432.414 138229.728  8.11%   101 3070s
 158553 64478     cutoff   56      150432.414 138229.728  8.11%   101 3077s
 158613 64725 infeasible   51      150432.414 138240.400  8.10%   101 3083s
 159306 64772 140237.750   48 1846 150432.414 138261.564  8.09%   101 3087s
 159401 65022 141928.920   48 1568 150432.414 138267.062  8.09%   101 3093s
 160033 65043 145370.707   60 1439 150432.414 138279.147  8.08%   101 3100s
 160087 65256 147190.782   61 1409 150432.414 138279.147  8.08%   101 3106s
 160761 65451 145162.631   49 1729 150432.414 138294.047  8.07%   101 3111s
 161221 65494     cutoff   53      150432.414 138308.375  8.06%   101 3115s
 161355 65699 149169.858   52 1681 150432.414 138323.018  8.05%   101 3120s
 162159 66077 infeasible   45      150432.414 138348.884  8.03%   101 3129s
 162805 66102 148600.020   56 1485 150432.414 138363.773  8.02%   101 3136s
 162893 66330 144600.700   50 1610 150432.414 138370.933  8.02%   101 3141s
 163474 66524 149699.644   60 1396 150432.414 138388.625  8.01%   101 3147s
 164064 66553 149965.634   57 1608 150432.414 138409.199  7.99%   101 3152s
 164158 66796 143133.864   58 1684 150432.414 138417.458  7.99%   101 3157s
 164750 66996 138896.389   47 1835 150432.414 138430.539  7.98%   101 3163s
 165326 67044 145464.148   51 1583 150432.414 138444.887  7.97%   101 3168s
 165419 67210 141236.812   47 2395 150432.414 138446.433  7.97%   101 3173s
 165999 67244 140685.531   57 1818 150432.414 138458.108  7.96%   101 3177s
 166142 67506 146194.133   54 1570 150432.414 138462.266  7.96%   101 3185s
 167033 67522 149361.818   48 1629 150432.414 138478.990  7.95%   101 3193s
 167082 67687 146127.542   51 1370 150432.414 138483.133  7.94%   101 3197s
 167498 67829 146573.164   49 1549 150432.414 138501.521  7.93%   101 3201s
 167814 67863     cutoff   60      150432.414 138510.725  7.92%   101 3205s
 167902 68108 150218.182   59 1700 150432.414 138510.725  7.92%   101 3210s
 168505 68179 143421.347   49 1675 150432.414 138528.572  7.91%   101 3215s
 169063 68408 infeasible   43      150432.414 138543.759  7.90%   101 3223s
 169319 68635 145654.888   50 1473 150432.414 138552.783  7.90%   101 3229s
 169895 68663 149210.610   56 1171 150432.414 138569.766  7.89%   101 3234s
 169976 68860 149779.501   56 1371 150432.414 138574.300  7.88%   101 3239s
 170568 68881 144715.988   47 1507 150432.414 138591.354  7.87%   101 3243s
 170655 69088 147258.824   51 1447 150432.414 138591.640  7.87%   101 3248s
 171175 69190 150224.067   65  753 150432.414 138611.078  7.86%   101 3253s
 171492 69268 144643.968   52 1970 150432.414 138615.578  7.86%   101 3257s
 171724 69479 150301.699   64 1198 150432.414 138620.716  7.85%   101 3263s
 172382 69497 139719.091   47 1974 150432.414 138641.460  7.84%   101 3267s
 172447 69660     cutoff   55      150432.414 138641.460  7.84%   101 3273s
 172903 69705 145736.286   54 1895 150432.414 138655.328  7.83%   101 3276s
 173024 69919 149673.553   52 1553 150432.414 138661.156  7.82%   101 3281s
 173631 70000 146766.384   58 1295 150432.414 138676.876  7.81%   101 3286s
 173898 70220 147925.242   57 1581 150432.414 138686.566  7.81%   101 3291s
 174535 70257 143142.788   61 1666 150432.414 138705.971  7.80%   101 3296s
 174631 70464 145526.990   52 1451 150432.414 138709.670  7.79%   101 3300s
 175181 70644 146818.684   51 1569 150432.414 138726.701  7.78%   101 3305s
 175746 70860     cutoff   63      150432.414 138737.381  7.77%   101 3315s
 176404 71047 140652.480   46 1644 150432.414 138757.165  7.76%   101 3322s
 176856 71208     cutoff   65      150432.414 138762.524  7.76%   101 3328s
 177359 71305 149974.642   55 1297 150432.414 138782.259  7.74%   101 3334s
 177603 71611     cutoff   58      150432.414 138789.291  7.74%   101 3341s
 178429 71617 150058.353   57 1594 150432.414 138804.360  7.73%   101 3351s
 178509 71846 143637.756   51 1781 150432.414 138819.832  7.72%   101 3357s
 179273 71951 139083.986   49 1959 150432.414 138830.846  7.71%   101 3364s
 179582 71962 148648.075   50 1454 150432.414 138844.765  7.70%   101 3369s
 179634 72175 146385.913   58 1538 150432.414 138849.401  7.70%   101 3375s
 180395 72304 infeasible   50      150432.414 138867.579  7.69%   101 3380s
 180812 72330 149062.770   56 1252 150432.414 138879.035  7.68%   101 3386s
 180866 72557 141216.942   51 1682 150432.414 138879.035  7.68%   101 3391s
 181465 72723 150430.947   41 1791 150432.414 138897.930  7.67%   101 3396s
 181927 72726 142160.079   46 1795 150432.414 138906.094  7.66%   101 3401s
 181932 72737 144446.223   47 1778 150432.414 138906.094  7.66%   101 3406s
 181983 72756 146739.073   45 1360 150432.414 138906.914  7.66%   101 3412s
 182024 72787 141739.894   45 1722 150432.414 138907.408  7.66%   101 3417s
 182077 72804 147011.051   48 1564 150432.414 138907.408  7.66%   101 3422s
 182119 72846     cutoff   51      150432.414 138907.408  7.66%   101 3428s
 182220 72854 150018.109   61 1217 150432.414 138909.177  7.66%   101 3433s
 182246 72876 150278.428   62 1164 150432.414 138914.841  7.66%   101 3438s
 182303 72925     cutoff   59      150432.414 138915.048  7.66%   101 3443s
 182430 72927 infeasible   50      150432.414 138919.987  7.65%   101 3449s
 182449 72954 143998.350   58 1835 150432.414 138921.469  7.65%   101 3453s
 182507 73100     cutoff   63      150432.414 138922.040  7.65%   101 3459s
 183032 73113 149028.707   51 1502 150432.414 138922.040  7.65%   101 3465s
 183085 73304 147530.468   48 1870 150432.414 138932.637  7.64%   101 3471s
 183604 73348 147822.327   57 1293 150432.414 138937.558  7.64%   101 3476s
 183703 73577 infeasible   58      150432.414 138937.558  7.64%   101 3482s
 184396 73591     cutoff   53      150432.414 138964.547  7.62%   101 3488s
 184444 73777 149056.979   55 1421 150432.414 138967.963  7.62%   101 3494s
 184992 73951 145535.502   57 1652 150432.414 138980.590  7.61%   101 3500s
 185636 74299 146700.553   54 1772 150432.414 138995.271  7.60%   101 3510s
 186300 74469 145019.657   50 1563 150432.414 139012.084  7.59%   101 3515s
 186813 74499 149097.293   56 1626 150432.414 139019.491  7.59%   101 3523s
 186927 75813 141965.240   44 1636 150432.414 139023.245  7.58%   101 3556s
 190855 75813     cutoff   50      150432.414 139110.796  7.53%   101 3562s
 190863 75817 144666.481   47 1867 150432.414 139111.829  7.53%   101 3568s
 190887 75846 144984.451   43 1967 150432.414 139113.046  7.52%   101 3571s
 190962 75856     cutoff   50      150432.414 139117.976  7.52%   101 3578s
 190986 75890 142397.813   50 1502 150432.414 139118.470  7.52%   101 3581s
 191076 75952 148694.213   63 1373 150432.414 139118.470  7.52%   101 3585s
 191788 76191 147758.919   58 1573 150432.414 139137.357  7.51%   101 3594s
 191921 76368 147766.246   59 1570 150432.414 139139.371  7.51%   101 3599s
 192396 76454 141897.372   59 1610 150432.414 139160.367  7.49%   101 3603s
 192590 76576 149405.328   56 1390 150432.414 139163.809  7.49%   101 3608s
 193121 76604 139896.580   49 1828 150432.414 139178.998  7.48%   101 3614s
 193183 76812 149482.566   59 1474 150432.414 139178.998  7.48%   101 3619s
 193760 76941     cutoff   63      150432.414 139201.035  7.47%   101 3623s
 194074 77013 147087.707   61 1552 150432.414 139206.086  7.46%   101 3627s
 194316 77185 148853.364   51 1464 150432.414 139210.674  7.46%   101 3632s
 194770 77233 143657.538   45 1886 150432.414 139224.108  7.45%   101 3635s
 194867 77418 150054.792   50 1576 150432.414 139224.108  7.45%   101 3640s
 195490 77609 149376.891   55 1416 150432.414 139240.219  7.44%   101 3648s
 195923 77619 150133.424   62 1653 150432.414 139252.807  7.43%   101 3653s
 195959 77790 149658.064   65 1401 150432.414 139252.807  7.43%   101 3658s
 196489 77925 145353.642   50 1599 150432.414 139260.712  7.43%   101 3663s
 196943 77955 145544.744   53 1557 150432.414 139273.147  7.42%   101 3666s
 197030 78213 148640.575   47 1671 150432.414 139273.147  7.42%   101 3674s
 197802 78224 infeasible   57      150432.414 139296.189  7.40%   101 3678s
 197850 78332 144301.124   51 1641 150432.414 139300.749  7.40%   101 3682s
 198185 78398     cutoff   66      150432.414 139311.951  7.39%   101 3686s
 198374 78610 140169.720   50 1773 150432.414 139317.480  7.39%   101 3692s
 198886 78620     cutoff   53      150432.414 139323.429  7.38%   101 3697s
 198980 78828 147913.837   54 1678 150432.414 139328.101  7.38%   101 3703s
 199597 78919 149317.127   50 1424 150432.414 139345.130  7.37%   101 3708s
 200009 79042 142729.147   44 1705 150432.414 139362.659  7.36%   101 3713s
 200334 79200 148195.515   57 1675 150432.414 139376.434  7.35%   101 3718s
 200745 79213 149771.966   63 1211 150432.414 139386.047  7.34%   101 3723s
 200816 79461 146840.697   53 1476 150432.414 139390.574  7.34%   101 3729s
 201443 79515 150058.091   60  725 150432.414 139407.571  7.33%   101 3734s
 201606 79619 150097.361   61 1219 150432.414 139412.552  7.33%   101 3740s
 202095 79872 149870.714   51 1567 150432.414 139427.186  7.32%   101 3750s
 202803 79937 148713.014   64 1140 150432.414 139448.028  7.30%   101 3755s
 203013 80149 143024.989   50 1792 150432.414 139452.073  7.30%   101 3761s
 203583 80210 140495.677   50 1782 150432.414 139471.482  7.29%   101 3765s
 203738 80430 144415.027   51 1568 150432.414 139475.982  7.28%   101 3771s
 204320 80537 140181.477   49 1743 150432.414 139489.918  7.27%   101 3776s
 204607 80632 139822.773   51 1760 150432.414 139501.212  7.27%   101 3780s
 204873 80761 149223.569   56 1439 150432.414 139508.302  7.26%   101 3785s
 205457 81026 145858.041   50 1498 150432.414 139525.637  7.25%   101 3795s
 206214 81340 146411.275   53 1348 150432.414 139540.216  7.24%   101 3805s
 206857 81354 148654.572   57 1166 150432.414 139551.145  7.23%   101 3811s
 206950 81569     cutoff   64      150432.414 139553.407  7.23%   101 3816s
 207564 81716 149395.966   53 1576 150432.414 139569.713  7.22%   101 3823s
 207919 81997 149902.701   61 1499 150432.414 139580.873  7.21%   101 3832s
 208919 82010 149112.949   56 1653 150432.414 139601.951  7.20%   101 3838s
 208948 82188     cutoff   57      150432.414 139602.865  7.20%   101 3843s
 209480 82210 146322.348   54 1590 150432.414 139622.196  7.19%   101 3846s
 209555 82430 140533.833   47 1670 150432.414 139625.881  7.18%   101 3852s
 210115 82466 147314.727   56 1467 150432.414 139634.609  7.18%   101 3856s
 210153 82521 147887.625   57 1460 150432.414 139634.609  7.18%   101 3864s
 210316 82561 149573.499   66 1669 150432.414 139634.609  7.18%   101 3869s
 210503 82756 142632.791   47 1626 150432.414 139652.051  7.17%   101 3875s
 211080 82778     cutoff   52      150432.414 139662.780  7.16%   101 3881s
 211200 82993 148405.464   49 1633 150432.414 139666.264  7.16%   101 3887s
 211845 83120     cutoff   57      150432.414 139684.217  7.14%   101 3893s
 212185 83164 147313.439   49 1486 150432.414 139690.438  7.14%   101 3898s
 212297 83337     cutoff   54      150432.414 139696.955  7.14%   101 3903s
 212810 83518 145464.267   48 1781 150432.414 139710.804  7.13%   101 3909s
 213345 83540 150066.798   74 1180 150432.414 139720.789  7.12%   101 3915s
 213402 83690 149964.050   47 1602 150432.414 139720.789  7.12%   101 3920s
 214005 83944 146025.084   48 1655 150432.414 139743.104  7.11%   101 3930s
 214831 84170 146594.955   51 1128 150432.414 139763.792  7.09%   101 3939s
 215300 84190 147373.098   52 1517 150432.414 139772.873  7.09%   101 3945s
 215381 84375 145933.234   51 1507 150432.414 139775.157  7.08%   101 3951s
 216029 84561     cutoff   54      150432.414 139780.455  7.08%   101 3956s
 216572 84567 150287.225   57 1657 150432.414 139797.588  7.07%   101 3962s
 216619 84761 149799.034   59 1343 150432.414 139797.588  7.07%   101 3968s
 217218 84777     cutoff   64      150432.414 139815.003  7.06%   101 3972s
 217298 84934 142756.468   48 1524 150432.414 139818.471  7.06%   101 3978s
 217838 85131 141137.891   50 1696 150432.414 139825.704  7.05%   101 3985s
 218355 85228 149573.878   52 1464 150432.414 139834.325  7.05%   101 3990s
 218565 85489     cutoff   60      150432.414 139834.325  7.05%   101 3998s
 219454 85487 148708.839   52 1991 150432.414 139851.268  7.03%   101 4007s
 219489 85697 144090.880   54 1881 150432.414 139852.820  7.03%   101 4013s
 220142 85890 143675.010   49 1424 150432.414 139865.854  7.02%   101 4018s
 220711 85907 149620.016   52 1423 150432.414 139875.613  7.02%   101 4023s
 220763 86101 148921.444   51 1607 150432.414 139875.613  7.02%   101 4029s
 221397 86261 141221.834   46 1822 150432.414 139892.373  7.01%   101 4034s
 221870 86284 148487.231   58 1376 150432.414 139901.251  7.00%   101 4040s
 222379 86530 149531.969   61 1579 150432.414 139917.382  6.99%   101 4050s
 222701 86769 150038.961   59 1387 150432.414 139924.719  6.98%   101 4059s
 223306 86845 141417.483   52 1737 150432.414 139938.608  6.98%   101 4063s
 223527 86894 150316.288   55 1317 150432.414 139944.924  6.97%   101 4067s
 223726 87048 144859.161   47 1690 150432.414 139950.918  6.97%   101 4071s
 224186 87065 145820.914   60 1300 150432.414 139959.909  6.96%   101 4077s
 224266 87261     cutoff   61      150432.414 139959.909  6.96%   101 4082s
 224840 87358 144308.353   58 1621 150432.414 139967.789  6.96%   101 4086s
 225144 87484 149181.634   55 1688 150432.414 139975.462  6.95%   101 4091s
 225476 87566 148905.624   48 1648 150432.414 139985.637  6.94%   101 4095s
 225687 87769 142696.780   45 1823 150432.414 139988.406  6.94%   101 4100s
 226331 87968 148746.892   50 1555 150432.414 140004.940  6.93%   101 4109s
 226901 88079 146829.545   50 1532 150432.414 140015.845  6.92%   101 4113s
 227235 88169 147512.392   51 1469 150432.414 140027.718  6.92%   101 4118s
 227499 88313 148522.924   52 1618 150432.414 140034.559  6.91%   101 4123s
 228023 88334 143088.626   45 1645 150432.414 140042.306  6.91%   101 4129s
 228111 88690 141796.283   48 1699 150432.414 140047.749  6.90%   101 4138s
 229028 88756 141535.353   52 1709 150432.414 140061.009  6.89%   101 4141s
 229226 88911 145896.951   58 1573 150432.414 140063.887  6.89%   101 4145s
 229705 89107 140868.527   49 1639 150432.414 140072.837  6.89%   101 4154s
 230195 89175 144935.859   56 1526 150432.414 140085.988  6.88%   101 4158s
 230471 89229 146616.821   45 1614 150432.414 140095.857  6.87%   101 4162s
 230750 89333 146655.148   53 1607 150432.414 140103.277  6.87%   101 4166s
 231077 89420 147029.351   54 1620 150432.414 140108.920  6.86%   101 4170s
 231352 89603 149354.814   61  849 150432.414 140115.757  6.86%   101 4176s
 231879 89627 149476.949   55 1348 150432.414 140124.860  6.85%   101 4180s
 231950 89829 infeasible   56      150432.414 140126.557  6.85%   101 4185s
 232471 89908 145924.285   62 1414 150432.414 140138.272  6.84%   101 4190s
 232734 89920 150021.785   59 1226 150432.414 140138.272  6.84%   101 4197s
 232800 90061 145416.978   52 1666 150432.414 140144.275  6.84%   101 4202s
 233301 90067 149912.623   62 1703 150432.414 140155.838  6.83%   101 4207s
 233351 90288 148293.144   50 1706 150432.414 140155.838  6.83%   101 4213s
 233910 90398     cutoff   56      150432.414 140174.702  6.82%   101 4217s
 234257 90484 144792.840   51 1705 150432.414 140181.824  6.81%   101 4222s
 234493 90709 142296.714   51 1862 150432.414 140187.164  6.81%   101 4228s
 235109 90719     cutoff   58      150432.414 140200.722  6.80%   101 4236s
 235131 90896     cutoff   48      150432.414 140202.289  6.80%   101 4241s
 235740 91124 infeasible   66      150432.414 140213.640  6.79%   101 4247s
 236301 91126 infeasible   62      150432.414 140222.906  6.79%   101 4255s
 236385 91329 147918.674   51 1518 150432.414 140232.659  6.78%   101 4261s
 236942 91502     cutoff   50      150432.414 140240.295  6.78%   101 4266s
 237470 91527 142971.352   50 1741 150432.414 140248.029  6.77%   101 4272s
 237536 91756     cutoff   57      150432.414 140248.365  6.77%   101 4277s
 238201 91927     cutoff   60      150432.414 140259.694  6.76%   101 4283s
 238738 91966 148643.868   48 1620 150432.414 140274.326  6.75%   101 4290s
 238855 93067 149273.006   49 1609 150432.414 140274.886  6.75%   101 4320s
 242359 93064 149557.321   57 1578 150432.414 140334.527  6.71%   101 4331s
 242413 93077 144690.885   52 1514 150432.414 140334.527  6.71%   101 4338s
 242453 93099 150165.045   58 1409 150432.414 140336.082  6.71%   101 4342s
 242530 93273     cutoff   59      150432.414 140338.161  6.71%   101 4347s
 242950 93331 infeasible   52      150432.414 140343.295  6.71%   101 4352s
 243169 93337 149303.953   52 1423 150432.414 140345.878  6.71%   101 4358s
 243213 93489 147395.235   53 1435 150432.414 140345.878  6.71%   101 4363s
 243611 93560 150089.260   56 1187 150432.414 140352.580  6.70%   101 4366s
 243792 93760 148910.675   52 1598 150432.414 140357.037  6.70%   101 4371s
 244344 93780 148654.952   59 1254 150432.414 140367.382  6.69%   101 4378s
 244424 93920 148758.725   62 1231 150432.414 140367.382  6.69%   101 4382s
 244906 93934     cutoff   61      150432.414 140389.740  6.68%   101 4386s
 244956 94073 148684.480   59 1696 150432.414 140390.459  6.68%   101 4391s
 245485 94125 146203.444   50 1562 150432.414 140394.020  6.67%   101 4395s
 245628 94319     cutoff   53      150432.414 140398.714  6.67%   101 4400s
 246317 94513 145041.078   44 1754 150432.414 140416.608  6.66%   101 4408s
 246818 94523 147373.507   63 1310 150432.414 140431.124  6.65%   101 4412s
 246880 94685 147832.242   64 1255 150432.414 140433.319  6.65%   101 4417s
 247517 94711 145292.073   50 1578 150432.414 140449.042  6.64%   101 4422s
 247575 94915 144951.772   50 1565 150432.414 140449.042  6.64%   101 4427s
 248194 94999 147190.570   45 1602 150432.414 140462.225  6.63%   101 4431s
 248515 95029 149006.137   55 1551 150432.414 140467.863  6.62%   101 4435s
 248581 95210 147632.006   63 1443 150432.414 140472.200  6.62%   101 4440s
 249201 95241 148873.114   60 1566 150432.414 140484.538  6.61%   101 4446s
 249270 96406 144658.657   46 1672 150432.414 140485.094  6.61%   101 4486s
 252900 96442     cutoff   53      150432.414 140547.641  6.57%   101 4494s
 252991 96462     cutoff   48      150432.414 140547.641  6.57%   101 4498s
 253197 96543 143117.347   47 1907 150432.414 140551.912  6.57%   101 4507s
 253407 96546 148701.441   59  284 150432.414 140553.515  6.57%   101 4515s
 253411 96546 148738.134   60  443 150432.414 140553.515  6.57%   101 4522s
 253418 96550 149254.970   59  312 150432.414 140553.515  6.57%   101 4529s
 253422 96552 148664.842   60  335 150432.414 140553.515  6.57%   101 4534s
 253428 96560 150091.650   59   49 150432.414 140553.515  6.57%   101 4541s
 253477 96571     cutoff   58      150432.414 140553.515  6.57%   101 4550s
 253538 96594 147591.870   49 1456 150432.414 140557.197  6.56%   101 4558s
 253635 96819 149995.232   56 1538 150432.414 140559.469  6.56%   101 4565s
 254411 96827     cutoff   68      150432.414 140575.138  6.55%   101 4573s
 254486 97055 141600.423   49 1532 150432.414 140576.574  6.55%   101 4579s
 255094 97145 146761.230   50 1595 150432.414 140579.787  6.55%   101 4585s
 255392 97382 146892.216   49 1586 150432.414 140591.395  6.54%   101 4592s
 256037 97385 146753.072   54 1603 150432.414 140604.801  6.53%   101 4598s
 256105 97646 145536.731   56 1593 150432.414 140606.824  6.53%   101 4604s
 256823 97769 148945.897   62 1501 150432.414 140620.359  6.52%   101 4609s
 257097 97949 149245.418   51 1610 150432.414 140624.135  6.52%   101 4616s
 257668 97977 146573.464   50 1715 150432.414 140633.350  6.51%   101 4622s
 257706 98154     cutoff   62      150432.414 140635.724  6.51%   101 4629s
 258320 98249     cutoff   61      150432.414 140645.344  6.51%   101 4635s
 258588 98450 142614.088   53 1758 150432.414 140649.354  6.50%   101 4641s
 259242 98492 145583.155   51 1454 150432.414 140665.375  6.49%   101 4647s
 259352 98671 145075.079   53 1472 150432.414 140667.112  6.49%   101 4653s
 259914 98743 149409.622   58 1609 150432.414 140674.776  6.49%   101 4659s
 260246 98938 142037.331   46 1436 150432.414 140683.800  6.48%   101 4667s
 260753 99091 146876.220   64 1498 150432.414 140687.916  6.48%   101 4674s
 261213 99260 143824.424   48 1739 150432.414 140695.059  6.47%   101 4682s
 261890 99291     cutoff   56      150432.414 140706.207  6.47%   101 4691s
 262039 99524     cutoff   60      150432.414 140711.784  6.46%   101 4698s
 262765 99743 149717.930   49 1867 150432.414 140723.788  6.45%   101 4705s
 263502 99772 149332.827   61 1490 150432.414 140734.685  6.45%   101 4713s
 263613 99951 140949.199   47 1819 150432.414 140741.475  6.44%   101 4719s
 264295 100168 146654.690   60 1619 150432.414 140755.709  6.43%   101 4726s
 264837 100236 143635.302   52 1452 150432.414 140767.654  6.42%   101 4731s
 264963 100423 141637.450   60 1484 150432.414 140768.143  6.42%   102 4737s
 265601 100509 141316.133   47 1583 150432.414 140787.703  6.41%   102 4742s
 265831 100534 145658.301   51 1582 150432.414 140791.192  6.41%   102 4747s
 265937 100795 143529.402   48 1690 150432.414 140795.552  6.41%   102 4753s
 266570 100921 142445.490   54 1216 150432.414 140804.339  6.40%   102 4758s
 266917 101010 146624.410   54 1487 150432.414 140810.115  6.40%   102 4763s
 267213 101244 146287.883   47 1595 150432.414 140815.515  6.39%   102 4769s
 267816 101247 146136.723   53 1752 150432.414 140824.015  6.39%   102 4777s
 267852 101419 143274.892   52 1704 150432.414 140824.365  6.39%   102 4783s
 268500 101621 144759.737   52 1627 150432.414 140835.475  6.38%   102 4789s
 269028 101654 143622.641   54 1464 150432.414 140842.285  6.38%   102 4796s
 269132 101801 148561.039   51 1834 150432.414 140844.204  6.37%   102 4801s
 269610 101957 149482.605   56 1550 150432.414 140853.187  6.37%   102 4807s
 270038 101973 149280.062   77 1040 150432.414 140857.366  6.37%   102 4811s
 270083 102145 147356.608   53 1547 150432.414 140857.366  6.37%   102 4817s
 270570 102248 150376.186   59 1647 150432.414 140863.320  6.36%   102 4823s
 270823 102650     cutoff   59      150432.414 140865.382  6.36%   102 4835s
 272106 102644     cutoff   65      150432.414 140882.102  6.35%   102 4844s
 272143 102799 147470.531   51 1435 150432.414 140888.359  6.34%   102 4849s
 272640 102810     cutoff   60      150432.414 140894.501  6.34%   102 4854s
 272671 102945 148367.989   64 1572 150432.414 140895.629  6.34%   102 4859s
 273146 102970     cutoff   54      150432.414 140911.823  6.33%   102 4864s
 273289 103193     cutoff   55      150432.414 140916.321  6.33%   102 4870s
 273911 103205     cutoff   58      150432.414 140928.010  6.32%   102 4875s
 273965 103334 144562.362   50 1565 150432.414 140928.010  6.32%   102 4880s
 274498 103426 148511.501   57 1610 150432.414 140944.513  6.31%   102 4885s
 274742 103580 infeasible   49      150432.414 140948.031  6.30%   102 4891s
 275331 103621 142935.630   56 1759 150432.414 140957.820  6.30%   102 4897s
 275423 103779     cutoff   50      150432.414 140958.157  6.30%   102 4902s
 275932 103873 142645.225   43 1796 150432.414 140973.409  6.29%   102 4907s
 276306 103949 147189.310   52 1772 150432.414 140980.845  6.28%   102 4911s
 276478 104151 148674.166   55 1685 150432.414 140980.845  6.28%   102 4917s
 276913 104182 146833.201   60  856 150432.414 140987.820  6.28%   102 4923s
 277023 104352 150336.933   59 1651 150432.414 140990.316  6.28%   102 4929s
 277666 104375 149684.669   47 1419 150432.414 140996.910  6.27%   102 4933s
 277750 104530     cutoff   56      150432.414 140998.394  6.27%   102 4938s
 278214 104548 149484.610   50 1631 150432.414 141007.685  6.27%   102 4943s
 278263 104765 143094.692   49 1445 150432.414 141007.685  6.27%   102 4949s
 278891 104797     cutoff   70      150432.414 141011.158  6.26%   102 4953s
 278986 104955 147436.311   62  773 150432.414 141014.859  6.26%   102 4958s
 279486 105064 147710.248   62 1242 150432.414 141027.550  6.25%   102 4963s
 279774 105186 147099.215   58  846 150432.414 141031.737  6.25%   102 4968s
 280091 105223 147003.585   51 1673 150432.414 141039.012  6.24%   102 4973s
 280246 105382 147633.247   53 1623 150432.414 141040.719  6.24%   102 4979s
 280890 105425     cutoff   60      150432.414 141049.662  6.24%   102 4986s
 281036 106152     cutoff   53      150432.414 141052.314  6.24%   102 5009s
 283409 106182 146737.502   52 1663 150432.414 141089.397  6.21%   102 5018s
 283471 106193     cutoff   51      150432.414 141092.251  6.21%   102 5023s
 283543 106340 145869.301   53 1782 150432.414 141096.965  6.21%   102 5028s
 284063 106362 150160.344   55 1594 150432.414 141102.639  6.20%   102 5033s
 284132 106561 145913.316   52 1366 150432.414 141102.639  6.20%   102 5039s
 284672 106629 149173.785   55 1724 150432.414 141115.014  6.19%   102 5043s
 284906 106759     cutoff   56      150432.414 141121.215  6.19%   102 5049s
 285583 106776 143996.100   55 1813 150432.414 141133.784  6.18%   102 5055s
 285657 106914 149660.365   48 1436 150432.414 141135.182  6.18%   102 5060s
 286173 107045 147321.698   49 1766 150432.414 141136.885  6.18%   102 5066s
 286604 107116     cutoff   71      150432.414 141147.305  6.17%   102 5071s
 286762 107291 141779.874   50 1472 150432.414 141152.252  6.17%   102 5076s
 287344 107376 147926.492   48 1739 150432.414 141160.270  6.16%   102 5081s
 287649 107414 146819.828   49 1615 150432.414 141166.669  6.16%   102 5085s
 287845 107578 145674.484   52 1594 150432.414 141171.060  6.16%   102 5091s
 288343 107600 147523.301   51 1383 150432.414 141176.293  6.15%   102 5098s
 288389 107790 148954.634   52 1408 150432.414 141176.293  6.15%   102 5103s
 288996 107857 144096.968   50 1571 150432.414 141191.725  6.14%   102 5107s
 289204 108014 146009.087   52 1745 150432.414 141196.510  6.14%   102 5113s
 289798 108081 150135.567   46 1538 150432.414 141202.985  6.14%   102 5117s
 289944 108295 150249.163   56 1502 150432.414 141203.249  6.14%   102 5123s
 290515 108430 141489.555   49 1718 150432.414 141212.457  6.13%   102 5128s
 290854 108440 147068.240   50 1456 150432.414 141215.605  6.13%   102 5132s
 290911 108660 147229.149   51 1193 150432.414 141215.605  6.13%   102 5138s
 291458 108780 146401.629   41 1839 150432.414 141222.979  6.12%   102 5144s
 291711 109262 145910.426   39 2350 150432.414 141225.185  6.12%   103 5157s
 293013 109284 146550.519   48 2033 150432.414 141242.317  6.11%   103 5167s
 293075 109409 145787.798   53 1692 150432.414 141243.875  6.11%   103 5172s
 293500 109509 144701.733   48 1430 150432.414 141254.377  6.10%   103 5176s
 293687 109717 141668.380   48 1461 150432.414 141254.891  6.10%   103 5181s
 294214 109759 149031.146   57  686 150432.414 141261.897  6.10%   103 5187s
 294328 109961 150144.500   58 1460 150432.414 141263.690  6.09%   103 5192s
 294880 110164 148025.738   59 1546 150432.414 141269.968  6.09%   103 5198s
 295342 110201 146187.820   49 1207 150432.414 141279.514  6.08%   103 5205s
 295442 110417 143905.611   53 1136 150432.414 141279.514  6.08%   103 5211s
 296020 110563 150086.807   56 1483 150432.414 141286.401  6.08%   103 5217s
 296501 110566     cutoff   73      150432.414 141293.997  6.07%   103 5225s
 296562 110766 145271.544   48 1514 150432.414 141295.895  6.07%   103 5230s
 297082 110936 150428.144   61  832 150432.414 141303.872  6.07%   103 5236s
 297674 110958 148593.041   57  893 150432.414 141311.956  6.06%   103 5244s
 297749 111147 143571.082   55 1962 150432.414 141313.377  6.06%   103 5250s
 298346 111333     cutoff   55      150432.414 141318.225  6.06%   103 5256s
 298927 111359 148184.746   52 1193 150432.414 141331.536  6.05%   103 5261s
 298992 111536 145315.844   49 1575 150432.414 141331.646  6.05%   103 5266s
 299474 111627 infeasible   64      150432.414 141336.185  6.05%   103 5271s
 299793 111683 146526.015   50 1422 150432.414 141344.503  6.04%   103 5275s
 299897 111837 148918.119   53 1595 150432.414 141344.503  6.04%   103 5281s
 300409 111851 146712.130   48 1504 150432.414 141351.836  6.04%   103 5286s
 300474 112027 150330.175   54 1598 150432.414 141353.921  6.03%   103 5292s
 300976 112179 149461.784   54 1454 150432.414 141357.136  6.03%   103 5297s
 301331 112225 144797.749   53 1295 150432.414 141363.186  6.03%   103 5301s
 301451 112385 146423.711   54 1498 150432.414 141364.523  6.03%   103 5307s
 301945 112430     cutoff   66      150432.414 141370.025  6.02%   103 5313s
 302058 113166 148728.030   61 1419 150432.414 141371.305  6.02%   103 5338s
 304435 113224 144496.737   50 1191 150432.414 141394.770  6.01%   103 5343s
 304565 113251 148991.255   61  950 150432.414 141397.339  6.01%   103 5350s
 304623 113434 143325.787   58 1256 150432.414 141400.115  6.00%   103 5357s
 305275 113600 149276.310   52 1724 150432.414 141410.778  6.00%   103 5363s
 305803 113650     cutoff   54      150432.414 141413.949  6.00%   103 5367s
 305920 113841 148601.524   58 1831 150432.414 141416.855  5.99%   103 5373s
 306468 113911 149913.373   58 1113 150432.414 141423.144  5.99%   103 5378s
 306773 114106 143094.687   50 1543 150432.414 141430.058  5.98%   103 5384s
 307351 114170 145136.445   66  739 150432.414 141435.874  5.98%   103 5389s
 307469 114354 145105.220   48 1710 150432.414 141438.404  5.98%   103 5394s
 307983 114461     cutoff   57      150432.414 141446.232  5.97%   103 5399s
 308326 114461     cutoff   53      150432.414 141448.484  5.97%   103 5400s

Cutting planes:
  Gomory: 98
  Cover: 310
  Implied bound: 384
  Projected implied bound: 537
  Clique: 82
  MIR: 4228
  Mixing: 3
  StrongCG: 6
  Flow cover: 1858
  GUB cover: 4
  Inf proof: 53
  Zero half: 10
  RLT: 156
  Relax-and-lift: 215

Explored 308330 nodes (31781937 simplex iterations) in 5400.08 seconds (4221.94 work units)
Thread count was 8 (of 8 available processors)

Solution count 3: 150432 150498 164988 

Time limit reached
Best objective 1.504324135632e+05, best bound 1.414484842250e+05, gap 5.9721%

======================================================================
Leas=30000 ratio×1.0 LCM×1.0 bi×1.0 SL=0.97
======================================================================
  Status: TIME_LIMIT  |  Costo: 150,432.41€  |  Gap: 5.97%  |  Tempo: 5402.1s

  SKU  T   DC               B    c_CM    c_AM  ratio
  0    CM  [0]            3.0    2000    2632  1.316
  1    CM  [0]            3.0    2373    2937  1.237
  2    CM  [0]            2.0    3218    3367  1.046
  3    CM  [0]            2.0    3774    3589  0.951
  4    CM  [0]            2.0    4004    4333  1.082
  5    CM  [2]            2.0    4384    5135  1.171
  6    CM  [0]            2.0    5677    5387  0.949
  7    CM  [0]            2.0    6073    7633  1.257
  8    CM  [0]            2.0    6785    7633  1.125

  Stampanti: [0. 0. 0.]
  Costi:
    base_stock          :    101189.41€
    EOQ_CM              :      3979.94€
    backorder_LT        :     16226.09€
    trasporto           :      5255.06€
    produzione_CM       :     18174.88€
    holding_inv         :      5607.04€
    TOTALE              :    150432.41€

  DETTAGLIO ASSEGNAZIONE CLIENTI (y_sol):
    SKU 0 (CM) → DC0: clienti=[0, 1, 2, 3, 4, 5], μ_DC=3.00, B=3.0
    SKU 1 (CM) → DC0: clienti=[0, 1, 2, 3, 4, 5], μ_DC=2.70, B=3.0
    SKU 2 (CM) → DC0: clienti=[0, 1, 2, 3, 4, 5], μ_DC=2.50, B=2.0
    SKU 3 (CM) → DC0: clienti=[0, 1, 2, 3, 4, 5], μ_DC=2.10, B=2.0
    SKU 4 (CM) → DC0: clienti=[0, 1, 2, 3, 4, 5], μ_DC=2.00, B=2.0
    SKU 5 (CM) → DC2: clienti=[0, 1, 2, 3, 4, 5], μ_DC=1.80, B=2.0
    SKU 6 (CM) → DC0: clienti=[0, 1, 2, 3, 4, 5], μ_DC=1.60, B=2.0
    SKU 7 (CM) → DC0: clienti=[0, 1, 2, 3, 4, 5], μ_DC=1.40, B=2.0
    SKU 8 (CM) → DC0: clienti=[0, 1, 2, 3, 4, 5], μ_DC=1.30, B=2.0

  DETTAGLIO BASE STOCK PER DC:
    DC0: 8 SKU (0 AM), B_tot=18.0, stampanti=0
    DC2: 1 SKU (0 AM), B_tot=2.0, stampanti=0

  BACKORDER LT ANALITICO PER SKU:
    SKU   Tech      bo_LT        b_i   L_tech  μ_tot    β    E[BO]
    0     CM       425.09      49995   0.2500   3.00    3   0.0085
    1     CM       345.47      59332   0.2500   2.70    3   0.0058
    2     CM      2418.73      80460   0.2500   2.50    2   0.0301
    3     CM      2159.46      94356   0.2692   2.10    2   0.0229
    4     CM      2885.44     100103   0.3077   2.00    2   0.0288
    5     CM      2370.95     109609   0.3077   1.80    2   0.0216
    6     CM      2219.84     141918   0.3077   1.60    2   0.0156
    7     CM      1638.39     151831   0.3077   1.40    2   0.0108
    8     CM      1762.72     169627   0.3269   1.30    2   0.0104

  METRICHE ANOVA: n_AM=0, n_DC_CM=1.00, H̄=0.000, costo=150,432€

==========================================================================================
TABELLA RIASSUNTIVA — CON BACKORDER LT ANALITICO
==========================================================================================

      Leas  B_cap  n_AM  nDC_CM     H̄        Costo    Gap   t(s)
  -----------------------------------------------------------------
     15000   None     9    0.00  0.000      149,988   6.2%   5402
     15000      5     5    1.00  0.000      153,102   5.9%   5402
     30000   None     0    1.00  0.000      150,432   6.0%   5402